# GLiNER multi-v2.1 — DIMER E2E named-entity recognition adaptation tutorial (standalone)

[![GitHub](https://img.shields.io/badge/GitHub-181717?style=flat&logo=github&logoColor=white)](https://github.com/kurtvalcorza/gliner-ner-pipeline) [![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/kurtvalcorza/gliner-ner-pipeline/blob/main/tutorials/gliner_ner_colab.ipynb) [![Hugging Face](https://img.shields.io/badge/%F0%9F%A4%97%20Hugging%20Face-urchade%2Fgliner__multi--v2.1-ffcc4d?style=flat)](https://huggingface.co/urchade/gliner_multi-v2.1) [![Upstream](https://img.shields.io/badge/Upstream-urchade%2FGLiNER-181717?style=flat&logo=github&logoColor=white)](https://github.com/urchade/GLiNER) [![arXiv](https://img.shields.io/badge/arXiv-2311.08526-b31b1b.svg)](https://arxiv.org/abs/2311.08526)

**Profile:** `E2E`  
**Mode:** `GUIDED`  
**Notebook specification:** DIMER Notebook Specification 2.2 — **standalone** (§4)  
**Capability:** zero-shot and domain-adapted named-entity recognition with arbitrary label sets

**This notebook is standalone.** It carries the repository's package (3 modules under `src/gliner_ner_pipeline/`, at revision `04b30e615194`) verbatim in Section 2, the pinned model identity and the per-file SHA-256 manifest in Section 3, and the exact runtime pins in Section 1, so it keeps working after export even if the repository changes or disappears. Its only external dependencies are the pinned Python distributions and the Hugging Face Hub at the immutable revision `443d26d654e0324125a96bebd8e796c14ff2efe6` (~1160 MB, digest-verified before loading). It was generated by `tools/build_notebook.py` (build_notebook.py/2.1); edit the repository and regenerate rather than editing cells.

**Run all:** Selecting **Run all** in a fresh supported runtime builds an isolated environment from the hash-locked pins (the kernel's own packages are left alone, so no restart is needed), stages and digest-verifies both pinned snapshots (the GLiNER multi-v2.1 weights and the mDeBERTa-v3 tokenizer assets, about 1.16 GB), builds the deterministic 24-sentence biomedical NER sample in code (no download), validates it against the record contract, splits it with a seeded shuffle into 18 training and 6 validation sentences (refusing a split that leaves a label out of training), measures the zero-shot baseline on the validation split, fine-tunes the layers above the frozen mDeBERTa text encoder for three epochs, evaluates again and reports exact-span micro/macro F1 and the deltas, extracts entities from a new clinical sentence, exports the adapter weights with a tensor manifest to a `.pt` artifact, and reloads that artifact into a fresh pipeline to check every adapter tensor and the predictions on the whole validation split. The default path needs no repository clone, no DIMER worker or service, no credential, no upload dialog and no configuration edit (NOTEBOOK_SPEC 2.2 §5). It runs on CPU (the model stages take about a minute and a half on a laptop CPU); a CUDA runtime is used automatically when present. Building the isolated environment comes on top of this.

**Bring Your Own Data:** After the tutorial workflow completes, set `USE_BYOD = True` in Section 4, select that cell and choose **Runtime → Run after** (it re-runs Section 4 and every later cell; Sections 5 and 6 reload the pretrained model first, so the sample adaptation never carries over into your baseline). Supply one `.json` or `.jsonl` file through the upload dialog on Colab, or by path in `BYOD_PATH` on any runtime. Any entity-label set works: list your labels in `BYOD_LABELS` (comma-separated), or leave it empty to use every label found in the file; each label must occur in the training split. Put a sentence of your own in `BYOD_INFERENCE_TEXT` for Section 8. The record schema with an example, the minimum size (14 records at the default `VAL_FRACTION`) and the ceilings are printed in Section 4 before the upload, and the file passes through the same validation, split, baseline, adaptation, evaluation, inference, export and reload cells as the sample. Uploaded files stay inside this runtime. BYOD is optional and never part of the default path.

GLiNER is an encoder-based entity extraction model that matches **any label you type** (for example `person`, `disease` or `gene_protein`) against candidate text spans. The sentence and the label names are encoded together by mDeBERTa-v3; every span of up to 12 words gets a representation, every label gets one too, and a span is returned with a label when their match score is above a threshold (0.5 here). Because the labels are just text, the model works **zero-shot**: it has never been trained on your label set. It is often weaker on specialised vocabularies, and this notebook measures how much a short **domain adaptation** on 18 biomedical sentences changes that.

**Who this is for.** A learner who knows basic Python and has used Colab or Jupyter, and wants to see how a pretrained entity extractor is measured on labelled text and adapted to a new domain. No prior experience with GLiNER, named-entity recognition (NER) metrics or fine-tuning is assumed: entity spans, exact-span matching, precision, recall, F1, micro and macro averages and the frozen encoder are explained where they are first used and again in the **Glossary** at the end. A CPU runtime is enough.

**Input → Model → Output.**

| | Inference | Adaptation |
|---|---|---|
| Input | a text (up to 5,000 characters) and 1 to 25 label names | 18 training sentences, each a list of tokens with labelled entity spans `[start_token, end_token, label]` |
| Model | GLiNER multi-v2.1: mDeBERTa-v3 encoder → span and label representations → match scores, threshold 0.5 | the same model with the mDeBERTa encoder frozen and the layers above it (11.4 M parameters) trained with AdamW |
| Output | entities: `text`, `label`, character `start`/`end`, `score` | an adapter `.pt` file and exact-span precision / recall / F1 on the 6 validation sentences, before and after |

**How to use this notebook.** Choose **Runtime → Run all**. Sections 1–3 are **infrastructure** — the isolated environment, the carried code and the model verification — and can be run without study; their code is collapsed. The learning path starts in Section 4. Form fields (`# @param`) are the only values meant to be edited; the defaults reproduce the recorded path. Each stage asks you to **predict** before it runs, and the next cell opens with **What to notice** and a collapsible **Check your reasoning** block with a worked answer from this repository's CPU run of the same stages (the recorded Kaggle Tesla T4 run of the previous notebook version printed the same +0.2332 micro-F1 delta; GPU numbers can differ in the last decimals). Each optional experiment names the cell to change and the cells to re-run. Section 11 is a **change-one-thing activity**. **Troubleshooting**, a **Glossary** and a **Conclusion** template are at the end. Your notes are optional and are not required submissions.

**Roadmap:** 4 build, validate and split the sentences → 5 zero-shot baseline → 6 adapt the layers above the frozen encoder → 7 compare before and after → 8 extract entities from a new sentence → 9 export the adapter and check the reload → 10 write the outputs → 11 **change one thing: the number of epochs** → conclude. Core concepts are Sections 5 and 7 (what the model returns and how it is scored); evaluation practice is Sections 4 and 7 (split, baseline, delta); engineering and reproducibility are Sections 1–3, 9 and 10.

**Learning objectives:** by the end you should be able to (1) explain *text + label names → span and label representations → scored entities* and why GLiNER can take labels it was never trained on (Sections 5 and 8); (2) read exact-span precision, recall and F1, micro and macro, and the per-label counts behind them (Sections 5 and 7); (3) say which layers the adaptation trains and why the encoder stays frozen (Section 6); (4) predict and then check how the per-label F1 moves after adaptation, and say what 6 validation sentences can and cannot show (Section 7); (5) check that an exported adapter reproduces the evaluated model, tensor by tensor (Section 9); and (6) predict and then observe what one epoch instead of three changes (Section 11).

**This notebook does not demonstrate:** relation extraction, coreference resolution, nested or overlapping entities, unconstrained full-parameter fine-tuning of mDeBERTa on tiny data (prone to catastrophic forgetting), tuning the 0.5 score threshold, a benchmark evaluation (only 6 templated validation sentences are scored), or arbitrary unvalidated dataset formats. The repository exposes none of these.

## Prerequisites

- **Learner:** basic Python, and Colab or Jupyter familiarity. No prior NER knowledge: the notebook explains entity spans, token indices versus character offsets, exact-span matching, precision, recall, F1 and the micro/macro averages where they are first used, and the Glossary repeats them.
- **Runtime:** a fresh **Linux x86_64** runtime — Google Colab, Kaggle or Linux Jupyter. Section 1 builds its own Python 3.12.12 environment from a hash-locked list of manylinux wheels, so the kernel's own Python version does not matter, and a Windows or macOS kernel is not supported (Section 1 stops with that message). CPU is enough: the model stages took about 70 s on a laptop CPU in this repository's check, plus about 2 minutes for the 1.16 GB snapshot download; a CUDA runtime is used automatically when present. The locked install (PyTorch 2.14.0 with its CUDA libraries, a few GB) is the largest download of the run.
- **Expected warnings:** the cell after Section 3 prints `load_warnings`, the Python warnings captured while the model loaded. In this repository's CPU check it held one `UserWarning` from the SentencePiece tokenizer conversion about **byte fallback**; it does not change the tokens of the text used here. `transformers==4.57.6` may also log a line about an "incorrect regex pattern … `fix_mistral_regex`" while building the fast tokenizer; it refers to Mistral tokenizers, not this SentencePiece model, and is a log message, so it is not in `load_warnings`.
- **Data contract:** a record is `{"id": str, "tokenized_text": [str, ...], "ner": [[start_token, end_token, label], ...]}` (`tokens` is accepted for `tokenized_text`); token indices count from 0 and the end is **inclusive**, so `[0, 1, "person"]` covers the first two tokens. Spans may not overlap, ids are unique, a record has at most 384 tokens of at most 100 characters, a dataset at most 1,000 records. The seeded split must leave at least 4 records in each split, so **a BYOD dataset needs at least 14 records** at the default `VAL_FRACTION = 0.25`, and every label must occur in the training split. Section 4 prints this before the upload control.
- **Data:** the default sample is a deterministic 24-sentence synthetic biomedical dataset (`disease`, `chemical_drug`, `gene_protein`), built in code. Every sentence follows one template (a drug, then usually a gene or protein, then a disease), which matters when reading the results (Section 7).
- **Privacy:** Do not upload confidential or restricted data to a hosted runtime unless you are authorized to process it there. The default path uploads nothing.
- **External access:** the Hugging Face Hub only, to fetch the pinned `urchade/gliner_multi-v2.1` snapshot (~1160 MB in total) at revision `443d26d654e0…`. No repository access and no credentials are required; nothing is installed from this repository. The isolated environment also needs PyPI (`pypi.org`, `files.pythonhosted.org`) for the pinned `uv` wheel and the 50 hash-locked packages, and the managed CPython 3.12.12 build (python-build-standalone) that `uv` downloads.

## 1. Install the pinned runtime (isolated environment)

> **Infrastructure.** You may run this section without studying its implementation; the learning activities start after Section 3.

Hosted runtimes such as Colab import some packages, NumPy among them, before the first cell runs, and Python cannot swap a module that is already loaded. Installing the pins into the notebook's own Python would therefore leave mixed versions or require a manual restart. Instead, the next cell builds a separate environment (`dimer_isolated_env/`) and leaves the kernel's packages untouched: it downloads the pinned `uv` 0.12.15 wheel and refuses it unless its size and SHA-256 match, has `uv` install the managed CPython **3.12.12** (whatever Python the kernel itself runs), and installs the 50 packages of the carried hash-locked requirements (`tutorials/requirements-colab.lock.txt`, compiled from the pins below) with `--require-hashes --only-binary :all:`, so every package, direct or transitive, is the exact file that was locked. No repository code is installed. The lock holds manylinux x86_64 wheels, so the notebook supports **Linux x86_64 runtimes only** (Google Colab, Kaggle or Linux Jupyter); on any other platform the cell stops with that message. The printed dictionary names the isolated Python version and the kernel's.

In [ ]:
# @title Infrastructure: install the locked runtime into an isolated environment
# dimer: kernel cell (runs in the notebook kernel, not in the isolated environment)
import hashlib
import io
import os
import platform
import subprocess
import sys
import time
import urllib.error
import urllib.request
import zipfile
from pathlib import Path

PINS = [
    'torch==2.14.0',
    'torchvision==0.29.0',
    'torchaudio==2.11.0',
    'gliner==0.2.29',
    'transformers==4.57.6',
    'huggingface-hub==0.36.2',
    'safetensors==0.8.0',
    'numpy==2.5.3',
    'protobuf==6.31.1',
    'sentencepiece==0.2.2',
]
MANAGED_PYTHON = '3.12.12'
UV_URL = 'https://files.pythonhosted.org/packages/1e/fd/432451d732917c49152a291de3ef171aa6b0f1a22d39780fb2c1f085ca4c/uv-0.12.15-py3-none-manylinux_2_17_x86_64.manylinux2014_x86_64.whl'
UV_BYTES = 20081404
UV_SHA256 = 'aee9802f46bae436bd91751bb33ddeb379ef1596b5c19df193219d545d244b60'
LOCK_NAME = 'requirements-colab.lock.txt'
LOCK_SHA256 = '0518a5de4b09a62b9f88881b6c41cfd2e0c917e442e3b8f5adddfded8464cf94'
LOCKED_PACKAGES = 50
# The hash-locked requirements, compiled from the PINS above with `uv pip compile --generate-hashes` for manylinux x86_64.
LOCK_TEXT = r'''# This file was autogenerated by uv via the following command:
#    uv pip compile pyproject.toml -c <esm2-protein-pipeline tutorials/requirements-colab.lock.txt @ 37f7112, versions only> --python-version 3.12 --python-platform x86_64-manylinux_2_28 --generate-hashes --only-binary :all: -o tutorials/requirements-colab.lock.txt
certifi==2026.7.22 \
    --hash=sha256:62f22742b58a1a33014a2b6b706588a8d7e2a88ae7bd1a6ebe8c992928483775 \
    --hash=sha256:741e2c3b351ddf169a738da9f2c048608ff7f2c5cc02f1ebc6b118bb090d5d55
    # via
    #   -c esm2-protein-pipeline lock (versions only)
    #   requests
charset-normalizer==3.5.2 \
    --hash=sha256:01077390b03f7988f11d700a2194e69b119741a86b1a638b1db88891e3eced8e \
    --hash=sha256:01b0c0d2262a9e28e8484a278c7e1b5d650e3ac8cf2683d2967e25899f208bdf \
    --hash=sha256:04851f73ae72b8413dddadb16a49dfee95263553741fd42d546f7d66907e6be5 \
    --hash=sha256:0521c5665880b33d603717defa76c094048900010897909952397feb3039da56 \
    --hash=sha256:0774bf9bf620249fee3e0b8b9fd3065de213be30f3aa94ce2494b3b638949e26 \
    --hash=sha256:0891b9d3903c5571c03771ca669a4b0ec5618ca722a5c957d3d29cd4e5062848 \
    --hash=sha256:0c951d5e6dd9c2ff60609476752bee49da4206adde960ebc247766937f72e718 \
    --hash=sha256:0fed1d06615f022ee3b13caf5e8b180cfea32bb2c5aded8a9d44277afc040f93 \
    --hash=sha256:114e4d0c92d618409ed82a99e22b5c5e768fe995f2973f78265f4524f49d4640 \
    --hash=sha256:11912e4bb14baae7c5d8791aa55ba0a3a03ec6729073307b0f57270abaa713d3 \
    --hash=sha256:11a4d68a6ecda3292cb1e50239e111543ba5d709bb62a6b4ea1afcfa729d8875 \
    --hash=sha256:124fbf1a8ff966d87ae05bb8bd45a71f966055ed8bba320d0c7cf450bc5f4d0e \
    --hash=sha256:1461ac396c4fdb983a675f20aa555624f0ee18ac83d832b9244ffff3d8055275 \
    --hash=sha256:1503bccbeb36d5527790c3930327704c39af22de3112f1b1666a9f3ce15ee204 \
    --hash=sha256:15bb4005af6320d259dc7593ca84a38d7fe06a421dbcf7b910ae23979101e787 \
    --hash=sha256:15c44f7edfd477b06f517a5cc317fc1707edb9de2c865f43d4b6513907473234 \
    --hash=sha256:16fa0eccf81304b79c5cd87f9271c3b85dd9dd99245e4422ae9c0dd45e0f99d3 \
    --hash=sha256:183b88127acdb4fabe59d951ab424faf1af7b63cdbb5f776186c1ea2ffcaed98 \
    --hash=sha256:195c26fb65950f8fce54e26349852b7bdd7c5f120aeefbcc440b8a20faaed4a3 \
    --hash=sha256:1afb975bd5d68d5ce9f6b6d44fdf2f7e34b895a35e95708a7a91b20a3b51d187 \
    --hash=sha256:1b4cbc7c3491ccb4aa17fcd8165649d01cf39f76de1696da8631b5f71b85401d \
    --hash=sha256:1bc0baf5ef96b6ede57d47f4b8fe4d9d84019c3bfcbeb20a41edc6a6ee341f1f \
    --hash=sha256:1c50fe28bbc2ced33386f298650d91218076c05420e6cbd790b913adc41659e7 \
    --hash=sha256:1db38f4c5496827c1a501846d64d14c3b80c7e6714e406cd7dc36a9899fa1011 \
    --hash=sha256:211d5a3eb6af8f513b8d4ca19a8c1b7accab1b5f0d3175f9826b03c1a920dc1f \
    --hash=sha256:23851fb4e1b85ed3f6c2a27b777cdfe2e19fb5b38429a8faf38c7542b7665869 \
    --hash=sha256:254eb48b9fa5ee9898a3c445825a1f340fe53712a098904b39b0bddba8ea3cb1 \
    --hash=sha256:2625388c6c754520c37abaf3b41eb34d1cc4a373f457898f08606c8e362b891d \
    --hash=sha256:281cb91036248400f4cc957495cccd44c275c2e0c5854f7e45ac5cf7dc193847 \
    --hash=sha256:28a15fdad492a99b6eccfaaed66ef3f74050680545ea61ec8b2f4c538f1f1320 \
    --hash=sha256:28b4f0d66fb834ff90f28209ac7bce77868c45d8c93e26f906709d9b7c2e1af9 \
    --hash=sha256:2a925889534b3748302dae5dead07cc13480de1dac3aea80a941b729b471ef93 \
    --hash=sha256:2b7b3bbfb4fe8ef40600792d762fbaa9057559f9d3fad209525b7a22b99e91fd \
    --hash=sha256:2c9ad19a6cfcd5ea5c0d41161d22f9df1dcc277e9bef2751391334546a314c00 \
    --hash=sha256:2cc961b171b3f3440f410489ab3573e86aea8736134ebbb40ea1338b7f0831bc \
    --hash=sha256:2ce45c6627b22c47e390bc91a41c3d13032192e699fa0bea96e9671b373d69b0 \
    --hash=sha256:2e06a3a98f916dd41d27f3105e02e7a40181c98c94b9158733d03a6f80506c09 \
    --hash=sha256:304d5463e65a35d7bb0850550e0780395395f6fcf452f04db7d5ca7cecc425ac \
    --hash=sha256:304d8e4d493af723536393eee0c689eb7813f4a474c8b479dee63f1fdd98f621 \
    --hash=sha256:30fcd120b732aa79317f08dee04d7de0847822e4cf7ee0e9f445bb958832252c \
    --hash=sha256:31f3930700408d211f13378ccbe1c40845d8da54bd0681fac3a9b5aae81c7aa8 \
    --hash=sha256:34276fd796040bf0993ab33a369aa572e6979c7aab225a88893667ad8eac8f7a \
    --hash=sha256:355ad8011081dec5412240c087a9a0c9d4d5039f3ed11a3f13e18c2b29b56c51 \
    --hash=sha256:38a873987f3be698494da8b2e3085e29da02da7b633dce73e79c699a113d7bf0 \
    --hash=sha256:39de2a259fc954455c57274dc94c79d5842774e1247a016aff30bc0efed0f4ef \
    --hash=sha256:3d14b50de6bf4d0edf857a9386836846f982b8f524e188e2e68b96d702bcf4aa \
    --hash=sha256:3d21b8b13c7592db2ac5e544a6d83187b995257472b0c9e8351b6d507ae37ed6 \
    --hash=sha256:3d31298449090ab8d47b7b1b2a555ff73cac7ed438a08b7ac160980c7ebed649 \
    --hash=sha256:3ddacd27458c45bdacd6bd6db644bfb730efbf9e830310186e3045c9c5be8fb2 \
    --hash=sha256:3df041de8887954562c9b261cba85ca0e9ded74048daf125f45edcfaa4832229 \
    --hash=sha256:40ab6bffa02ae10a0581e6c198be7d2d8ca5c2a0c64e4ed3465d766df457573e \
    --hash=sha256:4275811936e2f06feff5e598fb42a1b7ae852da8e39605211892b56b81a34efd \
    --hash=sha256:443eae2bf318abeaf6f15d785138f71fd6de770e99a92158b8b814265e079115 \
    --hash=sha256:447441e76ec720b15e64418d32e092297340387053047c7c694f579efb0ee1d9 \
    --hash=sha256:4495c5002a7b28557e7e222e77e0b661183e432b7d6d2e788101e3f240e05b8c \
    --hash=sha256:44bd4fbb29dfbeba60e7d2bd000c59e4b21ddb3cc53912b14048d37092706d7c \
    --hash=sha256:4685902cf26edf013ed7a3da0f426ebba7a00ebb9541386d835afbf002c11cab \
    --hash=sha256:498dc3188ca05a68231ac3fdbfc7f57eb67e1343c30e0fea17f8218c1599b253 \
    --hash=sha256:4c2b5031f63e331e3839b40aed2dd6f191e9c07edbde303e7876846ea1946995 \
    --hash=sha256:4d48f2d08b9de5864e2c8744d4461b862fb149a18274abc8b698c45975573438 \
    --hash=sha256:4f87960d57feabfb618e4e0af6e7371645fa26a277860739d6e5d6e0012c92f0 \
    --hash=sha256:50e3adfb96fc189eb27b1cf62d3b598b89b4bb0420d93a3d3e42e137409011be \
    --hash=sha256:51cf45226a9b588d0d2b4880c62d686934b63ab0bd79ca23ab0e9762eb27441b \
    --hash=sha256:52aa6992700996af31f375de0c6bacd402b0097fe40b53c426b9f51a90ebabc7 \
    --hash=sha256:55ea99acb17b9325618de155a0cd6a2e8f5d10be008113e1d433bbb58db543b2 \
    --hash=sha256:56bc200a365efb37383b7852e4cc5898d3b2da5987289b543956cf8cad71018a \
    --hash=sha256:588461c2e8384d309bd63e5826019b6977bc66d629b99ac8737bb795d7b2cb5a \
    --hash=sha256:58ca3755ee7ff7f59b57789ec9833c9de9ea275405cdd240eda1f193112e398a \
    --hash=sha256:58f361dcbab699cf8f42db3f47c8e7fd1036f138c23a5d08de9fde5f425a730c \
    --hash=sha256:598a11a2c7ebaa5334bf698bf29568c9c390abac6a154d8170fedecd1cea38c5 \
    --hash=sha256:59f63901b0031c3136cf64704dcb21de0bbae62ce2c9529bc39d27665463de37 \
    --hash=sha256:5cde776b7cc66e4f6c99612cea4aa7269aa65863f7a15841b2c264f103822f4e \
    --hash=sha256:5e2b6b57e9733d39f0c9fd3185efa6b8e29652c4cd8fe94180272cf6ed9a78c4 \
    --hash=sha256:5fb29fb8cd1a46c27a1bf9613ad5ec2599310d46b4025d9556404a6b6a292800 \
    --hash=sha256:6045373d5a89a5ec71afde535db987ca28e76dfa276c2d4c818265b375d4b055 \
    --hash=sha256:619799369eeef6366ed3e8755a5670f4f2f0fb6b30a0fd7264dc0fdc2357058e \
    --hash=sha256:62588a277bfb59def052abd940703fa35107152bf479781a878617d60faf8fb5 \
    --hash=sha256:62603db9a7caa0802eaa28c1c46fecd7b3a263a774069c24c3c28c302448721c \
    --hash=sha256:65cd72beeeca9d3aaea1201e5923859f308f952f9c71de93f06063c79f0f7a3b \
    --hash=sha256:68eb192d85ab8e5f6ec69c2bc6ac0179fbf04a5ac1569d12fbef74883fe102d0 \
    --hash=sha256:6bd128f206a7752ae1f2ab6c61bf8a24ba28913a10df8b14c2637b973ff97a80 \
    --hash=sha256:6be488a102b8cf28d0391d8c4ba7748938ae28b78ad901f8585520fca33ead1a \
    --hash=sha256:7218e8f32b0956cfcd048fd42d9d5779809745ca1d86113ca56f66e7ae1549c4 \
    --hash=sha256:7441d755b7ab94f8d4eb3e43ec05482d760842fd263d003a99102d742cd835e2 \
    --hash=sha256:749e97e1b32313717a565abbe321bc2190bc8b35f1a67e4cdbc7c56c8d8ffe58 \
    --hash=sha256:75a3ceed0724d625d64b86ca20aba182e4df462e04c2414fc941c0f523f06aac \
    --hash=sha256:780fbe7cab297b81dad9fb8dc5eb003c0468ffb0d9e5f65068c53a34661a96bc \
    --hash=sha256:78456a747de8dc58360ffa581f30a002baf5aa28cb262536545e91f113ed7639 \
    --hash=sha256:7967d08cf06dee78443b874f98c98036f624f3a4e73e11f9f64f5be4d25393cf \
    --hash=sha256:7a881931aa470808df94a8c380eed2bbbc76cd9dc622310f99665658c821eb6d \
    --hash=sha256:7dcd882da75ef9adf94903b1e3b9419e8aa8fb4c7396822b834b9ef7fb96954f \
    --hash=sha256:7e841fb9010836c992c9f12fcbd43a831de93a5f726fc1ccd8ca1d0268c5014c \
    --hash=sha256:7fdde2c9fd9e3eca40631e024664cf2584272cc8f96308cbe5fdfc930f51d8bc \
    --hash=sha256:8024d00c3faf3fc0c16e07a69f4405e8eac7cc0ab15f65fe6cf43827c4cf72b4 \
    --hash=sha256:80d02b6f04e92601a081dd97b23d3128033098bff5d35d392ddcc0476ea11253 \
    --hash=sha256:838dcc90063569a0448120554591a1d6c4a4ffe11babf048908793154ab86ade \
    --hash=sha256:849df64e889b2e17230d58410a03dba311a65b163508fd33679b2b737d4b7858 \
    --hash=sha256:87475fabc8d9996fd9c27debb395e642e8c838d78a00b6e932227a0e06b81e26 \
    --hash=sha256:87e50a3e7cb90af586b6c5faf23e302a970415ac73bd7bd90a515a04b427ef96 \
    --hash=sha256:89b53f3cda69831909888e0494f4fa0bcd3537e3e138dabeb620bd6ad946bae8 \
    --hash=sha256:8a893cc101149f80a653f82062ebc95b34525a2614382e1da5458fe7c6997249 \
    --hash=sha256:8b2bfab86aa71ae13aa41a6a26aab338e0db2b8bc75434b05aea89e011ff35a4 \
    --hash=sha256:8d86d6fc60743dc916eb79e2eb1ec4818e21e427731543af40a3021851174a13 \
    --hash=sha256:915563965d418f986e7e145accc592eae9e1a1be3566ff98a05d7a9ec42a76e1 \
    --hash=sha256:92888bb3187c5ba50500b00b3b310c9f2c651709d28036077680cb5255450a03 \
    --hash=sha256:93223adc95033dd47133a46ccfc316a0139176fd79085762e27202ec56018f03 \
    --hash=sha256:9373ad13ef0d2c0fb761e04e55bfdee5a08b52cef2c882c8fbe9935b1517152e \
    --hash=sha256:9409a8bf35cf78353942504b24a57de3d75b708997a1e4bd8db71ac8633ce364 \
    --hash=sha256:9b7f416ff0978e2f2249330527f0ad6fa02f4932e6199692d3b52da2048c19e4 \
    --hash=sha256:9bde855991b7e362c146535e3136a50bfaffc0487d38b33ca7e5edefc6e23849 \
    --hash=sha256:9cae88599c7219005d879f98e5ed53341e9a122af585e1091200358a3003d2a0 \
    --hash=sha256:9cf9b1a857e25c4baceeb3624e92a56df3668f398c4acba74e174d81fb4d1d3a \
    --hash=sha256:9f56f72050826f63dcee7a7f55b0a77168cb3bfc553fd405e7f8f9ece75a4036 \
    --hash=sha256:a090bb2c68df85450502e3e20d665e3a5af9c65a84d6508ed477badd49166fd3 \
    --hash=sha256:a192e2c40070d92c3ccf777e3a5c4ff515573cd2bb7ed0c537fdadbbec5bbf21 \
    --hash=sha256:a19a731138fc27d5682277d3b9df22855cea1239bce7fcec5f78f42ef2d1f3c3 \
    --hash=sha256:a66c3bc5ab1f0ff2164fc9965ddd611ff0802173f4b9d24554c563f6ab7e1d6e \
    --hash=sha256:a815775b6c38d4e0ff7bcffbeba67feded90202bb6a226b8dd35f1c855217413 \
    --hash=sha256:a89012d6d5476ee112d20d998570ed58df2260a852afb1758809cd6900411d21 \
    --hash=sha256:ae4f5fea5b8b8ccff88238cc8569303e5ee95efae67fa62922a311397a71f346 \
    --hash=sha256:b6856554c4f44d79fc2307d5768854310a8f0096e501c75637542c82292b0429 \
    --hash=sha256:b6b751274acb69d77b3323d6b7dbaa3c7fdfc1eb829b7eb61d262f32e1af9685 \
    --hash=sha256:b736353c0a625bbd5fcec108576e2385db3496f4f771f785ff32e108d3c3bc45 \
    --hash=sha256:b7fd005a73d9e657273b7a10dc71a9e03c8fb9ee6999798d6918ce095b81ac7f \
    --hash=sha256:b91363207bd9dc966a691e959bb47f64b30f7ac4b072be9968b366982f7db77c \
    --hash=sha256:ba0b1d2620edf869789c3879223f52bf2afc5d31b3cb47cc57b3a12c05e2aa9d \
    --hash=sha256:bbbfc8e28816f19d7c0f1816664980c0a9875d01b27cdf8eedddb639d9e108ad \
    --hash=sha256:bd16aabe4a02a297c23417aa17ac6299dbd8c49f673bcd645b4929b11f5a4400 \
    --hash=sha256:c0afc6800ba57ccc350374c5bd6150419915d95ce93cdbab2d783d75eaf30ecb \
    --hash=sha256:c6708715abcf3c73b99508253e961a9967f02fe536532834149574eda6de0d1c \
    --hash=sha256:c7c9ab723cde841fefb34efbad91e87f00a674b1fe1cd0784fde742bf2c154dc \
    --hash=sha256:c8f3d67aeaf55f017982b73683f0e7342ba2f6635a78f69ce89ebb26aa411e5c \
    --hash=sha256:c9790464842f85f437dbbb54417eda1e0e6bfc52dd8d22d6fd1c994b73b2dc74 \
    --hash=sha256:ca403d7e4798f525fdfc78e258820419cbbd0f0ecbab9de7840e3c017cf6b8cf \
    --hash=sha256:d008d90a7f2471519aef0c90dfbe73b3e6e4d5e66ac48e19154c17e89e98b604 \
    --hash=sha256:d19fbd981a488e22cd04883659ca6b08f50b5974f9fd7c95655ef6a043e5893f \
    --hash=sha256:d1befeed746d247c81127bb14de9dc3d30edb6e5976d34f83f86ed262b1d9105 \
    --hash=sha256:d2374b62878abb00cd8309b32af6c0b715cd02dec0ca74ef12e5069bdc64144a \
    --hash=sha256:d376bbd28b3a8999db1a103b3b388aee6f1ddeb3e51bc2172993efdcd86e064d \
    --hash=sha256:d4a7319f304a774bed22115bc891618e45f85065ab44ea6acd07d274e750519a \
    --hash=sha256:d6734d2ef8a50fbf8445c139477da401f50d62a0606bf00e20ec6d87773fefb1 \
    --hash=sha256:d760fe2a4d7c3b226cb9026d6a842868d52a7901bd98420e1baf14e80da85cf5 \
    --hash=sha256:d913de495d90407cd859d263bee2e5d1a4ed3eb6573c04e70d9ec619a7cbed7f \
    --hash=sha256:db19d07e2e0129e974a0e65d0064fc222a446cd5122c2fd4184d2af9fc734a9e \
    --hash=sha256:dca9ab98072a5a54ebacebdc45f53e645336b320c667410b061be1ca588ae709 \
    --hash=sha256:ddc7dacc8ece3a182e7f15cb862d1fd616b46d076cb1ae9dd232b2c38b655874 \
    --hash=sha256:ddf19c062bea7a0cc80f519243d2c01dd091be0cf952a0750d4ad576709559f5 \
    --hash=sha256:def79fa35ef0cef8d2accec024f4fdc7ead3012ff02f5215c783f39f03ef8cfc \
    --hash=sha256:df29a0a7107f7011e77f4eebdddec4c7331e24d787a0b21a46d63bdf7445da95 \
    --hash=sha256:e09a3942ecbdee5cce73ea9d42da82b81b72ac1bf031ce069b93b5adf4eac8cd \
    --hash=sha256:e242bb1c5e76e97dfa9e7f209a71e93a01d7f19ffdd5cfbb2e2d55b4f08f8ab0 \
    --hash=sha256:e243bd13217235fc7290c621941c3f5cc8b66e4872495be821d7436ba2fb838d \
    --hash=sha256:e2af3aad578aa6bd1384bcf4750fc285e5a9de53f40b7d41e5a0bf748edeb2b3 \
    --hash=sha256:e4e81e09c1578b8df602e3db08b0b3ea0a6947ad612f52bf8dc5ea8d47691f0c \
    --hash=sha256:e54da4baf05720032d527874d40b65fa4d7e5c6c6a43d0c3adbeffcaf275a2b3 \
    --hash=sha256:e80e6c2f55656b4824d72065abb4ddd6a525c74bd78a0aab5d9fc2cf4fb5af50 \
    --hash=sha256:ed2a239c0ea213acc1908150a3037257083c7c083128f1a4cec2ec4b97dca491 \
    --hash=sha256:ed905975ab14056a2e5eb1c376cb2e1ebc5396baf84163939c518556fccde9f5 \
    --hash=sha256:ee21e28f0430bd6dc9086c6e525d5e818a44a5ad19720c8a0ef766792f3eb5e5 \
    --hash=sha256:ee43c17b173d46a3212baa6ead3ae258eeabdae48c263a01ccf0218c366dd655 \
    --hash=sha256:ef4fcbf3327382cd4c9f540babd61248208af7b93eec4de397b4d5f58a09e288 \
    --hash=sha256:eff0ac9dbe711a4aee69bf04a83896aa9b85f19641264053a9f6d48573abb7dd \
    --hash=sha256:f0aa869112ef88429ae17820d99c3dd9504c9e9c671d3c246f3d7442cb051084 \
    --hash=sha256:f3c96f633825733f735c5a9cf21d21a257d8e1edf0b1cee0a064b9c424ca0f7d \
    --hash=sha256:f5833ad231be5eb6553de524a70f48d71b2c8563101750531e0b80184e175cd4 \
    --hash=sha256:f5ec61164adcec446f8969a3358ec3f9b26bbda3b9213e5586d219afa8df2915 \
    --hash=sha256:f7d486c83842422badd511868fd8a9a20e9407ace71564b6af47ce7e60a336c1 \
    --hash=sha256:fb9e68df06293761f9fe66ade60a9bc6d0f5e42b8acf2939a9158af86ab0e5bd \
    --hash=sha256:fc14a032f813bf5fe624d991960ea83e9715adc27e4c1830a2361eb1d02ac341 \
    --hash=sha256:fcff63213e8e6e47770541a4607175404f47cbb3ebea7b6058cc82d524a0e424 \
    --hash=sha256:fd1fbe0f116b6e55da77aca2c6ddcddcfac2186cbf78bdebf40fc156efca389d \
    --hash=sha256:fe9753dfee015c570d73df76f899f18444d41388bffcde097deba51c4fadbb9f
    # via
    #   -c esm2-protein-pipeline lock (versions only)
    #   requests
cuda-bindings==13.4.3 \
    --hash=sha256:044c03b056dcc5cecfad426a071187dd9e1e6817fbb363bc2f3a7520170b3e67 \
    --hash=sha256:0deff5b22462bb410859684299b1fc6a621780c43a4729342aa6ec28783485b4 \
    --hash=sha256:130ff1daae550db2cef559ba477f3a63d040756bd135f5deba4b5bdc4e110252 \
    --hash=sha256:145cc9b02dfb7bcc3288701b6f19c69c590e4451c62916ba0a8a3db9a64a91a8 \
    --hash=sha256:1fd7d8459b364aedc11f3e59703453ced823135f78a9111ca70feef8d56d4d21 \
    --hash=sha256:3d7a6506e625be59cdc119ef4423afa0fc3a4830b129dc010cbd51324b93d66d \
    --hash=sha256:4796864ce829bd95ef2ef0d23c6ba21bb64e08f7fab0a377302ed1affb6605c7 \
    --hash=sha256:52d7f3f5f7f014dddddc66cd802ed0ddf65ae99ad42b5619fae7b82e5ddd6771 \
    --hash=sha256:6c6bb1a4c4f7520e062669c6f3605a7016b2e65ba51b8922f92edd91d5fadf8d \
    --hash=sha256:7e11cfe8fec4c85ce79feda18124971c52596f0cbd642a94f5dafc257124a4b3 \
    --hash=sha256:81eef62bbb95cb4a705fb423b2ad1c63d716edb352ee2af0c28bf8a29cc8258c \
    --hash=sha256:a9ea13b9cfe711515ae83b5efc99101af4d3f8ad882f99724a839c8b5417fb7e \
    --hash=sha256:b89d6e738494b7b95c38e3413f86d32c24682c8e714870531a5a2b193a2fc50a \
    --hash=sha256:bbacde6f75665b197016b986164cfdaa33b17515e5e635a63ddb75926aaa71c3 \
    --hash=sha256:bc51990309b416e0780a11a921ac597ef9c0e52e1bdf5ae0b8e8c4766b66442b \
    --hash=sha256:bfbd3f7d4ac04dd41dc49121b9e408c8283992f47124c2290ecb79bbbadcca8e \
    --hash=sha256:c2af7e69d2557fdb4e5fc1169159fd08da7e861aa84fb83307a1a0396b6b0973 \
    --hash=sha256:d5f72bcfcdf3be23e1da3c792f68f508586f48d037bca8b10f552c4cca5971f2 \
    --hash=sha256:d6eb969920e28f66f8fc3b0b3afcb6e09381cc96bf8e8158d774e9488ae89980 \
    --hash=sha256:d7c6c9f46fca7f3fc61959ef9a2398ac656172145b43f408e0a6492360cf1c0c \
    --hash=sha256:df8b3767facca8acde216460df684dfc3826d519096c13adfd3030a55870dc79 \
    --hash=sha256:e52f66340785a51b8b77f4487329de188f3cabbf37c62e68f410a8299e8677ae \
    --hash=sha256:f8519603001c92bf83e7095df3b8211e3999a9c4ded096b57de0f3ff52b66368
    # via
    #   -c esm2-protein-pipeline lock (versions only)
    #   torch
cuda-pathfinder==1.8.3 \
    --hash=sha256:e29e59829c297a7a5233bd9cc71094fc5bddbd076951482670178f9eade39b1f
    # via
    #   -c esm2-protein-pipeline lock (versions only)
    #   cuda-bindings
cuda-toolkit==13.0.3.0 \
    --hash=sha256:d693caaa261214ddd7dbb60d68e71cbed884e68c2be7509778f3051da0b91c3f
    # via
    #   -c esm2-protein-pipeline lock (versions only)
    #   torch
filelock==4.0.9 \
    --hash=sha256:635e7d67fa92654eed444e75e9ca18426d34e77ad9c469bf4373f75a932f7b22 \
    --hash=sha256:9287fd61b99a806e5202be29a83034c0808a1b9830e820537c2f5773981f7eeb
    # via
    #   -c esm2-protein-pipeline lock (versions only)
    #   huggingface-hub
    #   torch
    #   transformers
fsspec==2026.9.0 \
    --hash=sha256:0f08147951c8cb31d844c3547d631053b127863b60be04cf06e121333ee0e2fe \
    --hash=sha256:8dd6e646e99ea382bd85f97a45e6b526a442d79423a7dc673f1e2756d05fcb5f
    # via
    #   -c esm2-protein-pipeline lock (versions only)
    #   huggingface-hub
    #   torch
gliner==0.2.29 \
    --hash=sha256:0c8cfb9f5c2daf7aff329ebb5ab1609052d7ad0aca0ef26049b9476b018b3fde \
    --hash=sha256:39fa8f33c027627e1d6724bff972ab197bf37cea27e401cd3d40f2f95909b0a4
    # via gliner-ner-pipeline (pyproject.toml)
hf-xet==1.6.0 \
    --hash=sha256:0e6e21fa3cdfcdcd76748564bf593870a5e013f47d97cf10aed63aa222cff5b7 \
    --hash=sha256:23379c2f9ec8696d952b16414a2bae72cad86a52df869b050698ba60f538c675 \
    --hash=sha256:2e58454a340b3556dfa4972d5451aff4fba8dd42a236600ba1a1d2b1514f0fef \
    --hash=sha256:35cec30d75c6f9eb9c16a77cef68e85a103b72e24d4b473714ec9ff06428bab9 \
    --hash=sha256:3dc3e35441ba395006af5aaacc40ef2e603c51ef46c3530b9156185f00935ea3 \
    --hash=sha256:4fc74352a17015bd0ee90038bc9efe38db894cde45f268b6712b04fce8cd0acb \
    --hash=sha256:5153e6bb103ad49d6ea9f1b2e230db5a2ea32551ad09a706d2f61d7c7c80d80e \
    --hash=sha256:5789835d7c6bc9436962853192082374297fb72d7eff7e7762ec25ceb7e25338 \
    --hash=sha256:633dc0cd71d32da58ab8c03ad38e2fac452c15c2b0a2866ebf6ededfe0a5061d \
    --hash=sha256:70cbb9c896901600128cb9b6f06e132954fbede1db30f31f7c6c63f84cb7c31d \
    --hash=sha256:75765820ce4700db3750c94acc8fe27c5fae4c9ec000a0dbac3ca082acf97765 \
    --hash=sha256:8fb4f71cba6129110c3374a33f919001ff130488fc23553698e34cc1c2a1198c \
    --hash=sha256:948f15d3a9545cfe5932f6bd8b440f6ae630aee108f14b7bd6c561f7c2dcc522 \
    --hash=sha256:d62671bb130879cef0ee4c9ebe47a14af6c66ec53e6d84dc15936e5ffdfac82f \
    --hash=sha256:f0906082d9932ae0c0057fa194041c22b4e2cdb46b2592ef3b91f020d62a081a \
    --hash=sha256:f2f7278c05c22fd60cb436cda1269649b3e81db65ecdc8496e5e164aa4143e7b \
    --hash=sha256:fb4fadde1b2b70bf4c0c14a6dccbe7194b1c28947fefd5bbe3fed9d940676c3b
    # via
    #   -c esm2-protein-pipeline lock (versions only)
    #   huggingface-hub
huggingface-hub==0.36.2 \
    --hash=sha256:1934304d2fb224f8afa3b87007d58501acfda9215b334eed53072dd5e815ff7a \
    --hash=sha256:48f0c8eac16145dfce371e9d2d7772854a4f591bcb56c9cf548accf531d54270
    # via
    #   -c esm2-protein-pipeline lock (versions only)
    #   gliner-ner-pipeline (pyproject.toml)
    #   gliner
    #   tokenizers
    #   transformers
idna==3.20 \
    --hash=sha256:a7db850025b95ded1eae8a46181a1a6c56c92c96f0e2b005d9ff8dc0210cab44 \
    --hash=sha256:ab7ae7122974553370f0bdb919e1a960b2cd1bc1ef0276416d896db81c14582c
    # via
    #   -c esm2-protein-pipeline lock (versions only)
    #   requests
jinja2==3.1.6 \
    --hash=sha256:0137fb05990d35f1275a587e9aee6d56da821fc83491a0fb838183be43f66d6d \
    --hash=sha256:85ece4451f492d0c13c5dd7c13a64681a86afae63a5f347908daf103ce6d2f67
    # via
    #   -c esm2-protein-pipeline lock (versions only)
    #   torch
markupsafe==3.0.3 \
    --hash=sha256:0303439a41979d9e74d18ff5e2dd8c43ed6c6001fd40e5bf2e43f7bd9bbc523f \
    --hash=sha256:068f375c472b3e7acbe2d5318dea141359e6900156b5b2ba06a30b169086b91a \
    --hash=sha256:0bf2a864d67e76e5c9a34dc26ec616a66b9888e25e7b9460e1c76d3293bd9dbf \
    --hash=sha256:0db14f5dafddbb6d9208827849fad01f1a2609380add406671a26386cdf15a19 \
    --hash=sha256:0eb9ff8191e8498cca014656ae6b8d61f39da5f95b488805da4bb029cccbfbaf \
    --hash=sha256:0f4b68347f8c5eab4a13419215bdfd7f8c9b19f2b25520968adfad23eb0ce60c \
    --hash=sha256:1085e7fbddd3be5f89cc898938f42c0b3c711fdcb37d75221de2666af647c175 \
    --hash=sha256:116bb52f642a37c115f517494ea5feb03889e04df47eeff5b130b1808ce7c219 \
    --hash=sha256:12c63dfb4a98206f045aa9563db46507995f7ef6d83b2f68eda65c307c6829eb \
    --hash=sha256:133a43e73a802c5562be9bbcd03d090aa5a1fe899db609c29e8c8d815c5f6de6 \
    --hash=sha256:1353ef0c1b138e1907ae78e2f6c63ff67501122006b0f9abad68fda5f4ffc6ab \
    --hash=sha256:15d939a21d546304880945ca1ecb8a039db6b4dc49b2c5a400387cdae6a62e26 \
    --hash=sha256:177b5253b2834fe3678cb4a5f0059808258584c559193998be2601324fdeafb1 \
    --hash=sha256:1872df69a4de6aead3491198eaf13810b565bdbeec3ae2dc8780f14458ec73ce \
    --hash=sha256:1b4b79e8ebf6b55351f0d91fe80f893b4743f104bff22e90697db1590e47a218 \
    --hash=sha256:1b52b4fb9df4eb9ae465f8d0c228a00624de2334f216f178a995ccdcf82c4634 \
    --hash=sha256:1ba88449deb3de88bd40044603fafffb7bc2b055d626a330323a9ed736661695 \
    --hash=sha256:1cc7ea17a6824959616c525620e387f6dd30fec8cb44f649e31712db02123dad \
    --hash=sha256:218551f6df4868a8d527e3062d0fb968682fe92054e89978594c28e642c43a73 \
    --hash=sha256:26a5784ded40c9e318cfc2bdb30fe164bdb8665ded9cd64d500a34fb42067b1c \
    --hash=sha256:2713baf880df847f2bece4230d4d094280f4e67b1e813eec43b4c0e144a34ffe \
    --hash=sha256:2a15a08b17dd94c53a1da0438822d70ebcd13f8c3a95abe3a9ef9f11a94830aa \
    --hash=sha256:2f981d352f04553a7171b8e44369f2af4055f888dfb147d55e42d29e29e74559 \
    --hash=sha256:32001d6a8fc98c8cb5c947787c5d08b0a50663d139f1305bac5885d98d9b40fa \
    --hash=sha256:3524b778fe5cfb3452a09d31e7b5adefeea8c5be1d43c4f810ba09f2ceb29d37 \
    --hash=sha256:3537e01efc9d4dccdf77221fb1cb3b8e1a38d5428920e0657ce299b20324d758 \
    --hash=sha256:35add3b638a5d900e807944a078b51922212fb3dedb01633a8defc4b01a3c85f \
    --hash=sha256:38664109c14ffc9e7437e86b4dceb442b0096dfe3541d7864d9cbe1da4cf36c8 \
    --hash=sha256:3a7e8ae81ae39e62a41ec302f972ba6ae23a5c5396c8e60113e9066ef893da0d \
    --hash=sha256:3b562dd9e9ea93f13d53989d23a7e775fdfd1066c33494ff43f5418bc8c58a5c \
    --hash=sha256:457a69a9577064c05a97c41f4e65148652db078a3a509039e64d3467b9e7ef97 \
    --hash=sha256:4bd4cd07944443f5a265608cc6aab442e4f74dff8088b0dfc8238647b8f6ae9a \
    --hash=sha256:4e885a3d1efa2eadc93c894a21770e4bc67899e3543680313b09f139e149ab19 \
    --hash=sha256:4faffd047e07c38848ce017e8725090413cd80cbc23d86e55c587bf979e579c9 \
    --hash=sha256:509fa21c6deb7a7a273d629cf5ec029bc209d1a51178615ddf718f5918992ab9 \
    --hash=sha256:5678211cb9333a6468fb8d8be0305520aa073f50d17f089b5b4b477ea6e67fdc \
    --hash=sha256:591ae9f2a647529ca990bc681daebdd52c8791ff06c2bfa05b65163e28102ef2 \
    --hash=sha256:5a7d5dc5140555cf21a6fefbdbf8723f06fcd2f63ef108f2854de715e4422cb4 \
    --hash=sha256:69c0b73548bc525c8cb9a251cddf1931d1db4d2258e9599c28c07ef3580ef354 \
    --hash=sha256:6b5420a1d9450023228968e7e6a9ce57f65d148ab56d2313fcd589eee96a7a50 \
    --hash=sha256:722695808f4b6457b320fdc131280796bdceb04ab50fe1795cd540799ebe1698 \
    --hash=sha256:729586769a26dbceff69f7a7dbbf59ab6572b99d94576a5592625d5b411576b9 \
    --hash=sha256:77f0643abe7495da77fb436f50f8dab76dbc6e5fd25d39589a0f1fe6548bfa2b \
    --hash=sha256:795e7751525cae078558e679d646ae45574b47ed6e7771863fcc079a6171a0fc \
    --hash=sha256:7be7b61bb172e1ed687f1754f8e7484f1c8019780f6f6b0786e76bb01c2ae115 \
    --hash=sha256:7c3fb7d25180895632e5d3148dbdc29ea38ccb7fd210aa27acbd1201a1902c6e \
    --hash=sha256:7e68f88e5b8799aa49c85cd116c932a1ac15caaa3f5db09087854d218359e485 \
    --hash=sha256:83891d0e9fb81a825d9a6d61e3f07550ca70a076484292a70fde82c4b807286f \
    --hash=sha256:8485f406a96febb5140bfeca44a73e3ce5116b2501ac54fe953e488fb1d03b12 \
    --hash=sha256:8709b08f4a89aa7586de0aadc8da56180242ee0ada3999749b183aa23df95025 \
    --hash=sha256:8f71bc33915be5186016f675cd83a1e08523649b0e33efdb898db577ef5bb009 \
    --hash=sha256:915c04ba3851909ce68ccc2b8e2cd691618c4dc4c4232fb7982bca3f41fd8c3d \
    --hash=sha256:949b8d66bc381ee8b007cd945914c721d9aba8e27f71959d750a46f7c282b20b \
    --hash=sha256:94c6f0bb423f739146aec64595853541634bde58b2135f27f61c1ffd1cd4d16a \
    --hash=sha256:9a1abfdc021a164803f4d485104931fb8f8c1efd55bc6b748d2f5774e78b62c5 \
    --hash=sha256:9b79b7a16f7fedff2495d684f2b59b0457c3b493778c9eed31111be64d58279f \
    --hash=sha256:a320721ab5a1aba0a233739394eb907f8c8da5c98c9181d1161e77a0c8e36f2d \
    --hash=sha256:a4afe79fb3de0b7097d81da19090f4df4f8d3a2b3adaa8764138aac2e44f3af1 \
    --hash=sha256:ad2cf8aa28b8c020ab2fc8287b0f823d0a7d8630784c31e9ee5edea20f406287 \
    --hash=sha256:b8512a91625c9b3da6f127803b166b629725e68af71f8184ae7e7d54686a56d6 \
    --hash=sha256:bc51efed119bc9cfdf792cdeaa4d67e8f6fcccab66ed4bfdd6bde3e59bfcbb2f \
    --hash=sha256:bdc919ead48f234740ad807933cdf545180bfbe9342c2bb451556db2ed958581 \
    --hash=sha256:bdd37121970bfd8be76c5fb069c7751683bdf373db1ed6c010162b2a130248ed \
    --hash=sha256:be8813b57049a7dc738189df53d69395eba14fb99345e0a5994914a3864c8a4b \
    --hash=sha256:c0c0b3ade1c0b13b936d7970b1d37a57acde9199dc2aecc4c336773e1d86049c \
    --hash=sha256:c47a551199eb8eb2121d4f0f15ae0f923d31350ab9280078d1e5f12b249e0026 \
    --hash=sha256:c4ffb7ebf07cfe8931028e3e4c85f0357459a3f9f9490886198848f4fa002ec8 \
    --hash=sha256:ccfcd093f13f0f0b7fdd0f198b90053bf7b2f02a3927a30e63f3ccc9df56b676 \
    --hash=sha256:d2ee202e79d8ed691ceebae8e0486bd9a2cd4794cec4824e1c99b6f5009502f6 \
    --hash=sha256:d53197da72cc091b024dd97249dfc7794d6a56530370992a5e1a08983ad9230e \
    --hash=sha256:d6dd0be5b5b189d31db7cda48b91d7e0a9795f31430b7f271219ab30f1d3ac9d \
    --hash=sha256:d88b440e37a16e651bda4c7c2b930eb586fd15ca7406cb39e211fcff3bf3017d \
    --hash=sha256:de8a88e63464af587c950061a5e6a67d3632e36df62b986892331d4620a35c01 \
    --hash=sha256:df2449253ef108a379b8b5d6b43f4b1a8e81a061d6537becd5582fba5f9196d7 \
    --hash=sha256:e1c1493fb6e50ab01d20a22826e57520f1284df32f2d8601fdd90b6304601419 \
    --hash=sha256:e1cf1972137e83c5d4c136c43ced9ac51d0e124706ee1c8aa8532c1287fa8795 \
    --hash=sha256:e2103a929dfa2fcaf9bb4e7c091983a49c9ac3b19c9061b6d5427dd7d14d81a1 \
    --hash=sha256:e56b7d45a839a697b5eb268c82a71bd8c7f6c94d6fd50c3d577fa39a9f1409f5 \
    --hash=sha256:e8afc3f2ccfa24215f8cb28dcf43f0113ac3c37c2f0f0806d8c70e4228c5cf4d \
    --hash=sha256:e8fc20152abba6b83724d7ff268c249fa196d8259ff481f3b1476383f8f24e42 \
    --hash=sha256:eaa9599de571d72e2daf60164784109f19978b327a3910d3e9de8c97b5b70cfe \
    --hash=sha256:ec15a59cf5af7be74194f7ab02d0f59a62bdcf1a537677ce67a2537c9b87fcda \
    --hash=sha256:f190daf01f13c72eac4efd5c430a8de82489d9cff23c364c3ea822545032993e \
    --hash=sha256:f34c41761022dd093b4b6896d4810782ffbabe30f2d443ff5f083e0cbbb8c737 \
    --hash=sha256:f3e98bb3798ead92273dc0e5fd0f31ade220f59a266ffd8a4f6065e0a3ce0523 \
    --hash=sha256:f42d0984e947b8adf7dd6dde396e720934d12c506ce84eea8476409563607591 \
    --hash=sha256:f71a396b3bf33ecaa1626c255855702aca4d3d9fea5e051b41ac59a9c1c41edc \
    --hash=sha256:f9e130248f4462aaa8e2552d547f36ddadbeaa573879158d721bbd33dfe4743a \
    --hash=sha256:fed51ac40f757d41b7c48425901843666a6677e3e8eb0abcff09e4ba6e664f50
    # via
    #   -c esm2-protein-pipeline lock (versions only)
    #   jinja2
mpmath==1.3.0 \
    --hash=sha256:7a28eb2a9774d00c7bc92411c19a89209d5da7c4c9a9e227be8330a23a25b91f \
    --hash=sha256:a0b2b9fe80bbcd81a6647ff13108738cfb482d481d826cc0e02f5b35e5c88d2c
    # via
    #   -c esm2-protein-pipeline lock (versions only)
    #   sympy
networkx==3.7 \
    --hash=sha256:e3fd2c13a7814cee3746340d8d7f8598a67f16a58bf47fb7f8793fab6efca1b0 \
    --hash=sha256:fd77a511bd90f39f3d016351345b52cf5319b813bdca01de3f755d3cca62e96a
    # via
    #   -c esm2-protein-pipeline lock (versions only)
    #   torch
numpy==2.5.3 \
    --hash=sha256:012e66aca395d795496446e52aeeb5866312a5d4d3f27da270e5a0b43f70dc5c \
    --hash=sha256:09d5a423c71ad5feb5625844ad58050e35df43871004b52ac9c0ad44a56775be \
    --hash=sha256:09ffa5d903faeaa5c4dd05009cf81c8bab9f2cb37c548b8d39b65b4cfa7c97f7 \
    --hash=sha256:0a59a421a32580a009e8a1751345bf829631b990dc1794b80514ab722b435def \
    --hash=sha256:116f96cadd935c6122e9228d676fe7ede19e741f5c8bb1c3cddbe0c51ccebea2 \
    --hash=sha256:1302b90c0e52281681b2975adfe8a860cb7b12216a27b4b0b4207c44bf7bccf0 \
    --hash=sha256:15aa985ac73a8db02db7663381aa109510449d3819d37206caed27b33a65a8a6 \
    --hash=sha256:1aad64d99730d013cfc6debafed22783b4fc5a7f4b8bc744d2d8cf7dcc880551 \
    --hash=sha256:1c80eabb4035ecf4ca9cd49cde8a9fdd69a729e63e6474887d1523ade7aa277f \
    --hash=sha256:1f3ed25271581281f2fccb1adcedfcde4c07362eec69189b50baf6f90e3ae159 \
    --hash=sha256:1fb6f8fb9ff0b3a69f52c66ce397b0246583e9f28616231b0e32ca49259a5fa6 \
    --hash=sha256:214045a5bf00113a146ab9ee9730c44501af6723cdf1f6830932f7b5ef2e7af0 \
    --hash=sha256:26e15e4aecd8617dfbaecb37d223e365d7b39411fba20454be2670a96aa74cb5 \
    --hash=sha256:2c25dfa72943e4336ddb6b0ee4277b47a0c85bede0807530ec68103bf58e2c10 \
    --hash=sha256:2d8240cb4c16fd831074aa2b2cf9fc54664d826341d61c372245b96a74a49a9a \
    --hash=sha256:350ba9783ce969cf9f7ce6e6a9a58e1a6e2a19ca025b7ee448c4db727706212a \
    --hash=sha256:4c8a6d2ebce6305fd82fbefca827775437147052a976ee7c94b36a0c1b52ac6c \
    --hash=sha256:4f8929ee6c96bfbd7b4ed2032e0c03af86fe1826740ab61ddabf9072d06e57ff \
    --hash=sha256:536f963710a4e63934d80ac0dc4f478804a83e9a84b6828018f25d09953ada33 \
    --hash=sha256:54a115e5a73b8fc44f0cebef486365a1894b5c9760685d4558b72b7c3eb846e0 \
    --hash=sha256:595d020938c84e320bcf40ad71089e108eac0d377cd018e14a8c094f39e98d85 \
    --hash=sha256:66a78fe4556c60aceda5916f9eacd638b18e9e681016ec302dcb4682d6d4d034 \
    --hash=sha256:6b05c171afb3aa07adbd20abc00aea86fe375beb0fdb9ef780ec5b7f63bab1c0 \
    --hash=sha256:6cef4bb1706dfec49243c05d921eefb4e190d41e2528b30d8035ea1f36b4c24a \
    --hash=sha256:6f24021b9f22bc6301c37b196974a92c1c18dccedb6fef3dd252e95f2d6adbe4 \
    --hash=sha256:71b39d9f935b6ec0f8753e3e2afb51e3efba6f2e05b68b32a40754d24bcd4a3c \
    --hash=sha256:71cad2b2a7451ab79d8f5e71b453485b6775963d5cf794179144a7463fe6e8ec \
    --hash=sha256:76c2c1e6bfa5c84adc6434dfbf013aa92096a7985221762c8f11fedfd20fff58 \
    --hash=sha256:8617bbfae4486cf99c9f899966699428d19da931d06ca94ad3da986c76e15997 \
    --hash=sha256:86bff898a431c0fb71f7610b75726e75a54d47b37edc9d537f48de63bb3c0b90 \
    --hash=sha256:8e4dd766076855b5ff7ea52fa5f07ce26286726e0f8bff446b7739d02e6ea204 \
    --hash=sha256:92f30e89b8ee0ecf363033576c422b2f58fed6a80bed0aa48dff6d14c654663e \
    --hash=sha256:93e1f5447e2b1e479d7bd74701e84746b86450cff1fc368b132d195e2b8f8211 \
    --hash=sha256:9a37475425b431b4d060f23b4f52cd2f3aef6bc7c654bd760adf0040eec9d435 \
    --hash=sha256:9deb49575e5b0b94ed72c8a64ec4d033381adc27e9060ae842971f697ba96104 \
    --hash=sha256:a5fa86b80fd24bcd1aff83ad23be44ea323de3f787be8f8b15d4a65621e25321 \
    --hash=sha256:a6391fafaba97500887132cd582abc6e19452b1ac775a47caa7b24490e152058 \
    --hash=sha256:a72f874bc9e10e4b8f80426fb49716d5141f64442a0c8418065093ec8017fbb0 \
    --hash=sha256:ac7bb1c52d445bd4f8f7f97fefe6abc3a084dc4d63df50d79b17fa2b78e89297 \
    --hash=sha256:adc1ada2662f8a5f960b8a10d9986897e7499ef07e06d4cfe7197f8cce923c07 \
    --hash=sha256:b00eefbcf0f292945c4b4dec2ae845389ef5bcdcd596e6e4328051db5b5ba694 \
    --hash=sha256:b0521d0f4aebb6e06189451025fa17a913287b13c03d5fe05c017333b654ea5b \
    --hash=sha256:b5d93cf48f687479941d12b69c873ad2cc76bbd487f0091c2200636497f34034 \
    --hash=sha256:b7e18c623bb5c95acb3b3328861272816ba199fb531921c5d6d0b675f1fde9e3 \
    --hash=sha256:bd4cb9ad3c7889b9b3fe0a9a9fb5d2ed26f9879bff2608d9f01aed147a20d231 \
    --hash=sha256:be5a8381859b6da607c84f4f7d6847725f1cf1853ef8a2c9e115b7d58bef47dc \
    --hash=sha256:befa1ae5bd6030b3f512b43ff3fa5290bbed6b84411a44244b14adf835f5b89d \
    --hash=sha256:bf63afbe037eb5d2fe87fbcc7778e61da53ebaf21d938a4515aa73b62532a5d4 \
    --hash=sha256:c00abe94c1a69d75d827dcf1c025b25c8a45d230b3bcd77a9020883a1b047653 \
    --hash=sha256:c2381f82999704f818e2c987a865050e285ec3621262c66d40f5a96c8f899f8e \
    --hash=sha256:c76d5dde9f445058f83d0c02af00557a4db91de9a9a57c0df87d1535001d654b \
    --hash=sha256:cb189f09db39283b26bfd061ec16189e14f71c6755207f72a0f7540867afe5b9 \
    --hash=sha256:ccb32e0525d29e8b0572eb84c9a57af0e7a4e615726927506f55063c62414034 \
    --hash=sha256:ccbc4665079665c3cf3bab4db9f6b095370cd6437d66be549b6c2a1fd19e1958 \
    --hash=sha256:d1c89973648c85069c5046ad460f7b8a00218b29a2e42359ac8cc63e9ab94832 \
    --hash=sha256:df2d5874ff183595a4ba404edd04f6bd9b5505c1d7708573f6a6c17489a67563 \
    --hash=sha256:e01c918ac3d48e18a927cf7b14a26a3e29ff2bdf2eacb976da0aecd6a43ed034 \
    --hash=sha256:e6ab667ba76450084eb64013762c438ea76d9d29cc676dcd6c2e9892ba37f841 \
    --hash=sha256:e931e4f499e0dc7ef29d269a8e5b35dd722e5d14be07df6240166ea7c6532fae \
    --hash=sha256:f54660b0eb6b0b9f36e7fe1cdfdff472028dd0d14acd9b9b65098efbad059469 \
    --hash=sha256:f59a878c33d6b88122d80d239bb3b845d58708750b0cb06a09aebb9b18ec696c \
    --hash=sha256:f7fabeb6cea87d65f3b926de33d03fb016cfdc29314c90974383b5582ae72891 \
    --hash=sha256:f9579f383d1bf9df80081e72760e84960a7fd4f88cf0c9e535a8597c9bb646f5 \
    --hash=sha256:f9a2353b37a1a9e78fd82b27ad7e2a32a2d036604d18f02b05e3136c62ca3b09 \
    --hash=sha256:fc36dc566135b5eceec4cf89758fcb719266a019ef07dae1754ae7c9f617ef3e \
    --hash=sha256:ffdc76bfcae6b255dff75202c5e7feaf95b40246bc0a17944facc1fecf9f79ab
    # via
    #   -c esm2-protein-pipeline lock (versions only)
    #   gliner-ner-pipeline (pyproject.toml)
    #   gliner
    #   torchvision
    #   transformers
nvidia-cublas==13.1.1.3 \
    --hash=sha256:37936a16db8fe4ac1f065c2139360608a543a09275cb1a1af612e08cfa065436 \
    --hash=sha256:b6cdce694e47ff6aadf0a69df1cab6628d696f5ff56e8d16af50309d855fa20f \
    --hash=sha256:b7a210458267ac818974c53038fbec2e969d5c99f305ab15c72522fa9f001dd5
    # via
    #   -c esm2-protein-pipeline lock (versions only)
    #   cuda-toolkit
    #   nvidia-cudnn-cu13
    #   nvidia-cusolver
nvidia-cuda-cupti==13.0.85 \
    --hash=sha256:4eb01c08e859bf924d222250d2e8f8b8ff6d3db4721288cf35d14252a4d933c8 \
    --hash=sha256:683f58d301548deeefcb8f6fac1b8d907691b9d8b18eccab417f51e362102f00 \
    --hash=sha256:796bd679890ee55fb14a94629b698b6db54bcfd833d391d5e94017dd9d7d3151
    # via
    #   -c esm2-protein-pipeline lock (versions only)
    #   cuda-toolkit
nvidia-cuda-nvrtc==13.0.88 \
    --hash=sha256:6bcd4e7f8e205cbe644f5a98f2f799bef9556fefc89dd786e79a16312ce49872 \
    --hash=sha256:ad9b6d2ead2435f11cbb6868809d2adeeee302e9bb94bcf0539c7a40d80e8575 \
    --hash=sha256:d27f20a0ca67a4bb34268a5e951033496c5b74870b868bacd046b1b8e0c3267b
    # via
    #   -c esm2-protein-pipeline lock (versions only)
    #   cuda-toolkit
    #   nvidia-cublas
nvidia-cuda-runtime==13.0.96 \
    --hash=sha256:7f82250d7782aa23b6cfe765ecc7db554bd3c2870c43f3d1821f1d18aebf0548 \
    --hash=sha256:ef9bcbe90493a2b9d810e43d249adb3d02e98dd30200d86607d8d02687c43f55 \
    --hash=sha256:f79298c8a098cec150a597c8eba58ecdab96e3bdc4b9bc4f9983635031740492
    # via
    #   -c esm2-protein-pipeline lock (versions only)
    #   cuda-toolkit
nvidia-cudnn-cu13==9.24.0.43 \
    --hash=sha256:67a7273b5cf062f9446fd76cf464351a1c0f66501e6cd78f6675c0d604d8ac87 \
    --hash=sha256:71f181cd810e90f9b6023b01186fe82d13d65f0ec098581ee201d39fad769e4b \
    --hash=sha256:a6812a554a1ff0413e9c52b84c26c050380649ab9615f9c16bded368ce9f421f
    # via
    #   -c esm2-protein-pipeline lock (versions only)
    #   torch
nvidia-cufft==12.0.0.61 \
    --hash=sha256:2708c852ef8cd89d1d2068bdbece0aa188813a0c934db3779b9b1faa8442e5f5 \
    --hash=sha256:2abce5b39d2f5ae12730fb7e5db6696533e36c26e2d3e8fd1750bdd2853364eb \
    --hash=sha256:6c44f692dce8fd5ffd3e3df134b6cdb9c2f72d99cf40b62c32dde45eea9ddad3
    # via
    #   -c esm2-protein-pipeline lock (versions only)
    #   cuda-toolkit
nvidia-cufile==1.15.1.6 \
    --hash=sha256:08a3ecefae5a01c7f5117351c64f17c7c62efa5fffdbe24fc7d298da19cd0b44 \
    --hash=sha256:bdc0deedc61f548bddf7733bdc216456c2fdb101d020e1ab4b88d232d5e2f6d1
    # via
    #   -c esm2-protein-pipeline lock (versions only)
    #   cuda-toolkit
nvidia-curand==10.4.0.35 \
    --hash=sha256:133df5a7509c3e292aaa2b477afd0194f06ce4ea24d714d616ff36439cee349a \
    --hash=sha256:1aee33a5da6e1db083fe2b90082def8915f30f3248d5896bcec36a579d941bfc \
    --hash=sha256:65b1710aa6961d326b411e314b374290904c5ddf41dc3f766ebc3f1d7d4ca69f
    # via
    #   -c esm2-protein-pipeline lock (versions only)
    #   cuda-toolkit
nvidia-cusolver==12.0.4.66 \
    --hash=sha256:02c2457eaa9e39de20f880f4bd8820e6a1cfb9f9a34f820eb12a155aa5bc92d2 \
    --hash=sha256:0a759da5dea5c0ea10fd307de75cdeb59e7ea4fcb8add0924859b944babf1112 \
    --hash=sha256:16515bd33a8e76bb54d024cfa068fa68d30e80fc34b9e1090813ea9362e0cb65
    # via
    #   -c esm2-protein-pipeline lock (versions only)
    #   cuda-toolkit
nvidia-cusparse==12.6.3.3 \
    --hash=sha256:2b3c89c88d01ee0e477cb7f82ef60a11a4bcd57b6b87c33f789350b59759360b \
    --hash=sha256:80bcc4662f23f1054ee334a15c72b8940402975e0eab63178fc7e670aa59472c \
    --hash=sha256:cbcf42feb737bd7ec15b4c0a63e62351886bd3f975027b8815d7f720a2b5ea79
    # via
    #   -c esm2-protein-pipeline lock (versions only)
    #   cuda-toolkit
    #   nvidia-cusolver
nvidia-cusparselt-cu13==0.8.1 \
    --hash=sha256:4dca476c50bf4780d46cd0bfbd82e2bc10a08e4fef7950917ce8d7578d22a23f \
    --hash=sha256:786ce87568c303fadb5afcc7102d454cd3040d75f6f8626f5db460d1871f4dd0 \
    --hash=sha256:dccbd362f91a7b9024d1f55ee9f548ac065027ff15d8c8b0db889ab3a8f31215
    # via
    #   -c esm2-protein-pipeline lock (versions only)
    #   torch
nvidia-nccl-cu13==2.30.7 \
    --hash=sha256:ca786ffa5a647c75d4d1f5cc72a6c4f537947e2ba8823d7c8aaf768e7a7b9f77 \
    --hash=sha256:cefa7fdb9710efd0f39c5f1be1d61ff6fc9a996c451265bd7fbdcf9455ed4b50
    # via
    #   -c esm2-protein-pipeline lock (versions only)
    #   torch
nvidia-nvjitlink==13.4.92 \
    --hash=sha256:25f74fad0d654271c921ac4dca614bd6258bc21791242fc7b2289dad7ae9c099 \
    --hash=sha256:9e4a7ff4f0cafa8c624917055b863dc11f5c2912ead23c462889e166f3b0e57d \
    --hash=sha256:b286f3a4f227a9363efdec263c7b91788cef1478d2b8a5fa8bab7f3e82ff82fd \
    --hash=sha256:e0391f24ed94ec879b84e3da4d4ec320c879aff681f2c7a638462f7199284323
    # via
    #   -c esm2-protein-pipeline lock (versions only)
    #   cuda-toolkit
    #   nvidia-cufft
    #   nvidia-cusolver
    #   nvidia-cusparse
nvidia-nvshmem-cu13==3.4.5 \
    --hash=sha256:290f0a2ee94c9f3687a02502f3b9299a9f9fe826e6d0287ee18482e78d495b80 \
    --hash=sha256:6dc2a197f38e5d0376ad52cd1a2a3617d3cdc150fd5966f4aee9bcebb1d68fe9
    # via
    #   -c esm2-protein-pipeline lock (versions only)
    #   torch
nvidia-nvtx==13.0.85 \
    --hash=sha256:4936d1d6780fbe68db454f5e72a42ff64d1fd6397df9f363ae786930fd5c1cd4 \
    --hash=sha256:cb7780edb6b14107373c835bf8b72e7a178bac7367e23da7acb108f973f157a6 \
    --hash=sha256:d66ea44254dd3c6eacc300047af6e1288d2269dd072b417e0adffbf479e18519
    # via
    #   -c esm2-protein-pipeline lock (versions only)
    #   cuda-toolkit
packaging==26.3 \
    --hash=sha256:94edc256424af38762eb31306eed28beb9f0efc50a8837492c9d6fd6004aed79 \
    --hash=sha256:d7193f7c8e4e93f444fde0262bf90af30e16fa0ad0ad44cb553c87339b23cd1c
    # via
    #   -c esm2-protein-pipeline lock (versions only)
    #   gliner
    #   huggingface-hub
    #   transformers
pillow==12.3.0 \
    --hash=sha256:00808c5e14ef63ac5161091d242999076604ff74b883423a11e5d7bbb38bf756 \
    --hash=sha256:04f01d28a6aaff387bf842a13be313df23ba0597a44f1a976c9feb3c6ff4711a \
    --hash=sha256:06ff022112bc9cbf83b60f8e028d94ad87b60621706487e65f673de61610ab59 \
    --hash=sha256:0740a512dc522224c77d9aa5a8d70d8b7d73fb91f2c21125d8d025d3b8990e45 \
    --hash=sha256:0847a763afefb695bc912d7c131e7e0632d4edc1d8698f58ddabec8e46b8b6d3 \
    --hash=sha256:0dd2064cbc55aaec028ef5fbb60fa47bb6c3e7918e07ff17935284b227a9d2df \
    --hash=sha256:0feb2e9d6ad6c9e3c06effe9d00f3f1e618a6643273576b016f591e9315a7139 \
    --hash=sha256:10e41f0fbf1eec8cfd234b8fe17a4caac7c9d0db4c204d3c173a8f9f6ef3232b \
    --hash=sha256:1182d52bc2d5e5d7d0949503aa7e36d12f42205dc287e4883f407b1988820d39 \
    --hash=sha256:164b31cd1a0490ab6efae01aa5df49da7061be0af1b30e035b6e9a1bfe34ee6e \
    --hash=sha256:1657923d2d45afb66526e5b933e5b3052e6bdea196c90d3abb2424e18c77dae8 \
    --hash=sha256:186941b6aef820ad110fb01fb06eb925374dc3a21b17e37ec9a53b250c6fe2d1 \
    --hash=sha256:1cca606cd25738df4ed873d5ad46bbdb3d83b5cbca291f6b4ff13a4df6b0bbe8 \
    --hash=sha256:21900ce7ba264168cd50defae43cd75d25c833ad4ad6e73ffc5596d12e25ac89 \
    --hash=sha256:236ff70b9312fb68943c703aa842ca6a758abfa45ac187a5e7c1452e96ef72b5 \
    --hash=sha256:23aceaa007d6172b02c277f0cd359c79492bbb14f7072b4ede9fbcaf20648130 \
    --hash=sha256:23d27a3e0307ec2244cc51e7287b919aa68d097504ebe19df4e76a98a3eea5bd \
    --hash=sha256:24870b09b224f7ae3c39ed07d10e819d06f8720bc551847b1d623832b5b0e28d \
    --hash=sha256:251bf95b67017e27b13d82f5b326234ca62d70f9cf4c2b9032de2358a3b12c7b \
    --hash=sha256:25b9b82bb22e6e2b3cd07b39c68b7b862001226cb3dff7130d1cb914121b39ed \
    --hash=sha256:28ce87c5ab450a9dd970b52e5aca5fe63ed432d18a2eaddd1979a00a1ba24ace \
    --hash=sha256:300557495eb45ebb8aec96c2da9c4be642fbf7cd937278b4013ba894ea8eb0eb \
    --hash=sha256:30f2aa603c41533cc25c05acd0da21636e84a315768feb631c937177db558931 \
    --hash=sha256:331b624368d4f1d069149002f25f44bc61c8919ce8ddb3c45bdad8f6e2d89510 \
    --hash=sha256:37d6d0a00072fd2948eb22bce7e1475f34569d90c87c59f7a2ec59541b77f7a6 \
    --hash=sha256:37dc8f7bbb66efe481bb60defacef820c950c24713fb44962ed6aa2a50966de1 \
    --hash=sha256:3b8182a766685eaa002637e28b4ec8d6b18819a0c71f579bf0dbaa5830297cce \
    --hash=sha256:3edce1d53195db527e0191f84b71d02022de0540bf43a16ed734ed7537b07385 \
    --hash=sha256:446c34dcc4324b084a53b705127dc15717b22c5e140ae0a3c38349d4efec071e \
    --hash=sha256:4998562bf62a445225f22e07c896bb04b35b1b1f2eb6d760584c9c51d7a5f78c \
    --hash=sha256:4b0a7fe987b14c31ebda6083f74f22b561fd3739bc0ac51e019622e3d72668c7 \
    --hash=sha256:4e8c2a84d977f50b9daed6eeaf3baef67d00d5d74d932288f02cb94518ee3ace \
    --hash=sha256:4f883547d4b7f0495ebe7056b0cc2aea76094e7a4abc8e933540f3271df27d9c \
    --hash=sha256:514435a37670e3e5e08f3945b68718b6ed329bb84367777e16f9f4dfe1e61a0f \
    --hash=sha256:53aa02d20d10c3d814d536aa4e5ac9b84ca0ff5a88377963b085ad6822f93e64 \
    --hash=sha256:5594fc43d548a7ed94949d139aa1341b270f1863f11cfd37f5a6c8b778a6b67f \
    --hash=sha256:571b9fcb07b97ef3a492028fb3d2dc0993ca23a06138b0315286566d29ef718a \
    --hash=sha256:57b3d78c95ba9059768b10e28b813002261d3f3dfc55cc48b0c988f625175827 \
    --hash=sha256:5afb51d599ea772b8365ae807ae557f18bccfe46ab261fd1c2a9ed700fc6eb17 \
    --hash=sha256:6b02afb9b97f65fbca5f31db6a2a3ba21aa93030225f150fa3f249717e938fb4 \
    --hash=sha256:6c0016e7b354317c4e9e525b937ac8596c38d2d232b419529b9cd7a1cd46e39a \
    --hash=sha256:71d6097b330eea8fd15097780c8e89cb1a8ce7838669f48c5bacd6f663dd4701 \
    --hash=sha256:756c768d0c9c2955feb7a56c37ea24aea2e369f8d36a88da270b6a9f19e62b5e \
    --hash=sha256:78cb2c6865a35ab8ff8b75fd122f6033b92a62c82801110e48ddd6c936a45d91 \
    --hash=sha256:7a743ff716f746fc19a9557f60dab1600d4613255f8a7aeb3cdde4db7eb15a66 \
    --hash=sha256:85f998ea1848bc6757289e739cfbdda3a04adfd58b02fc018ce54d754a5ce468 \
    --hash=sha256:8728f216dcdb6e6d555cf971cb34076139ad74b31fc2c14da4fafc741c5f6217 \
    --hash=sha256:877c3f311ff35410f690861c4409e7ccbf0cd2f878e50628a28e5a0bb689e658 \
    --hash=sha256:8cd2f7bdda092d99c9fc2fb7391354f306d01443d22785d0cbfafa2e2c8bb418 \
    --hash=sha256:8e95e1385e4998ae9694eeaa4730ba5457ff61185b3a55e2e7bea0880aef452a \
    --hash=sha256:962864dc93511324d51ddbb5b9f8731bf71675b93ca612a07441896f4688fb8c \
    --hash=sha256:9cf95fe4d0f84c82d282745d9bb08ad9f926efa00be4697e767b814ce40d4330 \
    --hash=sha256:9e881fca225083806662a5c43d627d215f258ff43c890f831966c7d7ba9c7402 \
    --hash=sha256:a2b55dd6b2a4c4b7d87ffa56bdb33fdc5fdb9a462173861a7bc097f17d91cb09 \
    --hash=sha256:a45650e8ce7fafffd731db8550230db6b0d306d181a90b67d3e6bca2f1990930 \
    --hash=sha256:a876864214e136f0eb367788dbd7df045f4806801518e2cfe9e13229cfe06d8f \
    --hash=sha256:ae26d61dfa7a47befdc7572b521024e8745f3d809bd95ca9505a7bba9ef849ec \
    --hash=sha256:af8d94b0db561cf68b88a267c5c44b49e134f525d0dc2cb7ed413a66bc23559a \
    --hash=sha256:b343699e8308bdc51978310e1c959c584e7869cc8c40780058c87da7781a1e94 \
    --hash=sha256:b3c777e849237620b022f7f297dd67705f9f5cf1685f09f02e46f93e92725468 \
    --hash=sha256:b629de27fda84b42cde7edef0d85f13b958b47f6e9bbcbba9b673c562a89bd8b \
    --hash=sha256:ba09209fbe443b4acccebe845d8a138b89a8f4fbaeedd44953490b5315d5e965 \
    --hash=sha256:ba54cfebe86920a559a7c4d6b9050791c20513650a1952ebe3368c7dc70306f8 \
    --hash=sha256:bcb46e2f9feff8d06323983bd83ed00c201fdcab3d74973e7072a889b3979fcd \
    --hash=sha256:bcc33feacfaefce60c12fd500a277533bdc02b10a19f7f6d348763d8140bbba7 \
    --hash=sha256:bf16ba1b4d0b6b7c8e534936632270cf70eb00dbe09005bc345b2677b726855c \
    --hash=sha256:cf1845d02ad822a369a49f2bb9345b1614744267682e7a03527dc3bf6eea1777 \
    --hash=sha256:d69141514cc30b774ceea5e3ed3a6635c8d8a96edf664689b890f4089111fb35 \
    --hash=sha256:d9c7f76c0673154f044e9d78c8655fb4213f6ca31a836df48b40fe5d187717b9 \
    --hash=sha256:dbce0b29841537a2fa4a214c2bbf14de3587c9680caa9b4e217568472490b28f \
    --hash=sha256:dc624f6bc473dacdf7ef7eb8678d0d08edf15cd94fad6ae5c7d6cc67a4e4902f \
    --hash=sha256:e158cb00350dc278f3b91551101aa7d12415a66ebf2c91d8d5ac14e56ddd3ad0 \
    --hash=sha256:e491916b378fba47242221bb9ead245211b70d504f495d105d17b14a24b4907c \
    --hash=sha256:e795b7eb908249c4e43c7c99fac7c2c75dab0c43566e37db472a355f63693d71 \
    --hash=sha256:e7e480451b9fa137494bccd3a7d69adbe8ac65a87d97be61e11f1b1050a5bac3 \
    --hash=sha256:e91206ee562682b51b98ef4b26a6ef48fd84e15fd4c4bc5ec768eb641d206838 \
    --hash=sha256:e9871b1ffbfa9656b60aeee92ed5136a5742696006fa322b29ea3d8da0ecc9cf \
    --hash=sha256:e9aeb04d6aef139de265b29683e119b638208f88cf73cdd1658aa07221165321 \
    --hash=sha256:ebaea975e03d3141d9d3a507df75c9b3ec90fa9d2ffd07567b3a978d9d790b26 \
    --hash=sha256:f0606c8bf2cdefea14a43530f7657cbbb7ecf1c4222512492ef4a4434a9501ec \
    --hash=sha256:f13c32a3abd6079a66d9526e18dad9b6d280384d49d7c54040cd57b6424041d9 \
    --hash=sha256:f7401aebd7f581d7f83a439d87d474999317ee099218e5ad25d125290990ba65 \
    --hash=sha256:fa4ecea169a355be7a3ade2c783e2ed12f0e40d2c5621cda8b3297faf7fbb9f5 \
    --hash=sha256:fbd139c8447d25dd750ab79ee274cc5e1fe80fc56340ab10b18a195e1b6eca3e \
    --hash=sha256:fdafc9cce40277e0f7a0feabce0ee50dd2fa1800f3b38015e51296b5e814048d \
    --hash=sha256:fe3cca2e4e8a592be0f269a1ca4835c25199d9f3ce815c8491048f785b0a0198 \
    --hash=sha256:ffd0c5368496f41b0944be820fcb7a838aa6e623d250b01acf2643939c3f99d7
    # via
    #   -c esm2-protein-pipeline lock (versions only)
    #   torchvision
protobuf==6.31.1 \
    --hash=sha256:0414e3aa5a5f3ff423828e1e6a6e907d6c65c1d5b7e6e975793d5590bdeecc16 \
    --hash=sha256:426f59d2964864a1a366254fa703b8632dcec0790d8862d30034d8245e1cd447 \
    --hash=sha256:4ee898bf66f7a8b0bd21bce523814e6fbd8c6add948045ce958b73af7e8878c6 \
    --hash=sha256:6f1227473dc43d44ed644425268eb7c2e488ae245d51c6866d19fe158e207402 \
    --hash=sha256:720a6c7e6b77288b85063569baae8536671b39f15cc22037ec7045658d80489e \
    --hash=sha256:7fa17d5a29c2e04b7d90e5e32388b8bfd0e7107cd8e616feef7ed3fa6bdab5c9 \
    --hash=sha256:8764cf4587791e7564051b35524b72844f845ad0bb011704c3736cce762d8fe9 \
    --hash=sha256:a40fc12b84c154884d7d4c4ebd675d5b3b5283e155f324049ae396b95ddebc39 \
    --hash=sha256:d8cac4c982f0b957a4dc73a80e2ea24fab08e679c0de9deb835f4a12d69aca9a
    # via gliner-ner-pipeline (pyproject.toml)
pyyaml==6.0.3 \
    --hash=sha256:00c4bdeba853cc34e7dd471f16b4114f4162dc03e6b7afcc2128711f0eca823c \
    --hash=sha256:0150219816b6a1fa26fb4699fb7daa9caf09eb1999f3b70fb6e786805e80375a \
    --hash=sha256:02893d100e99e03eda1c8fd5c441d8c60103fd175728e23e431db1b589cf5ab3 \
    --hash=sha256:02ea2dfa234451bbb8772601d7b8e426c2bfa197136796224e50e35a78777956 \
    --hash=sha256:0f29edc409a6392443abf94b9cf89ce99889a1dd5376d94316ae5145dfedd5d6 \
    --hash=sha256:10892704fc220243f5305762e276552a0395f7beb4dbf9b14ec8fd43b57f126c \
    --hash=sha256:16249ee61e95f858e83976573de0f5b2893b3677ba71c9dd36b9cf8be9ac6d65 \
    --hash=sha256:1d37d57ad971609cf3c53ba6a7e365e40660e3be0e5175fa9f2365a379d6095a \
    --hash=sha256:1ebe39cb5fc479422b83de611d14e2c0d3bb2a18bbcb01f229ab3cfbd8fee7a0 \
    --hash=sha256:214ed4befebe12df36bcc8bc2b64b396ca31be9304b8f59e25c11cf94a4c033b \
    --hash=sha256:2283a07e2c21a2aa78d9c4442724ec1eb15f5e42a723b99cb3d822d48f5f7ad1 \
    --hash=sha256:22ba7cfcad58ef3ecddc7ed1db3409af68d023b7f940da23c6c2a1890976eda6 \
    --hash=sha256:27c0abcb4a5dac13684a37f76e701e054692a9b2d3064b70f5e4eb54810553d7 \
    --hash=sha256:28c8d926f98f432f88adc23edf2e6d4921ac26fb084b028c733d01868d19007e \
    --hash=sha256:2e71d11abed7344e42a8849600193d15b6def118602c4c176f748e4583246007 \
    --hash=sha256:34d5fcd24b8445fadc33f9cf348c1047101756fd760b4dacb5c3e99755703310 \
    --hash=sha256:37503bfbfc9d2c40b344d06b2199cf0e96e97957ab1c1b546fd4f87e53e5d3e4 \
    --hash=sha256:3c5677e12444c15717b902a5798264fa7909e41153cdf9ef7ad571b704a63dd9 \
    --hash=sha256:3ff07ec89bae51176c0549bc4c63aa6202991da2d9a6129d7aef7f1407d3f295 \
    --hash=sha256:41715c910c881bc081f1e8872880d3c650acf13dfa8214bad49ed4cede7c34ea \
    --hash=sha256:418cf3f2111bc80e0933b2cd8cd04f286338bb88bdc7bc8e6dd775ebde60b5e0 \
    --hash=sha256:44edc647873928551a01e7a563d7452ccdebee747728c1080d881d68af7b997e \
    --hash=sha256:4a2e8cebe2ff6ab7d1050ecd59c25d4c8bd7e6f400f5f82b96557ac0abafd0ac \
    --hash=sha256:4ad1906908f2f5ae4e5a8ddfce73c320c2a1429ec52eafd27138b7f1cbe341c9 \
    --hash=sha256:501a031947e3a9025ed4405a168e6ef5ae3126c59f90ce0cd6f2bfc477be31b7 \
    --hash=sha256:5190d403f121660ce8d1d2c1bb2ef1bd05b5f68533fc5c2ea899bd15f4399b35 \
    --hash=sha256:5498cd1645aa724a7c71c8f378eb29ebe23da2fc0d7a08071d89469bf1d2defb \
    --hash=sha256:5cf4e27da7e3fbed4d6c3d8e797387aaad68102272f8f9752883bc32d61cb87b \
    --hash=sha256:5e0b74767e5f8c593e8c9b5912019159ed0533c70051e9cce3e8b6aa699fcd69 \
    --hash=sha256:5ed875a24292240029e4483f9d4a4b8a1ae08843b9c54f43fcc11e404532a8a5 \
    --hash=sha256:5fcd34e47f6e0b794d17de1b4ff496c00986e1c83f7ab2fb8fcfe9616ff7477b \
    --hash=sha256:5fdec68f91a0c6739b380c83b951e2c72ac0197ace422360e6d5a959d8d97b2c \
    --hash=sha256:6344df0d5755a2c9a276d4473ae6b90647e216ab4757f8426893b5dd2ac3f369 \
    --hash=sha256:64386e5e707d03a7e172c0701abfb7e10f0fb753ee1d773128192742712a98fd \
    --hash=sha256:652cb6edd41e718550aad172851962662ff2681490a8a711af6a4d288dd96824 \
    --hash=sha256:66291b10affd76d76f54fad28e22e51719ef9ba22b29e1d7d03d6777a9174198 \
    --hash=sha256:66e1674c3ef6f541c35191caae2d429b967b99e02040f5ba928632d9a7f0f065 \
    --hash=sha256:6adc77889b628398debc7b65c073bcb99c4a0237b248cacaf3fe8a557563ef6c \
    --hash=sha256:79005a0d97d5ddabfeeea4cf676af11e647e41d81c9a7722a193022accdb6b7c \
    --hash=sha256:7c6610def4f163542a622a73fb39f534f8c101d690126992300bf3207eab9764 \
    --hash=sha256:7f047e29dcae44602496db43be01ad42fc6f1cc0d8cd6c83d342306c32270196 \
    --hash=sha256:8098f252adfa6c80ab48096053f512f2321f0b998f98150cea9bd23d83e1467b \
    --hash=sha256:850774a7879607d3a6f50d36d04f00ee69e7fc816450e5f7e58d7f17f1ae5c00 \
    --hash=sha256:8d1fab6bb153a416f9aeb4b8763bc0f22a5586065f86f7664fc23339fc1c1fac \
    --hash=sha256:8da9669d359f02c0b91ccc01cac4a67f16afec0dac22c2ad09f46bee0697eba8 \
    --hash=sha256:8dc52c23056b9ddd46818a57b78404882310fb473d63f17b07d5c40421e47f8e \
    --hash=sha256:9149cad251584d5fb4981be1ecde53a1ca46c891a79788c0df828d2f166bda28 \
    --hash=sha256:93dda82c9c22deb0a405ea4dc5f2d0cda384168e466364dec6255b293923b2f3 \
    --hash=sha256:96b533f0e99f6579b3d4d4995707cf36df9100d67e0c8303a0c55b27b5f99bc5 \
    --hash=sha256:9c57bb8c96f6d1808c030b1687b9b5fb476abaa47f0db9c0101f5e9f394e97f4 \
    --hash=sha256:9c7708761fccb9397fe64bbc0395abcae8c4bf7b0eac081e12b809bf47700d0b \
    --hash=sha256:9f3bfb4965eb874431221a3ff3fdcddc7e74e3b07799e0e84ca4a0f867d449bf \
    --hash=sha256:a33284e20b78bd4a18c8c2282d549d10bc8408a2a7ff57653c0cf0b9be0afce5 \
    --hash=sha256:a80cb027f6b349846a3bf6d73b5e95e782175e52f22108cfa17876aaeff93702 \
    --hash=sha256:b30236e45cf30d2b8e7b3e85881719e98507abed1011bf463a8fa23e9c3e98a8 \
    --hash=sha256:b3bc83488de33889877a0f2543ade9f70c67d66d9ebb4ac959502e12de895788 \
    --hash=sha256:b865addae83924361678b652338317d1bd7e79b1f4596f96b96c77a5a34b34da \
    --hash=sha256:b8bb0864c5a28024fac8a632c443c87c5aa6f215c0b126c449ae1a150412f31d \
    --hash=sha256:ba1cc08a7ccde2d2ec775841541641e4548226580ab850948cbfda66a1befcdc \
    --hash=sha256:bdb2c67c6c1390b63c6ff89f210c8fd09d9a1217a465701eac7316313c915e4c \
    --hash=sha256:c1ff362665ae507275af2853520967820d9124984e0f7466736aea23d8611fba \
    --hash=sha256:c2514fceb77bc5e7a2f7adfaa1feb2fb311607c9cb518dbc378688ec73d8292f \
    --hash=sha256:c3355370a2c156cffb25e876646f149d5d68f5e0a3ce86a5084dd0b64a994917 \
    --hash=sha256:c458b6d084f9b935061bc36216e8a69a7e293a2f1e68bf956dcd9e6cbcd143f5 \
    --hash=sha256:d0eae10f8159e8fdad514efdc92d74fd8d682c933a6dd088030f3834bc8e6b26 \
    --hash=sha256:d76623373421df22fb4cf8817020cbb7ef15c725b9d5e45f17e189bfc384190f \
    --hash=sha256:ebc55a14a21cb14062aa4162f906cd962b28e2e9ea38f9b4391244cd8de4ae0b \
    --hash=sha256:eda16858a3cab07b80edaf74336ece1f986ba330fdb8ee0d6c0d68fe82bc96be \
    --hash=sha256:ee2922902c45ae8ccada2c5b501ab86c36525b883eff4255313a253a3160861c \
    --hash=sha256:efd7b85f94a6f21e4932043973a7ba2613b059c4a000551892ac9f1d11f5baf3 \
    --hash=sha256:f7057c9a337546edc7973c0d3ba84ddcdf0daa14533c2065749c9075001090e6 \
    --hash=sha256:fa160448684b4e94d80416c0fa4aac48967a969efe22931448d853ada8baf926 \
    --hash=sha256:fc09d0aa354569bc501d4e787133afc08552722d3ab34836a80547331bb5d4a0
    # via
    #   -c esm2-protein-pipeline lock (versions only)
    #   huggingface-hub
    #   transformers
regex==2026.9.29 \
    --hash=sha256:01000ddf0e3ffef97f2413ceb514f6313040106b6d18a03ee00a4fe35c1eb1db \
    --hash=sha256:0166844493626c5015c6088ee15c9ca2fd060ca15b7641d1657da6a58432ae33 \
    --hash=sha256:044265d77d94f5e3cb2fd72c76723807c429cb8c533e9d4672d0334a6f14f588 \
    --hash=sha256:0476e5bcbe6e1ba3d1c4cc7bbb1c3ba78e3b979b5c8a88d0a6a8cdd4992b8c84 \
    --hash=sha256:066d0e3dbfdd739bce2bf8c2a41dd16f73e3d8adc2eb06dd803a36a307f56075 \
    --hash=sha256:0b65c72739f981377c9c22e0c5c3cd7f42da7bd8a3c9209330fac772c7d893ed \
    --hash=sha256:0c992c19cd45058a4b92f68f139c93db168b48fb1f322c9a7cd620806afb6b51 \
    --hash=sha256:0cc63b5e47c12a48d90c7e9d7de6a035dd14f62868aaedbb4e0ff8ba2b8bfe7b \
    --hash=sha256:0dd8af32e9f7b56b7f95cc1fd79b23054c3bdc172392ae560acc24d57b7ffe71 \
    --hash=sha256:0def9fb6abac55492d6d51cddb7225d07d6f279e774e0adc08569a54a5fc8d46 \
    --hash=sha256:0fd2c901cc307a745ad4bc87f20060d7a0825a3371d1e93488af22e7a387f78f \
    --hash=sha256:1043aedf5917caa861bcb25a9c11460049656bdf0017a90a309fa8f255467725 \
    --hash=sha256:121a76a0985db80ceae9e171c337f8c927868e37d01b54e3ce87bc87f9c6a208 \
    --hash=sha256:143533cc4b6fbc5b95aca0a5b8d541088d374831593def000ec89322c220221d \
    --hash=sha256:14e953ff3607c92d7675bf79c4d4509ef6782aa8c08509f179f9b3d6d0679e86 \
    --hash=sha256:18ae8eed4526e35bdb754d61562b90bf5c00a67fdcf3cc1380dd59597486631b \
    --hash=sha256:19959129885356df0e97556856f77eb2888380dac18bed075a7c05c5128c618d \
    --hash=sha256:1ba8c6a416569ce0d37e83e28a254a61dc99a419084dfb6476cea02d997f74fa \
    --hash=sha256:1c2a0026062abcc321a53db4a185ceba0b59a66b5d37b0808917a88b55a5257f \
    --hash=sha256:1d9fe8091b2e89d470df68a9331111ed008ae8aae6bf1e8e1fba4086a495c84e \
    --hash=sha256:2089fe39c406784d90101c726755ffa1497bb74638fd434300d2b88006186de8 \
    --hash=sha256:23ae6fdad9e63e54038f5ef78aba2933faca61e24d432786589e737bc5522ebb \
    --hash=sha256:26ec4ccce55aa533fbd603d08911b01101a8fcfec987845ac3ae2c7087b2bde3 \
    --hash=sha256:2f7f7aa47b229f2b39a2ae2596d2ad5625d77b5eb9856fac2dab3eb506cdd0a0 \
    --hash=sha256:31b003f9a070335e2a8233ee9b14a3ca8e6d792012ae011f741bf0aaf11744c5 \
    --hash=sha256:32ab11df9677ca80bcbb5fe4eb1da9109a5019239a054836efc6fa1c64e683cf \
    --hash=sha256:33026515aebc0e70d1c89978e53e8d695d35d9e472f8d5b34465ba3c74028650 \
    --hash=sha256:34b6925af9853bf461950e6508910f179fd6e9b1a7ec8548e069606b7e51a26b \
    --hash=sha256:352cf115a810b357caa35193ab656ecf5ef41056855e82f292c99e8514f8d954 \
    --hash=sha256:39ab5894d971f9ac68baa6eca5c50387db579cfcacf36ae8df3feceb1815e6d0 \
    --hash=sha256:3a21a9509d0ee88e7a70e1ad228cd2f0e0fd1e187458db132e8a8d18c97daf9d \
    --hash=sha256:3c5c2ef13797466aa64170cbb66ad98a32351dd4127694cea7199f80f213750d \
    --hash=sha256:3e778bfccd63075167709136afbc251c1f683758d5bf49c803c60ac3f894ce6b \
    --hash=sha256:3f1e6cb402a89457582cd696f982559217d13484a193202c394015297968c86d \
    --hash=sha256:42e82e578c904445d4c8a35b8f28052cf567593215fa5db06266fbc6f77aaa2e \
    --hash=sha256:4408b2b27a95ca8cc48b7411945753773353b5c93b307754781086c99d3a576f \
    --hash=sha256:446654b29bfaa30500d80947eda42cef1449dc8a87f4e3cf061cc8485d3a1f0b \
    --hash=sha256:45010bcfe66df41522d56c9b6114e87ecc597a08970ff6a2ced24415c141ae5f \
    --hash=sha256:49ee178ca31c94621294bf9b8b676a92a2e6bba8af0529591753719e57edb621 \
    --hash=sha256:4d7d93613b01b0199961330e49cfc52d479b3d5776c56c691db31130c0a07d91 \
    --hash=sha256:4fb41211d2333eb930a51e0546a65999761cf1f572a4da56ef9b8a62966c06f2 \
    --hash=sha256:4fe97894d1b306c919b4e50def1e6f6c522f4d03a7283811f4d108f1ce5d3ac2 \
    --hash=sha256:554bffadcbcb6d5f4e5fb10a61cc52084b9a63d1dab5f10bcd2c4343972e8e2c \
    --hash=sha256:59b49507f47479e299a9e1bc41b5cb83a7afda0540625f1dbae886615978acbf \
    --hash=sha256:5eeb8edc6110d9194a4d0d54610f64c37a31c605b5dbb7e407fc6ec7fa34a4a1 \
    --hash=sha256:612b709381c0355b70d89cdb51b7f670591ed5cbbc0e3b5337488019dc667b65 \
    --hash=sha256:61956f074ecd123f55adca68ee3eab46e6a07ad3f8e64e6db95dfacb444f55c4 \
    --hash=sha256:6398d5145689503412cc1748895242598d8846b8967b851133b20dc2ed1e21e8 \
    --hash=sha256:65b408d8fcb273e3499e7ef2ce796810da1becd208c7fb4373692a242d79d461 \
    --hash=sha256:686ac5350fceae63830bb98805fcb8039325bf4c06d9f6f048ff65229d5bffa5 \
    --hash=sha256:6a1a824fbed817e0a891103886b68f063b1e83cc51bc97192a90a60195a9291f \
    --hash=sha256:6abb75ab16bc3281714a5b99548a2225db70dba1f995f6d7f7419b76eb5a8fbe \
    --hash=sha256:6f7121a8914ed13fcfe2099f895341bfb789f004d4c5a0bdece8fa667da10849 \
    --hash=sha256:7020ed44df30b3aa492c00ee3b52d0548c1f30c2c6c5bb13ae897680900d3413 \
    --hash=sha256:720537c7ea6f80dc61913184edb0ce2497a306b39ef19f28505b322553d52bdb \
    --hash=sha256:724184b4aafed865e4f13ca313fdcb43024300c028ec67319cfa16847d84685e \
    --hash=sha256:7c03031610e3e6ed1768a2b7a8fc84637c1257b50c5eacaf094c6e17a84fc563 \
    --hash=sha256:80a5ea3b4fd9d6a5b9a44f7976a9acaaab35aa3c1f6b29e5bd857dfabaded223 \
    --hash=sha256:80c7cadd3fd2bfde5df8aa0787e315812cad0c313a753095d02f4c2b6c01677b \
    --hash=sha256:80ea96f5c1a30bf09007d48466521d9c294bebe197c708c3359096e3e3691632 \
    --hash=sha256:864e9b87ac33c3fb9fb4ad48166d4fdb579c351d5c77deb0d34bccb36a775cd9 \
    --hash=sha256:87fb80cbe3557e27e7b28b995c2b2eedf689b8886f941ab93e0e288f0976518a \
    --hash=sha256:8873c4a11c50b9989168881aeb3f08859f469d809941866aa1feefd8be5431f6 \
    --hash=sha256:888d60953908dcf761aa320c3e390ab8556efbdb551ace63921de90f6ae0848d \
    --hash=sha256:8b5fcc4771732191b2b7d1dd68d8f0353f47f8d90b6150f6dce58bf1112442cb \
    --hash=sha256:8f39588af4731c8923c26810eb3b33f76f17633985e40f59c3cd45a33805a895 \
    --hash=sha256:9173db3be74a35cb6731701094b98120f7ee4876a287882a59cdea1fa7da342f \
    --hash=sha256:92f05c9c42bde5785dc48770bc2194d9f7442544156f951e19cd31b096cec562 \
    --hash=sha256:951733b1bbdb71e377cec567b409f1a7881b47cfcad84121aa74cb575fa425ea \
    --hash=sha256:957bb708e8057ab1649ba566456429d691ec9b90d1c9ad1af1ba7ffbbeaf05f2 \
    --hash=sha256:9916fda742cd4eede63b286f58c06718324265d727ce0856eb1aac86d0d150d6 \
    --hash=sha256:9e1d3a4cb7993b708f0ada8d0c84590efd853f169e7147d2202c9da503180242 \
    --hash=sha256:9e4482589065c8ecd761cff522dcd85f2d39e62f551e37e025d1c7d54772def3 \
    --hash=sha256:a5300757f8a68f5b6cc33f57338d72a0e3589c5cc9ad5f8504ea06f028be582a \
    --hash=sha256:a540abfab208e1b7ef2df231c40ef3b6cbb30a0aad6204e9b6a81c10a6794628 \
    --hash=sha256:a5758353650079898dc1b2b0e95aa51fa23a30d020e06f62c430dd08ee56cdd8 \
    --hash=sha256:a64b85a4760337cfefdb27d42da6ed8b58e8cde3f2d57b6ef43e76ef6ea9ef47 \
    --hash=sha256:a655d34b2a6943af32401f3d94f72e9d731f6ad16285815550bf2b4ee69d420a \
    --hash=sha256:a714befaacbd10092ffe4cea0d3c5f008fb9efe9bc322c715bcdfdee414b9a3d \
    --hash=sha256:a760da040b47767b4b873adfb7c3b691e9ba2fc60f113f9d0b88f1a62f323e85 \
    --hash=sha256:addd736a0547d553283adaf4e05d7104e7f2c7b0b092e9b4d28756825f14531f \
    --hash=sha256:ae4613d7d9dda60fcba95f846cc6f808017f1843f392cf9daad14a6534493d71 \
    --hash=sha256:b11b589e00095ec69cf79841a76360f9b079e95b0368a25b5ebb951ab0c157ff \
    --hash=sha256:b3e445b66c80b4eb4234e855ce94d9adc183eedbd632816228d89930b91b2c5b \
    --hash=sha256:b7b893976e7fe42053da64f2aa27239c24252fd2ec6df471e1be197c0addc3b1 \
    --hash=sha256:b84f186a7f0536fe4ff9a9fa12d06d007b9b71d4b5352ddcc41f59ad6522a312 \
    --hash=sha256:b89efc38431793d28b7cd91227e2f952ad7c48df19132b17f43a5fec3c14143b \
    --hash=sha256:b97a38fb4c732b6832db6bf108963adbcd82ef1268ba2025dce390f45af75efa \
    --hash=sha256:b9d74e4eee9ddb64c2e92d5d61472c59c21684c059eb7b68767be9628e977859 \
    --hash=sha256:bb90e7177944b6684738c1fc36aabd2dd00d1de3be7dbe09f91e196f1bc0dc81 \
    --hash=sha256:bec37990e3d6121f29ecfb594bd8f1bf009e9f7926daba2e50e3b27d3892a783 \
    --hash=sha256:bf3c49863c23a1ad6da9c30351aed6cff8d5ddbeb63c5c8420ae54e98c7d0138 \
    --hash=sha256:bf48516e35cf848390ea68850aba53e7c333720d2945b4d2c25b69fc5171723f \
    --hash=sha256:bfc71e6d970419c1309b3640305298643e2a734cad3f7cfb6d2ddee4175ab53d \
    --hash=sha256:c0094897d7d01f184b2d7fe8c56c66d64efe01b31f4b7d34205b391387df1111 \
    --hash=sha256:c03c6eb6ece86dfdcbb34799efaa339b093132e1aceed491ba5e08fe06cdf699 \
    --hash=sha256:c1a9a6651197fbed6f0212591418b9def774fc3f8324f78d1bf0e6a63e5f8aa1 \
    --hash=sha256:c3589f40749acce747510bf5d589d54e376cb0930ea58b35effac97e5312b0c1 \
    --hash=sha256:c4e38dd8f39c43a91d2410ad2b85610701b0979342c3df1d69eaf8e838c757d8 \
    --hash=sha256:c6c8fabf1dafc1f1ddcbb67896d3f93efb092e8c4b6322d7389b944e76a484e5 \
    --hash=sha256:c90fcf7804ea0a54b896ce0f2b9565350220b8d4890fd0db461a476a4c687963 \
    --hash=sha256:c9b602fae1e00b7c035d661ce85575365719192a7b46784bd71cf64c68053aa0 \
    --hash=sha256:ccb64d887a9db1cd76dbc0f92051a1a478a2a67e7f56c62d915cb881d7734704 \
    --hash=sha256:d06fcdecc10fc7954d7c8f27a03c96055fe525274dc84a7b0dbdc3d6b9e03dab \
    --hash=sha256:d0c3082bf79bcd6a614d55916590ad4b8f93200e10b97f463ea5d9d07c9b5f23 \
    --hash=sha256:d49c18f1ea294cf4adde2e5ac256e98c82ea9d708462ce4bf799dffa7cfe8a2c \
    --hash=sha256:d60030baaa7bfbb02d650c126cdcddcb6e33dbff14d819434c8fa2fdcaeeeba5 \
    --hash=sha256:d7cab119d0df0b9413f106b4d7fc34f2872d3574ed3806fb48959c830b1537da \
    --hash=sha256:d9b77b25b4f395f92de6099ab08e8ae2bc7e51dfe157f22900902243a5cc90c7 \
    --hash=sha256:dabee8f4935e731fb46b2a3091bdda0d3d94b3bbfb907d2b4f12eefce4009619 \
    --hash=sha256:db5e82ba15c142425b8406690032df89e39cca4a2e8afbbb9a3d84edc2373ac3 \
    --hash=sha256:dc79d36d0618752265f0d575915bdc5c5130ecb9c9f6b3bcefeae32e4bdfafcf \
    --hash=sha256:ddfa987262763c3c22a8367d2a49c244b018a74c3a8e3ab1a864119ad45c5633 \
    --hash=sha256:e1172147d28d8fbcf8cb8d26c41506169f5ad8fe9ec969cb116835a19d4d8eca \
    --hash=sha256:e11edba5bc344a32b029a7af9d4b3173982dd79eeafa0b9dbd787364414b0509 \
    --hash=sha256:e2c89e9b762c57f59d5e99ee8b20202adb892e35f8d3485741340999ca55058e \
    --hash=sha256:e31f72490b7c12f7790e1e25c3afffd20503ee1bfb43461d7838b871ff244b19 \
    --hash=sha256:e8c65ef3862a8ad6e86492b6ed9327805dd66904c012bd3649dc67d822ed6c34 \
    --hash=sha256:ebb8912f565b8cdbbf27debfe00df04202c20e2f651b9e32767930c5eace3621 \
    --hash=sha256:ed511a0708e2297e1d6431e7fb217e3402791e491e02da800658ace4973df1bb \
    --hash=sha256:edf06545875f3efa31560d94121e95c7fd70d98b1dfedc0157097d79b13b52ea \
    --hash=sha256:f0fe9834e5aeccaf19a0d8feb296d66a24be1a7c9922002f842a682cd5abb787 \
    --hash=sha256:f1a0d5117230dd46b399a30a38afa44f79c99f3168988fdc4f425c3f928b39df \
    --hash=sha256:f37964e4a5e993d2fd45147741e9dff7f34a2d8c00ab94c4ea0514a4677f959e \
    --hash=sha256:f57dc6b8fef170f105d2cf5cdce254f47b137d7755086cf7050f47e16582abba \
    --hash=sha256:f93bc1c3486ef3747e07c9d7c1d0a147b8fbaab975f80e348aed6f71309dfaca \
    --hash=sha256:fb00027a09a8f9f08028b40dce4c933cf73e4833240ed356583fdc9cfa721566 \
    --hash=sha256:fb99cc9d45f48895d9d67f6a0b8a57f08d39c174d9f25ad97a313e0470267b1c \
    --hash=sha256:fdd88ed5e20b1bcdd234421e454962c971aa44b653bdb7f1ea9ef683e90fb649 \
    --hash=sha256:fe3fa1dd453ed5c7f5ea23a26218329790ed7197a99b90e94330e313959a7f52
    # via
    #   -c esm2-protein-pipeline lock (versions only)
    #   transformers
requests==2.34.2 \
    --hash=sha256:2a0d60c172f83ac6ab31e4554906c0f3b3588d37b5cb939b1c061f4907e278e0 \
    --hash=sha256:f288924cae4e29463698d6d60bc6a4da69c89185ad1e0bcc4104f584e960b9ed
    # via
    #   -c esm2-protein-pipeline lock (versions only)
    #   huggingface-hub
    #   transformers
safetensors==0.8.0 \
    --hash=sha256:040070828e36dc8e122178bbbd5830ff9e97920affb84cbe0f46442497bed358 \
    --hash=sha256:096ec1a98435df7beb08853bb5aa9081a84f23d0adc67ed1a0a10550f608373f \
    --hash=sha256:2ddf52eac562eda224f99acfa7889d02968c1fd59a5b011ae7d8137c37e9c02d \
    --hash=sha256:3ae091f16662658bdc019a4ff6cb4c085bb7d725eb5978b183ffd265863b6d2d \
    --hash=sha256:4124502b78f03534117c848f87a39b8f31e577b15eff423bf8bfb95f2a8c30d0 \
    --hash=sha256:4a95ae2b05d7726d751da4ebf626a2ca782b706e101bd894c95bc2450b1cffcc \
    --hash=sha256:7a46e5ff292c356d6991e60942ba7f79817682d3a2cef0702136448cb9c4d235 \
    --hash=sha256:7bc0a787ba8a35be368ee3574edfa2b1ad389eebd0a72e482ae275490e3f6c98 \
    --hash=sha256:87eec7ffed2b809f05a398a8becb7d013f19f7837cd15d9748580d6cf30dbaf4 \
    --hash=sha256:8e080062fcde23be189565e1c3305d16751a218ecf9412c8601e64204eb6f846 \
    --hash=sha256:8e9f537aa183a38ace122d27303dcd986b26bd2a7591f9181d7f0c396f4677ca \
    --hash=sha256:c554f85858e05226d3c2828e32395e677434685d6d94594a41643361c5e837f0 \
    --hash=sha256:c80201d22cbf405b80647a60ada77bba06c8fba2da2743ba1e89cdcc39a81f25 \
    --hash=sha256:f7838e5135a406ad3e02efdcb8cf2e5397d368b0154537c4fec682dbc544d452 \
    --hash=sha256:fabaf3e0f18a6618d9b36560682562157f77c2b71fcffc7b432be2baed9d753d \
    --hash=sha256:fcdd41ec4628fee5799f807c73c353629130fbd942aa23d83c623dd6c9d52d78 \
    --hash=sha256:fd6f3f93c9a0a7cc2788ee63fb763353d4bd2e89b0751bc78fcf7dda00bea774
    # via
    #   -c esm2-protein-pipeline lock (versions only)
    #   gliner-ner-pipeline (pyproject.toml)
    #   gliner
    #   transformers
sentencepiece==0.2.2 \
    --hash=sha256:046b15ea22d8042e2e173561d464ec3b64a9c2081324df70ebce7bf7ebb3e497 \
    --hash=sha256:0e2aae42960392d6dcb9a72d8e1e65a97294c965071b43c7b3429a42f350250e \
    --hash=sha256:1120e0791540615e650b2e9bea835bf38a7362455d8ab62dee7968219c2d79a0 \
    --hash=sha256:1402d8ee36f0d851cea8eee4dbb85fea14643b7503cf4d00d102eec0fe3ca719 \
    --hash=sha256:1416b92f2f010333786fe6306ed2631121d5ea492219b0841e967b6765e64107 \
    --hash=sha256:16c84ddef8d3084a8af37208acd365b08092ca089080f1a71fbfdd911adda9b3 \
    --hash=sha256:1edb10e520e4bddf74d85b0f5ae74cc2d60c2b448885080bfb618bc2b3a49f6b \
    --hash=sha256:201a8e0f55501a76e08dbf2c54bc45f4642b379271e89c667d517bfbc2191f2a \
    --hash=sha256:252908153eeec06c3ca3a32077e64a49d572e3d89881475b4e0f02d99d9fcc7c \
    --hash=sha256:38111ed1f79268f399c505028023d5eaaf0ab4e5eafceb709468b0d3323e7838 \
    --hash=sha256:3ab3f1ae98970b5590e2209341522718900ba19bcc2c207ffaa6bd417ad960c5 \
    --hash=sha256:3d2b5e824b5622038dc7b490897efe05ebbbb9e7350fc142f3ecc8789ef9bdf6 \
    --hash=sha256:3ec27c152a1f1b24bc9168b55a5880f3c16e2334e697da6f55a1046a22405a3d \
    --hash=sha256:3f5851441ab1ef8634963a5100b733a8bbeefe623e0c5c005b1f1f3880e574cf \
    --hash=sha256:3fd9ce2ab4460c713cfdeb4aca693ca6732a11538e05fb332d5af42e3d7fde25 \
    --hash=sha256:44284adc6fbe9d5bdd480541431a3d93f674fa44736714d3ad4bcee8283ace7d \
    --hash=sha256:443ac618c7a2a1377cf5c82581fbb849591d14e656d5e5a3e4682d4e36a34e4e \
    --hash=sha256:46ba07b543add034de0ff47ac5f907e9a06682f91d85121a972764628933be6b \
    --hash=sha256:4f0603267cd15b92b68c2c0e852a441507614b70dc7773659baa6b8c214a91fd \
    --hash=sha256:524e2a85c028a0d2f9935191fa751e5ef9d9bcc39616f70ab14b28d0369c9936 \
    --hash=sha256:54a83df9260a89c1734256e620fe1f1a6bfedd7547139d4dc1384efac11a3a85 \
    --hash=sha256:59d6588712101ccfcae9b03692be3aaae1514c2078666d7b05f15ba3a702e41b \
    --hash=sha256:63250cfab8b80a1ef82a614eb2b3cadfec2c405f870cedc139d08e2f063eb708 \
    --hash=sha256:64b656f025355cf8c51abe9fbe3848540756c6d7ca5e6791b1afa664bc24c7cb \
    --hash=sha256:65d84ec36888de4a848eee5f910e67fbc79b064685ef1e10a502e14520ead9c9 \
    --hash=sha256:69e9dc8078e128286ed3b975e37c837ba96e215a50c3ef9f3f8b7ab9e5a832a0 \
    --hash=sha256:6dd76f3e5c8b2eb8a3a3efee787bbf5b9a66e52a048fe09cab85eca33fec6790 \
    --hash=sha256:70d4ca6f4d06df7f0ccab6fe4f49c8a712c8c8b6847b4f0af9a0e1dbb0e0337e \
    --hash=sha256:72b7825b331b1b7e7c45be2e674b3e3c65af608fa376bad2d851b20aaf0cdc78 \
    --hash=sha256:741b4b367140e9b5c36b5a14c72179f2c946d991ea9a7c031a2a1ee6ad097b99 \
    --hash=sha256:74f0ee601047c0c12a783088b51be4e6214a62ecd9e02278c477433cd16e0ed9 \
    --hash=sha256:76ff5814db72e7462dece042d7593cdf102b8ec82c2b1cc201a2add34ee3050d \
    --hash=sha256:77c3ce990b23441e5ecfa5bce181fd6f408b564aeb6d7e1d1e7de9c5612501c8 \
    --hash=sha256:79bac5a251f23a7341e28fda9ce0d5319edf45328239ce037c0682936f137906 \
    --hash=sha256:7c6e7bf684dc12145bfa685d3060beaea55139134ba848289bee514ed42e7383 \
    --hash=sha256:7fc14c1585139fa6b68775e616a6b90cf622ebf219f9558c0aeaf5d253ee6c9b \
    --hash=sha256:89625fb43765cccaa1443b9adb61f283e5fe4cb1536728205d06bada730caa53 \
    --hash=sha256:8b2db2056c97224e122054fd794543cde5d24b7cae28424f6e3eb79bbe08e42b \
    --hash=sha256:8d44b20234905ff022b7d535f79d1f823ad7670c9851cc4f03cdc34787cdb3ab \
    --hash=sha256:8eed98514bffe5ecac37f493f91869c351fbb05629328bfdbc08502c6c094dc0 \
    --hash=sha256:8f1f61592e7cabd45d49ce8cc0ef42ca655c091e037153754fb3fa59725b5914 \
    --hash=sha256:b23fe17779834d3c27aaf2edac9486d04cca1a7deb8f5facda35150ac6263a91 \
    --hash=sha256:bc7b0b1da20f856bfac5f84b2673fe534b167e41980b27442ca8f78c2b7eb77e \
    --hash=sha256:c62bd361cec1f5b556eb8210264ecfff37486cd990c3386cc00310f26c54090a \
    --hash=sha256:c76c9b3324efd79029eeb0fd2ced1964bdbeca7d45e030b46fa3ef3cf74f8032 \
    --hash=sha256:c798f0b327bac10dc95cdac77b9a197ab2bd7dd1e60ebd7586a12d918d4be711 \
    --hash=sha256:c8a168b040bc61681293f79a949b5d911c8e25086f4260285b8d97ab5f1195da \
    --hash=sha256:caad9566e2ef0e5640d36032c69b0edc7ac6028277b93d93815898804fac450c \
    --hash=sha256:cbce24284f51f71d10a42b7b9c964dcb9048b28f1c8e5db40bcbcb6f428cba6a \
    --hash=sha256:cd810878180a52950e5a61f25ada5248a453bbdbafe474f89514135fbc1f633d \
    --hash=sha256:d254c98ca6387655400b3959c33c83efd807f5edeb608e3aca45800ceaa77151 \
    --hash=sha256:d795c4ac689a57f9d4ba2288126ec7901d389ad5827d2f8b8533c883974fe563 \
    --hash=sha256:df88b0c34f2fa909d322f7b06b1398e1e81af4b2f42a7b8e3556f928b25d1811 \
    --hash=sha256:eb8da9d9a9b418422c21a07fd19b9d9228692b7a7468a45eec6b11642d3c808b \
    --hash=sha256:f7c06c751c19d923435a54bff4f7e66e728fad160e8da28254f133abc9725820 \
    --hash=sha256:fa9f5ef0e2a82233dd0b8b32ea3f5710e0c44afbc07ed3620219f32601e56090 \
    --hash=sha256:fd523c4992041faa5c2b3cde62253d11a96c30d73a34afe48a486e8e2254cd1c
    # via
    #   gliner-ner-pipeline (pyproject.toml)
    #   gliner
setuptools==84.0.0 \
    --hash=sha256:51a52592b3b99e102b609654876bd65f19f999935166d1352678931132b0c670 \
    --hash=sha256:f4695c21257f0d9b537ec2692c941d02ee143b7cc1276941349a546573b2ef73
    # via
    #   -c esm2-protein-pipeline lock (versions only)
    #   torch
sympy==1.14.0 \
    --hash=sha256:d3d3fe8df1e5a0b42f0e7bdf50541697dbe7d23746e894990c030e2b05e72517 \
    --hash=sha256:e091cc3e99d2141a0ba2847328f5479b05d94a6635cb96148ccb3f34671bd8f5
    # via
    #   -c esm2-protein-pipeline lock (versions only)
    #   torch
tokenizers==0.22.2 \
    --hash=sha256:143b999bdc46d10febb15cbffb4207ddd1f410e2c755857b5a0797961bbdc113 \
    --hash=sha256:1a62ba2c5faa2dd175aaeed7b15abf18d20266189fb3406c5d0550dd34dd5f37 \
    --hash=sha256:1c774b1276f71e1ef716e5486f21e76333464f47bece56bbd554485982a9e03e \
    --hash=sha256:1e418a55456beedca4621dbab65a318981467a2b188e982a23e117f115ce5001 \
    --hash=sha256:1e50f8554d504f617d9e9d6e4c2c2884a12b388a97c5c77f0bc6cf4cd032feee \
    --hash=sha256:2249487018adec45d6e3554c71d46eb39fa8ea67156c640f7513eb26f318cec7 \
    --hash=sha256:25b85325d0815e86e0bac263506dd114578953b7b53d7de09a6485e4a160a7dd \
    --hash=sha256:29c30b83d8dcd061078b05ae0cb94d3c710555fbb44861139f9f83dcca3dc3e4 \
    --hash=sha256:319f659ee992222f04e58f84cbf407cfa66a65fe3a8de44e8ad2bc53e7d99012 \
    --hash=sha256:369cc9fc8cc10cb24143873a0d95438bb8ee257bb80c71989e3ee290e8d72c67 \
    --hash=sha256:37ae80a28c1d3265bb1f22464c856bd23c02a05bb211e56d0c5301a435be6c1a \
    --hash=sha256:38337540fbbddff8e999d59970f3c6f35a82de10053206a7562f1ea02d046fa5 \
    --hash=sha256:473b83b915e547aa366d1eee11806deaf419e17be16310ac0a14077f1e28f917 \
    --hash=sha256:544dd704ae7238755d790de45ba8da072e9af3eea688f698b137915ae959281c \
    --hash=sha256:64d94e84f6660764e64e7e0b22baa72f6cd942279fdbb21d46abd70d179f0195 \
    --hash=sha256:753d47ebd4542742ef9261d9da92cd545b2cacbb48349a1225466745bb866ec4 \
    --hash=sha256:791135ee325f2336f498590eb2f11dc5c295232f288e75c99a36c5dbce63088a \
    --hash=sha256:9ce725d22864a1e965217204946f830c37876eee3b2ba6fc6255e8e903d5fcbc \
    --hash=sha256:a6bf3f88c554a2b653af81f3204491c818ae2ac6fbc09e76ef4773351292bc92 \
    --hash=sha256:bfb88f22a209ff7b40a576d5324bf8286b519d7358663db21d6246fb17eea2d5 \
    --hash=sha256:c9ea31edff2968b44a88f97d784c2f16dc0729b8b143ed004699ebca91f05c48 \
    --hash=sha256:df6c4265b289083bf710dff49bc51ef252f9d5be33a45ee2bed151114a56207b \
    --hash=sha256:e10bf9113d209be7cd046d40fbabbaf3278ff6d18eb4da4c500443185dc1896c \
    --hash=sha256:f01a9c019878532f98927d2bacb79bbb404b43d3437455522a00a30718cdedb5
    # via
    #   -c esm2-protein-pipeline lock (versions only)
    #   transformers
torch==2.14.0 \
    --hash=sha256:0e7cf18cb0d8bd666b6120932e29c7aef3502b61a08b44da4839580c539a7cdb \
    --hash=sha256:2fb30099be1fec9d163a6dd05ace176305ce8cf914b4d7a5f4f6d7a326aa4b60 \
    --hash=sha256:44b044b9f6f633d982839422a57433d6a1da520037fd88e0c8a47efde589b3b8 \
    --hash=sha256:4dbea8a10d80b10dff2be1ed467d2617b704fe6bf76bbfad4194774c8f6d886b \
    --hash=sha256:553aec938d37d77b783bcf801e638cad068870e7643c47eadac21eb180f551ed \
    --hash=sha256:6981872df75eb5409c439050d36b67cb57d88b4e11080516e2aa7410f6730705 \
    --hash=sha256:731784e3914843c6bcc7aba3987ff7610ac57dbbc816a5d6b9b62e04c240a641 \
    --hash=sha256:731b9ebdea402b8b1996d4c2ae613b16660e559b19e47bdc45d970568bc91c53 \
    --hash=sha256:84bf384779a10c02fc3c6bdbab71a9cb66b0dd93c652d1ed5d6dfc0cb37e5962 \
    --hash=sha256:860423e970f2ce02c4476e8e2d1350131b1c5c5a5e4912180e78b50b53241efa \
    --hash=sha256:8d9e232b6376c62f3090237889fb1cb6887c0ece9f73e6c1052e80fc1481f999 \
    --hash=sha256:96383c62423c3f2767023c4a94774b6435e2c006691daf396e9ddb55d412fbc6 \
    --hash=sha256:9d4b1022a5d9b71282ec67ad0d9e7235870096b8a246dc1c32d6ea1fc83dc998 \
    --hash=sha256:ada340e62591d06a2bcc2d68170f20f45f0b0665d372dc510a8ed7eb3b1d609a \
    --hash=sha256:ae530ddd3f3b94248b77f1fd3313c4f3405bd0d17283d505b81574816767133a \
    --hash=sha256:b2cd92bce63d40bf6fc2e5d840fc2f0063bd180a242daa761cb6088cc2f46e27 \
    --hash=sha256:b985c7defeb8d28691b7513ebaecc64946c5f68ec770e9f4ddba39688864f46a \
    --hash=sha256:c1f844f1c750e87df4b68bc3afbc0e2b0c7ef19d7b8f666e48bdcf6a0c4f0056 \
    --hash=sha256:cad84f41bbdf3dcf333ce394aeeaf25237c4d94fd6b659ba5eb813c829978823 \
    --hash=sha256:cd8cd8f714d511ccdca907282d1da3d9be8322d4e1520b9c3bce39a5c1318a4b \
    --hash=sha256:d2526f71e6638133b97b3cd2881ece3df521460a4727030e8ae2a72a7d3ae31d \
    --hash=sha256:d30207cb89e713dfd6afb5d592b30d9ef252acf80407c5543157ea7b07c4d9a0 \
    --hash=sha256:d70c2b41bde81efde59fcefb147b9247fd062497e778677661b7de1bee5c8c99 \
    --hash=sha256:fecffb58f51fd643d213acd68da21cc3fc19bea05a3bc64b4ee55128f47a4963
    # via
    #   -c esm2-protein-pipeline lock (versions only)
    #   gliner-ner-pipeline (pyproject.toml)
    #   gliner
    #   torchvision
torchaudio==2.11.0 \
    --hash=sha256:00e9f71ab9c656f0abdb40c515bd65d4658ab0ad380dee27a2efd7d51dabd3d6 \
    --hash=sha256:13cff988697ccbad539987599f9dc672f40c417bed67570b365e4e5002bbd096 \
    --hash=sha256:1424638adb8bb40087bc7b6eb103e8e4fe398210f09076f33b7b5e61501b5d66 \
    --hash=sha256:1a07ec72fd6f26a588c39b5f029e0130d16bb40bc4221635580bf8fb18fcbc80 \
    --hash=sha256:1be3767064364ae82705bdf2b15c1e8b41fea82c4cd04d47428a8684b634b6ed \
    --hash=sha256:1c1101c1243ef0e4063ec63298977e2d3655c15cf88d9eb0a1bd4fe2db9f47ea \
    --hash=sha256:478110f981e5d40a8d82221732c57a56c85a1d5895fb8fe646e86ee15eded3bd \
    --hash=sha256:492dd64645e9d0bb843e94f1d9a4d1e31426262ffc594fafecc1697df9df5eb9 \
    --hash=sha256:5847fe2022b17c6580aeb39c8797a443411cc09edfd9183cd50ac1a3b8ccf97c \
    --hash=sha256:6503c0bdb29daf2e6281bb70ea2dfe2c3553b782b619eb5d73bdadd8a3f7cecf \
    --hash=sha256:67f6edac29ed004652c11db5c19d9debb5d835695930574f564efc8bdd061bba \
    --hash=sha256:6ebb59c694909eccb5d61b7cc199d297692012c43286e36d92983aa7bad7586d \
    --hash=sha256:73dab4841f94d888bc7c2aed7b5547c643edc974306919fe1adfb65d57cccf4b \
    --hash=sha256:79fb3cb99169fd41bd9719647261402a164da0d105a4d81f42a3260844ec5e79 \
    --hash=sha256:7e2da1df4f6fe885c46db350a0dc90a0dff4b54541dff8846faa904d255e2bfe \
    --hash=sha256:88fb5e29f670a33d9bac6aabb1d2734460cf6e461bde5cdc352826035851b16d \
    --hash=sha256:986f4df5ed17b003dc52489468601720090e65f964f8bebccf90eb45bba75744 \
    --hash=sha256:9fe3083c62e035646483a14e180d33561bdc2eed436c9ab1259c137fb7120b4a \
    --hash=sha256:a1cf1acc883bee9cb906a933572fed6a8a933f86ef34e9ea7d803f72317e8c1b \
    --hash=sha256:b034d7672f1c415434f48ef17807f2cce47f29e8795338c751d4e596c9fbe8b5 \
    --hash=sha256:bb59ba4452bbbe95d75ad3ef18df9824955625f36698ce9a5998a4a9f3c1ba1d \
    --hash=sha256:bc653defca1c16154398517a1adc98d0fb7f1dd08e58ced217558d213c2c6e29 \
    --hash=sha256:bda09ea630ae7207384fb0f28c35e4f8c0d82dd6eba020b6b335ad0caa9fed49 \
    --hash=sha256:be7ad472acb16d16e98c005f0219b0db06a47dfe8f7b4d177062e1638f871e3b \
    --hash=sha256:cc09cd1f6015b8549e7fe255fb1be5346b57e7fee06541d3f3dbb012d8c4715f \
    --hash=sha256:da2725e250866da42a12934c9a6552f65a18b7187fd7a6221387f0e605fb3b96 \
    --hash=sha256:e3f9696a9ef1d49acc452159b052370c636406d072e9d8f10895fda87b591ea9 \
    --hash=sha256:ed404c4399ad7f172c86a47c1b25293d322d1d58e26b10b0456a86cf67d37d84
    # via
    #   -c esm2-protein-pipeline lock (versions only)
    #   gliner-ner-pipeline (pyproject.toml)
torchvision==0.29.0 \
    --hash=sha256:01a029fa1b2eacac27e1e4f9e0dba24d112bcbd523261b294e4b4a1c7f8330bc \
    --hash=sha256:0e3b99c62f7f095153887330c2c60c9ebc2da84dfbe08eea564a41ec361629cf \
    --hash=sha256:0e631a2f8b24732672d35224d2574ff89a78d56bef5364ef6094a625fa5f1771 \
    --hash=sha256:2428a13706a354ee3901fd63e3f1ef8c2a59f6e20d24cce13210b4e61be08f7a \
    --hash=sha256:28a3964a9e6db34354d4ca440d2c2038deeef24e7219e8d6f49d9d174f44830c \
    --hash=sha256:710df2c4f03ab6da161520c10d9467f4c0201ca314fd1432499a48c9d7a5818f \
    --hash=sha256:784c8b8de9e81e02dea093373bc35040791378df2daec0017e657985183bf862 \
    --hash=sha256:80789f9b50f5277302e7020c8128344e0269b12f6e3b5855034e56bb7081b874 \
    --hash=sha256:83db0299170502b038640444214c99d7b54bad72db447f0554deb8d2c4f02f94 \
    --hash=sha256:85fa54bec1f7d9227b5e4e201ed4bac92247ee10fb19148d92f9836e3e91d5c3 \
    --hash=sha256:942a3b3fb4e981e1abb7846679ee3882dfc733533a30b68df0dadf6c06f238a9 \
    --hash=sha256:994687b818cac0e6d34cb407a41458e2c3262cb7db927465841c9522ba7ebb92 \
    --hash=sha256:9b99a7385da8d706f2dc14cacffb8cbf653d48fc37013a38c41a36659acc3731 \
    --hash=sha256:a292ca7044236a2702ec3d4664de135e01fc9fb3ae9422db3fecc819546e831c \
    --hash=sha256:add88479361b2bf08790338fce4ba11a11fdc1ae508e80a3a013daf1c0d2893d \
    --hash=sha256:b7a736eaa6b2e22476c95ceb62986d195f3e600cdd7d196d7329aa2dfc951994 \
    --hash=sha256:bb74d08d65785b51c61d4d4ec167bfa50a5ea644b57555a33ba2f4435b0d05dd \
    --hash=sha256:bbe15455b59a6c9d822584fd3faaaac50ee972101d7bb7e2887da456e148a3ea \
    --hash=sha256:bd12e152640d1024fca15deaf6f39c53702e12eb398db4b2b038a5a97ab85f7f \
    --hash=sha256:ce3b58631f0b7c8828e8675301bff5cb0623aaed704e3ac91079e89607753205 \
    --hash=sha256:dc5717feef9a0b430b052895db71f1d695939575b74344da981120e9f4da7620 \
    --hash=sha256:e8fdf234d76dca6fc47bc5f4af86ecf1ef57fab6285a7b21681bdcbb494e4c1b \
    --hash=sha256:ebf54f6744556ebd8d7b5f9b6515e2060e5d3156723970dc257bf093f63b2170 \
    --hash=sha256:f9e9627d5036cac8e6de76aeb2f1acc67841ff5ff4b63c4cf78ff954efeb073e
    # via
    #   -c esm2-protein-pipeline lock (versions only)
    #   gliner-ner-pipeline (pyproject.toml)
tqdm==4.70.1 \
    --hash=sha256:c293e525e6fef9c20e8728fd4612df02a0aa31bb5fe91ecd93e123b1b7bffa73 \
    --hash=sha256:cefd0eca11b2a37a3aee776544d4f4ae913f02688135b5556b8788dfa474afc4
    # via
    #   -c esm2-protein-pipeline lock (versions only)
    #   gliner
    #   huggingface-hub
    #   transformers
transformers==4.57.6 \
    --hash=sha256:4c9e9de11333ddfe5114bc872c9f370509198acf0b87a832a0ab9458e2bd0550 \
    --hash=sha256:55e44126ece9dc0a291521b7e5492b572e6ef2766338a610b9ab5afbb70689d3
    # via
    #   -c esm2-protein-pipeline lock (versions only)
    #   gliner-ner-pipeline (pyproject.toml)
    #   gliner
triton==3.8.0 \
    --hash=sha256:1b84e7d512490ba529111260fa6f7cad8b254a6bb5fbdf41d5ef9a5e57f52d0a \
    --hash=sha256:1f0497218e26b7d79773ad9c2a3fa3b539ee69f587a13fac2e552b1d322a8015 \
    --hash=sha256:1f6b48d0591929a3867973acac3dccd4e058585f91bfb41022de496c9ffab304 \
    --hash=sha256:372285307d4c44ee74cee32de0b4f04bd157e071427e38c6f4ee3e3beb2194f4 \
    --hash=sha256:387dae4cb0089a7b6ba1a428ae0782b65c4c58f57d94617cb22ca8593d8ccbca \
    --hash=sha256:398f4b009c7ab08ed9aeb1d1282dee822945b53f399a5e12a4fecb643cf2007d \
    --hash=sha256:68988ac85d5e7086baeda0ddc175af9667db7529b3c5e11a5c0601b8bef2200a \
    --hash=sha256:6d914c52f89dc942b1819db959e7c07a5999f678ec760f677625c706b7e0d743 \
    --hash=sha256:74217bb56ed8692759227758e4c4b3bd2d608a209c1a7a081bf361fb4c2c1bf9 \
    --hash=sha256:a9c404c69ed4a39e8ec632eaf6b9fe058a060bf98979c177f6ef666f06bb8d50 \
    --hash=sha256:b7004666652f500ed854a86988e4b3d69d247188b5d2092b5df1e44f4a954099 \
    --hash=sha256:e91ffa46d095b252248297292dd22bcbacd53a125a0c2eefbbbf74925a320bc3
    # via
    #   -c esm2-protein-pipeline lock (versions only)
    #   torch
typing-extensions==4.16.0 \
    --hash=sha256:481caa481374e813c1b176ada14e97f1f67a4539ce9cfeb3f350d78d6370c2e8 \
    --hash=sha256:dc983d19a509c94dba722ee6abd33940f7c05a89e243c47e907eb4db6f1a43e5
    # via
    #   -c esm2-protein-pipeline lock (versions only)
    #   huggingface-hub
    #   torch
urllib3==2.8.0 \
    --hash=sha256:0cf3cae568d36aa9576b28dfb35f11328f1cb974ca7647d9475ebb86c75ac6e3 \
    --hash=sha256:63bf2ead4c879426ebf22ef2a781eeb4aa3b4ae798a0435506f8687fd5bb9b63
    # via
    #   -c esm2-protein-pipeline lock (versions only)
    #   requests
'''

SKIP_INSTALL = os.environ.get("DIMER_NOTEBOOK_CI_PREINSTALLED") == "1"
ISOLATED_ENV = Path(os.environ.get("DIMER_ISOLATED_ENV", "dimer_isolated_env")).resolve()
ISOLATED_PYTHON = ISOLATED_ENV / "bin" / "python"
ISOLATED_TOOLS = ISOLATED_ENV.with_name(ISOLATED_ENV.name + "_tools")

if SKIP_INSTALL:
    print("DIMER_NOTEBOOK_CI_PREINSTALLED=1: the pins are already installed; the notebook runs in this kernel.")
else:
    if platform.system() != "Linux" or platform.machine() != "x86_64":
        raise RuntimeError("This notebook needs a Linux x86_64 runtime (Google Colab, Kaggle or Linux Jupyter): its locked environment is built for manylinux x86_64.")
    setup_started = time.perf_counter()
    if hashlib.sha256(LOCK_TEXT.encode("utf-8")).hexdigest() != LOCK_SHA256:
        raise RuntimeError("The carried lock does not match its digest: regenerate the notebook from the repository instead of editing this cell.")
    ISOLATED_TOOLS.mkdir(parents=True, exist_ok=True)
    lock_path = ISOLATED_TOOLS / LOCK_NAME
    lock_path.write_text(LOCK_TEXT, encoding="utf-8", newline="\n")
    for attempt in range(3):
        try:
            with urllib.request.urlopen(UV_URL, timeout=90) as response:
                wheel = response.read(UV_BYTES + 1)
            break
        except (urllib.error.URLError, TimeoutError, ConnectionError):
            if attempt == 2:
                raise
            time.sleep(2**attempt)
    if len(wheel) != UV_BYTES or hashlib.sha256(wheel).hexdigest() != UV_SHA256:
        raise RuntimeError("The pinned uv wheel failed its size/SHA-256 check: refusing to run it. Run this cell again; if it repeats, the download is being altered.")
    with zipfile.ZipFile(io.BytesIO(wheel)) as archive:
        member = next(name for name in archive.namelist() if name.endswith(".data/scripts/uv"))
        uv = ISOLATED_TOOLS / "uv"
        uv.write_bytes(archive.read(member))
    uv.chmod(0o700)
    # uv gets no kernel Python path; the managed interpreter is downloaded once and reused on a re-run.
    uv_env = dict(os.environ)
    for name in ("PYTHONPATH", "PYTHONHOME", "PYTHONSTARTUP"):
        uv_env.pop(name, None)
    if not ISOLATED_PYTHON.is_file():
        subprocess.run([str(uv), "venv", "--quiet", "--managed-python", "--python", MANAGED_PYTHON, str(ISOLATED_ENV)], env=uv_env, check=True)
    isolated_version = subprocess.run([str(ISOLATED_PYTHON), "-c", "import platform; print(platform.python_version())"], env=uv_env, check=True, capture_output=True, text=True).stdout.strip()
    if isolated_version != MANAGED_PYTHON:
        raise RuntimeError(f"{ISOLATED_ENV} holds Python {isolated_version}, not {MANAGED_PYTHON}: delete that folder (or start a fresh runtime) and run this cell again.")
    subprocess.run([str(uv), "pip", "install", "--quiet", "--python", str(ISOLATED_PYTHON), "--require-hashes", "--only-binary", ":all:", "--index-url", "https://pypi.org/simple", "-r", str(lock_path)], env=uv_env, check=True)
    print({"isolated_environment": str(ISOLATED_ENV), "isolated_python": isolated_version, "kernel_python": platform.python_version(), "locked_packages": LOCKED_PACKAGES, "setup_seconds": round(time.perf_counter() - setup_started)})

{'isolated_environment': '/content/dimer_isolated_env', 'isolated_python': '3.12.12', 'kernel_python': '3.13.15', 'locked_packages': 50, 'setup_seconds': 48}


The next cell starts one Python process in that environment and routes **every later code cell** to it. Printed output comes back to the notebook as usual, variables persist from cell to cell, and an error stops **Run all** as it would in the kernel. These two cells are marked `# dimer: kernel cell` and are the only cells that run in the kernel. If you re-run a single cell later, it still runs in the isolated environment with the variables created so far; to start over, restart the session and choose **Run all**. `DIMER_NOTEBOOK_CI_PREINSTALLED=1` lets an executor that has already installed exactly these pins run every cell in its own kernel instead.

In [ ]:
# @title Infrastructure: route the remaining cells to the isolated environment
# dimer: kernel cell (runs in the notebook kernel, not in the isolated environment)
import signal
from multiprocessing.connection import Connection

# The worker runs in the isolated environment. It executes each routed cell in one persistent namespace and sends
# back printed text, displayed objects and matplotlib figures, so every cell behaves as it would in the kernel.
_WORKER_SOURCE = r"""
import ast, base64, builtins, io, linecache, os, signal, sys, traceback, types
from multiprocessing.connection import Connection

_send = Connection(int(sys.argv[1]), readable=False)
_recv = Connection(int(sys.argv[2]), writable=False)


class _Stream(io.TextIOBase):
    def __init__(self, name):
        self._name = name

    @property
    def encoding(self):
        return "utf-8"

    def writable(self):
        return True

    def isatty(self):
        return False

    def write(self, text):
        if text:
            _send.send(("stream", self._name, str(text)))
        return len(text)


sys.stdout, sys.stderr = _Stream("stdout"), _Stream("stderr")


def _figure_bundle(fig):
    buffer = io.BytesIO()
    fig.savefig(buffer, format="png", bbox_inches="tight")
    return {"image/png": base64.b64encode(buffer.getvalue()).decode("ascii"), "text/plain": repr(fig)}


def _flush_figures():
    plt = sys.modules.get("matplotlib.pyplot")
    if plt is None:
        return
    for number in plt.get_fignums():
        _send.send(("display", _figure_bundle(plt.figure(number))))
    plt.close("all")


def _mimebundle(obj):
    if hasattr(obj, "savefig"):
        return _figure_bundle(obj)
    data = {"text/plain": repr(obj)}
    for method, mime in (("_repr_html_", "text/html"), ("_repr_markdown_", "text/markdown"), ("_repr_png_", "image/png")):
        render = getattr(obj, method, None)
        if callable(render):
            try:
                value = render()
            except Exception:
                value = None
            if isinstance(value, bytes):
                value = base64.b64encode(value).decode("ascii")
            if value is not None:
                data[mime] = value
    return data


def display(*objects, **kwargs):
    for obj in objects:
        _send.send(("display", _mimebundle(obj)))


try:
    import matplotlib

    matplotlib.use("Agg")
    import matplotlib.pyplot

    matplotlib.pyplot.show = lambda *args, **kwargs: _flush_figures()
except ImportError:
    pass

if os.environ.get("DIMER_KERNEL_IS_COLAB") == "1":
    # google.colab only exists in the kernel; forward the BYOD upload dialog to it.
    def _upload():
        _send.send(("upload",))
        reply = _recv.recv()
        if reply[1] is None:
            raise RuntimeError("The notebook kernel could not open the upload dialog.")
        return reply[1]

    import importlib.machinery

    def _stub(name, package):
        # A spec on every stub: importlib.util.find_spec("google.colab") (accelerate does this) raises on a None __spec__.
        module = types.ModuleType(name)
        module.__spec__ = importlib.machinery.ModuleSpec(name, None, is_package=package)
        if package:
            module.__path__ = []
        return module

    try:
        import google
    except ImportError:
        google = _stub("google", True)
        sys.modules["google"] = google
    _colab = _stub("google.colab", True)
    _files = _stub("google.colab.files", False)
    _files.upload = _upload
    _colab.files = _files
    google.colab = _colab
    sys.modules["google.colab"] = _colab
    sys.modules["google.colab.files"] = _files

_main = types.ModuleType("__main__")
_main.__dict__.update(__builtins__=builtins, display=display)
sys.modules["__main__"] = _main
_count = 0
while True:
    # An interrupt only lands inside a running cell; between cells it is ignored.
    signal.signal(signal.SIGINT, signal.SIG_IGN)
    try:
        message = _recv.recv()
    except EOFError:
        break
    if message[0] != "run":
        continue
    _count += 1
    filename = f"<isolated cell {_count}>"
    source = message[1]
    linecache.cache[filename] = (len(source), None, source.splitlines(True), filename)
    try:
        signal.signal(signal.SIGINT, signal.default_int_handler)
        tree = ast.parse(source, filename)
        tail = ast.Expression(tree.body.pop().value) if tree.body and isinstance(tree.body[-1], ast.Expr) else None
        exec(compile(tree, filename, "exec"), _main.__dict__)
        if tail is not None:
            value = eval(compile(tail, filename, "eval"), _main.__dict__)
            if value is not None:
                display(value)
        _flush_figures()
        signal.signal(signal.SIGINT, signal.SIG_IGN)
        _send.send(("done",))
    except BaseException as exc:
        signal.signal(signal.SIGINT, signal.SIG_IGN)
        frames = exc.__traceback__.tb_next if exc.__traceback__ is not None else None
        _send.send(("error", "".join(traceback.format_exception(type(exc), exc, frames)), f"{type(exc).__name__}: {exc}"))
"""


class IsolatedCellError(RuntimeError):
    """A routed cell raised inside the isolated environment; its traceback is printed above."""


class IsolatedRuntime:
    """One persistent worker process in the isolated environment, fed one cell at a time."""

    def __init__(self, python, display=None):
        to_kernel_r, to_kernel_w = os.pipe()
        to_worker_r, to_worker_w = os.pipe()
        env = dict(os.environ, MPLBACKEND="Agg", PYTHONUNBUFFERED="1", DIMER_NOTEBOOK_CI_PREINSTALLED="1", HF_HUB_DISABLE_IMPLICIT_TOKEN="1")
        for name in ("PYTHONPATH", "PYTHONHOME", "PYTHONSTARTUP", "HF_TOKEN", "HUGGING_FACE_HUB_TOKEN"):
            env.pop(name, None)
        env["DIMER_KERNEL_IS_COLAB"] = "1" if "google.colab" in sys.modules else "0"
        self.proc = subprocess.Popen(
            [str(python), "-c", _WORKER_SOURCE, str(to_kernel_w), str(to_worker_r)],
            pass_fds=(to_kernel_w, to_worker_r),
            env=env,
            start_new_session=True,  # interrupts reach the worker only through run(), exactly once
        )
        os.close(to_kernel_w)
        os.close(to_worker_r)
        self._recv = Connection(to_kernel_r, writable=False)
        self._send = Connection(to_worker_w, readable=False)
        if display is None:
            from IPython.display import display
        self._display = display

    def _exited(self):
        return RuntimeError(
            f"The isolated environment's Python process exited (code {self.proc.wait()}); a crash of this kind is "
            "usually running out of memory. Restart the session and choose Run all again."
        )

    def run(self, source):
        try:
            self._send.send(("run", source))
        except OSError:
            raise self._exited() from None
        while True:
            try:
                message = self._recv.recv()
            except EOFError:
                raise self._exited() from None
            except KeyboardInterrupt:
                self.proc.send_signal(signal.SIGINT)
                continue
            kind = message[0]
            if kind == "stream":
                (sys.stdout if message[1] == "stdout" else sys.stderr).write(message[2])
            elif kind == "display":
                self._display(message[1], raw=True)
            elif kind == "upload":
                self._send.send(("upload", self._colab_upload()))
            elif kind == "error":
                sys.stderr.write(message[1])
                raise IsolatedCellError(message[2]) from None
            elif kind == "done":
                return

    @staticmethod
    def _colab_upload():
        try:
            from google.colab import files
        except ImportError:
            return None
        return files.upload()

    def close(self):
        self._send.close()
        self.proc.wait(timeout=30)


def _is_user_cell():
    # ipykernel transforms a cell before executing it; its caller knows whether this is a silent frontend request.
    frame = sys._getframe(2)
    while frame is not None:
        local = frame.f_locals
        if "silent" in local and "store_history" in local:
            return bool(local["store_history"]) and not bool(local["silent"])
        frame = frame.f_back
    return True


def _route_to_isolated_runtime(lines):
    source = "".join(lines)
    if not source.strip() or "# dimer: kernel cell" in source or not _is_user_cell():
        return lines
    return [f"_DIMER_ISOLATED_RUNTIME.run({source!r})\n"]


if SKIP_INSTALL:
    print("Routing disabled: the notebook runs in this kernel.")
else:
    from IPython import get_ipython as _kernel_shell

    _ip = _kernel_shell()
    _ip.input_transformers_cleanup[:] = [
        t for t in _ip.input_transformers_cleanup if getattr(t, "__name__", "") != "_route_to_isolated_runtime"
    ]
    if isinstance(globals().get("_DIMER_ISOLATED_RUNTIME"), IsolatedRuntime):
        _DIMER_ISOLATED_RUNTIME.close()
    _DIMER_ISOLATED_RUNTIME = IsolatedRuntime(ISOLATED_PYTHON)
    _ip.input_transformers_cleanup.append(_route_to_isolated_runtime)
    print(f"Every later code cell now runs in {ISOLATED_PYTHON} (pid {_DIMER_ISOLATED_RUNTIME.proc.pid}).")

Every later code cell now runs in /content/dimer_isolated_env/bin/python (pid 1157).


### Record the runtime

This is the first cell that runs in the isolated environment. Nothing is installed here: the locked environment already holds the pins listed in the cell. It records the notebook's source revision and the versions actually imported. Look for a dictionary reporting the notebook's source revision, the isolated Python (3.12.12), `torch`, `transformers`, `gliner` versions, and whether CUDA is available. (The pip install and stale-import guard in the cell are skipped: the isolated worker, like an executor that has pre-installed exactly these pins, sets `DIMER_NOTEBOOK_CI_PREINSTALLED=1`.)

In [ ]:
# @title Infrastructure: record the runtime
import importlib
import importlib.metadata
import os
import platform
import subprocess
import sys

PINS = [
    'torch==2.14.0',
    'torchvision==0.29.0',
    'torchaudio==2.11.0',
    'gliner==0.2.29',
    'transformers==4.57.6',
    'huggingface-hub==0.36.2',
    'safetensors==0.8.0',
    'numpy==2.5.3',
    'protobuf==6.31.1',
    'sentencepiece==0.2.2',
]
NOTEBOOK_SOURCE = {
    'repository': 'gliner-ner-pipeline',
    'repository_revision': '04b30e615194b56d13b12ea513cd09720ce2d925',
    'embedded_module': 'src/gliner_ner_pipeline/pipeline.py',
    'embedded_modules': ['src/gliner_ner_pipeline/metrics.py', 'src/gliner_ner_pipeline/pipeline.py', 'src/gliner_ner_pipeline/samples.py'],
    'module_sha256': '5636cd0d2dbc93e14bc1a0d7615a80e92e955ca5a4c4c1407c371d396495761a',
    'generator': 'build_notebook.py/2.1',
    'notebook_spec': '2.2',
}
SKIP_INSTALL = os.environ.get('DIMER_NOTEBOOK_CI_PREINSTALLED') == '1'

def _installed_version(distribution):
    try:
        return importlib.metadata.version(distribution)
    except importlib.metadata.PackageNotFoundError:
        return None

if not SKIP_INSTALL:
    # Capture every distribution already imported in this runtime, whatever its module name
    # (PIL -> pillow), so a pinned install that replaces a loaded package is detected and the
    # notebook stops with a restart instruction instead of continuing with mixed versions.
    _module_dists = importlib.metadata.packages_distributions()
    _loaded = sorted({d for m in list(sys.modules) for d in _module_dists.get(m.partition('.')[0], ())})
    loaded = {distribution: _installed_version(distribution) for distribution in _loaded}
    subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', *PINS], check=True)
    importlib.invalidate_caches()
    stale = []
    for distribution, before in loaded.items():
        installed = _installed_version(distribution)
        if before is not None and before != installed:
            stale.append(f'{distribution}: loaded={before}, installed={installed}')
    if stale:
        raise RuntimeError('Core dependencies changed while older modules were loaded: ' + '; '.join(stale) + '. Restart the runtime, then rerun from the top.')

import torch, transformers, gliner
print({'notebook_source': NOTEBOOK_SOURCE, 'python': platform.python_version(), 'torch': torch.__version__, 'transformers': transformers.__version__, 'gliner': gliner.__version__, 'cuda': torch.cuda.is_available()})

/content/dimer_isolated_env/lib/python3.12/site-packages/torch/jit/_script.py:1491: FutureWarning: `torch.jit.script` is deprecated. Please switch to `torch.compile` or `torch.export`.
  warnings.warn(


{'notebook_source': {'repository': 'gliner-ner-pipeline', 'repository_revision': '04b30e615194b56d13b12ea513cd09720ce2d925', 'embedded_module': 'src/gliner_ner_pipeline/pipeline.py', 'embedded_modules': ['src/gliner_ner_pipeline/metrics.py', 'src/gliner_ner_pipeline/pipeline.py', 'src/gliner_ner_pipeline/samples.py'], 'module_sha256': '5636cd0d2dbc93e14bc1a0d7615a80e92e955ca5a4c4c1407c371d396495761a', 'generator': 'build_notebook.py/2.1', 'notebook_spec': '2.2'}, 'python': '3.12.12', 'torch': '2.14.0+cu130', 'transformers': '4.57.6', 'gliner': '0.2.29', 'cuda': True}


## 2. Pipeline code (carried verbatim from `src/gliner_ner_pipeline/` @ `04b30e615194`)

> **Infrastructure.** You may run this section without studying its implementation; the learning activities start after Section 3.

The next 3 cell(s) **are** the repository's package, module by module in dependency order: the pinned identity constants, snapshot verification (`verify_snapshot`), staged download (`stage_missing_files`), the named operational ceilings, the public validation and evaluation helpers, and the pipeline class. The text is the modules', byte for byte, except for the rewrite rules listed in `tools/build_notebook.py` (1 rule(s), plus the removal of package-relative `from .x import` lines, whose names are already defined by the preceding cells). The repository's parity test (`tests/test_notebook_parity.py`) fails whenever these cells and the modules diverge, so what you run here is what the repository tests. Nothing in these cells runs a model yet.

**Module 1/3:** `src/gliner_ner_pipeline/metrics.py`

In [ ]:
# @title Infrastructure: carried module `src/gliner_ner_pipeline/metrics.py` (you may run it without reading it)
"""Evaluation metrics and reporting for GLiNER multi-v2.1 named-entity recognition.

Computes exact-span micro and macro Precision, Recall, and F1 across dataset records,
per-class performance breakdowns, and baseline delta comparisons.
"""

from __future__ import annotations

from collections.abc import Mapping, Sequence
from typing import Any


def compute_span_f1(
    hits: int,
    n_predicted: int,
    n_gold: int,
) -> dict[str, float]:
    """Calculate Precision, Recall, and F1 from counts."""
    precision = hits / n_predicted if n_predicted > 0 else 0.0
    recall = hits / n_gold if n_gold > 0 else 0.0
    f1 = (2.0 * precision * recall / (precision + recall)) if (precision + recall) > 0.0 else 0.0
    return {
        "precision": round(float(precision), 4),
        "recall": round(float(recall), 4),
        "f1": round(float(f1), 4),
    }


def evaluate_ner_dataset(
    predictions_per_record: Sequence[Sequence[Mapping[str, Any]]],
    gold_records: Sequence[Mapping[str, Any]],
    labels: Sequence[str],
) -> dict[str, Any]:
    """Compute micro/macro exact-span NER evaluation metrics across records.

    Each prediction is expected to have 'start', 'end', and 'label'.
    Gold records must have 'spans' with 'start', 'end', and 'label'.
    """
    if len(predictions_per_record) != len(gold_records):
        raise ValueError(
            f"number of prediction sets ({len(predictions_per_record)}) != gold records ({len(gold_records)})"
        )

    label_set = list(labels)
    per_class_counts: dict[str, dict[str, int]] = {
        lbl: {"hits": 0, "predicted": 0, "gold": 0} for lbl in label_set
    }

    total_hits = 0
    total_predicted = 0
    total_gold = 0

    for preds, gold_rec in zip(predictions_per_record, gold_records, strict=True):
        gold_spans = gold_rec.get("spans") or gold_rec.get("entities", [])
        gold_tuples = {(int(s["start"]), int(s["end"]), str(s["label"])) for s in gold_spans}
        pred_tuples = {(int(p["start"]), int(p["end"]), str(p["label"])) for p in preds}

        total_predicted += len(pred_tuples)
        total_gold += len(gold_tuples)

        for p in pred_tuples:
            lbl = p[2]
            if lbl in per_class_counts:
                per_class_counts[lbl]["predicted"] += 1

        for g in gold_tuples:
            lbl = g[2]
            if lbl in per_class_counts:
                per_class_counts[lbl]["gold"] += 1

        hits = pred_tuples & gold_tuples
        total_hits += len(hits)

        for h in hits:
            lbl = h[2]
            if lbl in per_class_counts:
                per_class_counts[lbl]["hits"] += 1

    micro = compute_span_f1(total_hits, total_predicted, total_gold)

    class_metrics: dict[str, dict[str, Any]] = {}
    f1_values: list[float] = []

    for lbl in label_set:
        c = per_class_counts[lbl]
        res = compute_span_f1(c["hits"], c["predicted"], c["gold"])
        class_metrics[lbl] = {
            **res,
            "hits": c["hits"],
            "predicted": c["predicted"],
            "gold": c["gold"],
        }
        if c["gold"] > 0:
            f1_values.append(res["f1"])

    macro_f1 = round(sum(f1_values) / len(f1_values), 4) if f1_values else 0.0

    return {
        "micro_precision": micro["precision"],
        "micro_recall": micro["recall"],
        "micro_f1": micro["f1"],
        "macro_f1": macro_f1,
        "hits": total_hits,
        "n_predicted": total_predicted,
        "n_gold": total_gold,
        "per_class": class_metrics,
        "labels": list(labels),
    }

**Module 2/3:** `src/gliner_ner_pipeline/pipeline.py` (carried verbatim; see the note above)

In [ ]:
# @title Infrastructure: carried module `src/gliner_ner_pipeline/pipeline.py` (you may run it without reading it)
from __future__ import annotations

import hashlib
import json
import warnings
from collections.abc import Callable, Mapping, Sequence
from dataclasses import dataclass, field
from pathlib import Path
from typing import Any

MODEL_ID = "urchade/gliner_multi-v2.1"
MODEL_REVISION = "443d26d654e0324125a96bebd8e796c14ff2efe6"
MODEL_LICENSE = "apache-2.0"
MODEL_KEY = "gliner-multi-v2.1"
ARTIFACT_FORMAT = "org.valcorza.gliner-ner.adapter.v1"
ARTIFACT_FORMAT_VERSION = "1.1"  # 1.1 adds `adapter_manifest` (tensor names, shapes, dtypes, SHA-256)
_WEIGHTS_ROOT = Path.cwd() / "weights"  # standalone rewrite (build_notebook.py): working-directory-relative
DEFAULT_WEIGHTS_DIR = _WEIGHTS_ROOT / MODEL_KEY
MANIFEST_NAME = "dimer-base-manifest.json"

# The GLiNER snapshot holds only gliner_config.json + model.safetensors; its `model_name` names the
# encoder whose tokenizer and config the gliner library resolves at load time. Those files are pinned
# here as a SECOND snapshot with its own manifest (no encoder weights: model.safetensors carries them),
# and the loader points the library at that verified directory instead of the Hub or an HF cache.
ENCODER_MODEL_ID = "microsoft/mdeberta-v3-base"
ENCODER_REVISION = "a0484667b22365f84929a935b5e50a51f71f159d"
ENCODER_LICENSE = "mit"
ENCODER_KEY = "mdeberta-v3-base-tokenizer"
ENCODER_WEIGHTS_DIR = _WEIGHTS_ROOT / ENCODER_KEY
DEFAULT_THRESHOLD = 0.5  # upstream predict_entities default; the caller owns tuning it
MAX_LABELS = 25  # gliner_config.json max_types: the most entity types seen per example in training
MAX_TEXT_CHARS = 5_000  # per call; gliner_config.json max_len is 384 words, longer text is cut by the library
MAX_LABEL_CHARS = 100


def _verify_manifest(root: Path, model_id: str, revision: str) -> dict[str, Any]:
    manifest_path = root / MANIFEST_NAME
    if not manifest_path.is_file():
        raise FileNotFoundError(f"snapshot manifest not found: {manifest_path}")
    manifest = json.loads(manifest_path.read_text(encoding="utf-8"))
    if manifest.get("modelId") != model_id:
        raise ValueError(f"manifest modelId {manifest.get('modelId')!r} != {model_id!r}")
    if manifest.get("revision") != revision:
        raise ValueError(f"manifest revision {manifest.get('revision')!r} != {revision!r}")
    for entry in manifest["files"]:
        file_path = root / entry["path"]
        if not file_path.is_file():
            raise FileNotFoundError(f"snapshot file missing: {file_path}")
        size = file_path.stat().st_size
        if size != entry["bytes"]:
            raise ValueError(f"{entry['path']}: size {size} != manifest {entry['bytes']}")
        digest = hashlib.sha256()
        with open(file_path, "rb") as fh:
            for chunk in iter(lambda: fh.read(1 << 20), b""):
                digest.update(chunk)
        if digest.hexdigest() != entry["sha256"]:
            raise ValueError(f"{entry['path']}: sha256 {digest.hexdigest()} != manifest {entry['sha256']}")
    return manifest


def verify_snapshot(path: str | Path | None = None) -> dict[str, Any]:
    """Check the GLiNER snapshot against its DIMER manifest; raise naming the first mismatch."""
    root = Path(path) if path is not None else DEFAULT_WEIGHTS_DIR
    return _verify_manifest(root, MODEL_ID, MODEL_REVISION)


def verify_encoder_snapshot(path: str | Path | None = None) -> dict[str, Any]:
    """Check the mDeBERTa tokenizer/config snapshot against its own manifest (second supply-chain check)."""
    root = Path(path) if path is not None else ENCODER_WEIGHTS_DIR
    return _verify_manifest(root, ENCODER_MODEL_ID, ENCODER_REVISION)


def _hub_download(
    relative_path: str, root: Path, model_id: str = MODEL_ID, revision: str = MODEL_REVISION
) -> None:
    """Fetch one manifest-listed file at the pinned revision straight into the snapshot directory."""
    from huggingface_hub import hf_hub_download

    hf_hub_download(model_id, relative_path, revision=revision, local_dir=str(root))


def _stage_missing(
    root: Path,
    model_id: str,
    revision: str,
    allow_download: bool,
    downloader: Callable[[str, Path], None] | None,
) -> list[str]:
    manifest_path = root / MANIFEST_NAME
    if not manifest_path.is_file():
        raise FileNotFoundError(f"manifest not found: {manifest_path}")
    with open(manifest_path, encoding="utf-8") as fh:
        manifest = json.load(fh)
    if manifest.get("modelId") != model_id or manifest.get("revision") != revision:
        raise ValueError(
            f"manifest names {manifest.get('modelId')}@{manifest.get('revision')}, "
            f"package pins {model_id}@{revision}; refusing to stage"
        )
    missing = [entry["path"] for entry in manifest["files"] if not (root / entry["path"]).is_file()]
    if not missing:
        return []
    if not allow_download:
        raise FileNotFoundError(
            f"snapshot at {root} is missing {missing}; pass allow_download=True to fetch them at {revision}"
        )
    fetch = downloader or (lambda rel, dst: _hub_download(rel, dst, model_id, revision))
    for relative_path in missing:
        fetch(relative_path, root)
    return missing


def stage_missing_files(
    path: str | Path | None = None,
    *,
    allow_download: bool = False,
    downloader: Callable[[str, Path], None] | None = None,
) -> list[str]:
    """Fetch GLiNER manifest entries that are absent locally (a fresh clone commits the manifest but
    git-ignores the weights). Returns the relative paths fetched; `verify_snapshot` still runs after."""
    root = Path(path) if path is not None else DEFAULT_WEIGHTS_DIR
    return _stage_missing(root, MODEL_ID, MODEL_REVISION, allow_download, downloader)


def stage_missing_encoder_files(
    path: str | Path | None = None,
    *,
    allow_download: bool = False,
    downloader: Callable[[str, Path], None] | None = None,
) -> list[str]:
    """Same as `stage_missing_files` for the mDeBERTa tokenizer/config snapshot at ENCODER_REVISION."""
    root = Path(path) if path is not None else ENCODER_WEIGHTS_DIR
    return _stage_missing(root, ENCODER_MODEL_ID, ENCODER_REVISION, allow_download, downloader)


def entity_f1(predicted: Sequence[dict[str, Any]], gold: Sequence[dict[str, Any]]) -> dict[str, float]:
    """Exact-span micro precision/recall/F1: a hit is an identical (start, end, label) triple."""
    pred_set = {(int(e["start"]), int(e["end"]), str(e["label"])) for e in predicted}
    gold_set = {(int(e["start"]), int(e["end"]), str(e["label"])) for e in gold}
    hits = len(pred_set & gold_set)
    precision = hits / len(pred_set) if pred_set else 0.0
    recall = hits / len(gold_set) if gold_set else 0.0
    f1 = 2 * precision * recall / (precision + recall) if precision + recall else 0.0
    return {"precision": precision, "recall": recall, "f1": f1, "hits": hits}


INPUT_SCHEMA: dict[str, Any] = {
    "input": "one text string plus 1..MAX_LABELS unique, non-empty, caller-supplied entity-type labels",
    "text_chars": [1, MAX_TEXT_CHARS],
    "labels": [1, MAX_LABELS],
    "label_chars": [1, MAX_LABEL_CHARS],
    "threshold": [0.0, 1.0],
    "preprocessing": (
        "the gliner library tokenizes with the pinned mDeBERTa-v3 tokenizer and truncates at "
        "gliner_config.json max_len = 384 words, so text beyond that is silently cut; returned spans are "
        "character offsets into the exact string you passed"
    ),
}


def _check_inputs(text: Any, labels: Any, threshold: Any) -> tuple[str, list[str], float]:
    """Raise TypeError/ValueError naming the first violated ceiling; return the checked request.

    ``GLiNERPipeline.detect`` and ``validate_inputs`` both route through this function so their
    acceptance criteria cannot diverge.
    """
    if not isinstance(text, str):
        raise TypeError(f"text must be str, got {type(text).__name__}")
    if not text.strip():
        raise ValueError("text is empty")
    if len(text) > MAX_TEXT_CHARS:
        raise ValueError(f"text has {len(text)} chars; ceiling is {MAX_TEXT_CHARS} (chunk it first)")
    if isinstance(labels, str | bytes) or not isinstance(labels, Sequence):
        raise TypeError("labels must be a list of str")
    if not 1 <= len(labels) <= MAX_LABELS:
        raise ValueError(f"labels must hold 1..{MAX_LABELS} items, got {len(labels)}")
    for i, label in enumerate(labels):
        if not isinstance(label, str) or not label.strip():
            raise TypeError(f"labels[{i}] must be a non-empty str")
        if len(label) > MAX_LABEL_CHARS:
            raise ValueError(f"labels[{i}] has {len(label)} chars; ceiling is {MAX_LABEL_CHARS}")
    if len(set(labels)) != len(labels):
        raise ValueError("labels must be unique")
    bad_type = isinstance(threshold, bool) or not isinstance(threshold, int | float)
    if bad_type or not 0.0 <= threshold <= 1.0:
        raise ValueError("threshold must be a number in [0, 1]")
    return text, list(labels), float(threshold)


def validate_inputs(
    text: str,
    labels: Sequence[str],
    threshold: float = DEFAULT_THRESHOLD,
    *,
    names: Sequence[str] | None = None,
) -> dict[str, Any]:
    """Validation stage: return the input manifest (schema, observations, request, verdict).

    Rejection is reported by raising exactly as ``detect`` would; a caller that wants the finding
    recorded catches the exception and stores ``str(exc)`` under ``findings``. Both pinned snapshot
    identities are recorded, because this pipeline verifies two (GLiNER and its mDeBERTa encoder).
    """
    checked_text, checked_labels, checked_threshold = _check_inputs(text, labels, threshold)
    if names is not None and len(names) != 1:
        raise ValueError("names must have exactly one entry (detect takes one text)")
    return {
        "schema": dict(INPUT_SCHEMA),
        "inputs": [
            {
                "id": names[0] if names else "text-0",
                "chars": len(checked_text),
                "words": len(checked_text.split()),
            }
        ],
        "labels": checked_labels,
        "threshold": checked_threshold,
        "verdict": "accepted",
        "findings": [],
        "model_id": MODEL_ID,
        "model_revision": MODEL_REVISION,
        "encoder_model_id": ENCODER_MODEL_ID,
        "encoder_revision": ENCODER_REVISION,
    }


def evaluation_report(
    result: Mapping[str, Any],
    gold: Sequence[Mapping[str, Any]] | None = None,
    *,
    sample_kind: str = "synthetic",
) -> dict[str, Any]:
    """Evaluation stage: a machine-readable report even when nothing is measurable.

    With ``gold`` spans — dicts carrying ``start``, ``end`` and ``label`` — the report carries the
    repository's ``entity_f1`` (exact-span micro precision/recall/F1) as sample-sanity evidence;
    without them the verdict is ``not-measurable`` and the report says what labelled data would make
    the task measurable.
    """
    entities = list(result["entities"])
    base = {
        "task": "zero-shot named-entity recognition with a caller-supplied label set",
        "decision_rule": (
            f"a span is kept when its score reaches the caller's threshold "
            f"(default DEFAULT_THRESHOLD={DEFAULT_THRESHOLD}); the pipeline ships no tuned operating point"
        ),
        "score_semantics": (
            "each entity score is the model's own uncalibrated span score, not a probability that the "
            "span is correct"
        ),
        "threshold": result.get("threshold", DEFAULT_THRESHOLD),
        "labels": list(result.get("labels", [])),
        "sample_kind": sample_kind,
        "n_entities": len(entities),
        "baselines": [],
        "model_id": MODEL_ID,
        "model_revision": MODEL_REVISION,
        "encoder_model_id": ENCODER_MODEL_ID,
        "encoder_revision": ENCODER_REVISION,
    }
    if gold is None:
        return {
            **base,
            "metrics": [],
            "verdict": "not-measurable",
            "reason": "no gold spans were supplied for the evaluated text",
            "needs": (
                "gold (start, end, label) spans over the same label set on text from your domain, scored "
                "with entity_f1 across enough documents to state a dispersion; exact-span matching also "
                "requires the annotation guideline to agree with the model's span boundaries"
            ),
        }
    gold_spans = list(gold)
    scores = entity_f1(entities, gold_spans)
    return {
        **base,
        "metrics": [
            {
                "id": "entity_f1",
                "value": scores["f1"],
                "precision": scores["precision"],
                "recall": scores["recall"],
                "hits": scores["hits"],
                "n_predicted": len(entities),
                "n_gold": len(gold_spans),
                "matching": "exact (start, end, label) triple",
                "estimation": "one text, no dispersion estimate",
            }
        ],
        "verdict": "sample-sanity",
        "reason": (
            f"{len(gold_spans)} gold span(s) on one tutorial text; exact-span sanity evidence, not an "
            "NER benchmark"
        ),
        "needs": (
            "a labelled span set from the deployment domain, with the same label vocabulary and the same "
            "boundary convention, for any generalisable precision/recall/F1 claim"
        ),
    }


def _tensor_record(tensor: Any) -> dict[str, Any]:
    """Shape, dtype and SHA-256 of a tensor's bytes (the adapter manifest entry)."""
    data = tensor.detach().cpu().contiguous()
    return {
        "shape": list(data.shape),
        "dtype": str(data.dtype).replace("torch.", ""),
        "sha256": hashlib.sha256(data.numpy().tobytes()).hexdigest(),
    }


def _parameter_summary(model: Any) -> dict[str, Any]:
    """Trainable and frozen parameter counts, in total and per top-level module (two name components)."""
    trainable: dict[str, int] = {}
    frozen: dict[str, int] = {}
    for name, param in model.named_parameters():
        module = ".".join(name.split(".")[:2])
        bucket = trainable if param.requires_grad else frozen
        bucket[module] = bucket.get(module, 0) + int(param.numel())
    return {
        "trainable_parameters": sum(trainable.values()),
        "frozen_parameters": sum(frozen.values()),
        "trainable_modules": dict(sorted(trainable.items())),
        "frozen_modules": dict(sorted(frozen.items())),
    }


def _local_encoder_class(root: Path, encoder_dir: Path) -> type:
    """The concrete GLiNER class for this config, with `model_name` redirected to the verified encoder
    directory so the library reads the tokenizer and AutoConfig from disk (local_files_only, no cache)."""
    from gliner import GLiNER

    config_dict = json.loads((root / "gliner_config.json").read_text(encoding="utf-8"))
    base = GLiNER._get_gliner_class(GLiNER._config_from_dict(config_dict))

    class LocalEncoderGLiNER(base):
        @classmethod
        def _load_config(cls, config_file: Path, **overrides: Any) -> Any:
            config = super()._load_config(config_file, **overrides)
            if config.model_name != ENCODER_MODEL_ID:
                raise ValueError(f"gliner_config.json names encoder {config.model_name!r}, not the pinned id")
            config.model_name = str(encoder_dir)
            return config

    return LocalEncoderGLiNER


@dataclass
class GLiNERPipeline:
    """Zero-shot and adapted NER pipeline. `_runner(text, labels, threshold)` returns gliner entity dicts."""

    _runner: Callable[[str, list[str], float], list[dict[str, Any]]]
    device: str
    load_warnings: list[str] = field(default_factory=list)
    model: Any = None
    # True once `adapt` or `load_artifact` changed the weights in memory (no longer the pretrained model).
    adapted: bool = False

    @classmethod
    def from_pretrained(
        cls,
        device: str | None = None,
        weights_dir: str | Path | None = None,
        allow_download: bool = False,
        encoder_dir: str | Path | None = None,
    ) -> GLiNERPipeline:
        root = Path(weights_dir) if weights_dir is not None else DEFAULT_WEIGHTS_DIR
        enc = Path(encoder_dir) if encoder_dir is not None else ENCODER_WEIGHTS_DIR
        with warnings.catch_warnings(record=True) as caught:
            warnings.simplefilter("always")
            if (root / MANIFEST_NAME).is_file():
                stage_missing_files(root, allow_download=allow_download)
                verify_snapshot(root)
                stage_missing_encoder_files(enc, allow_download=allow_download)
                verify_encoder_snapshot(enc)
                loader = _local_encoder_class(root, enc)
                model = loader.from_pretrained(
                    str(root), model_dir=str(root), local_files_only=True, map_location="cpu"
                )
            elif allow_download:
                from gliner import GLiNER  # Hub path: the library fetches the encoder assets unpinned

                model = GLiNER.from_pretrained(MODEL_ID, revision=MODEL_REVISION, map_location="cpu")
            else:
                raise FileNotFoundError(f"no verified snapshot at {root} and allow_download=False")
        # Validate and verify the snapshots before importing model libraries.
        import torch
        resolved_device = device or ("cuda:0" if torch.cuda.is_available() else "cpu")
        model = model.to(resolved_device).eval()
        messages = [f"{w.category.__name__}: {w.message}" for w in caught]

        def runner(text: str, labels: list[str], threshold: float) -> list[dict[str, Any]]:
            with torch.inference_mode():
                return model.predict_entities(text, labels, threshold=threshold)

        return cls(runner, resolved_device, messages, model=model)

    def detect(
        self,
        text: str,
        labels: Sequence[str],
        threshold: float = DEFAULT_THRESHOLD,
    ) -> dict[str, Any]:
        """Extract spans for the caller-supplied `labels`; `threshold` is the upstream score cutoff."""
        text, labels, threshold = _check_inputs(text, labels, threshold)
        if self.model is not None:
            import torch

            with torch.inference_mode():
                raw = self.model.predict_entities(text, list(labels), threshold=threshold)
        else:
            raw = self._runner(text, list(labels), threshold)

        entities = []
        for e in raw:
            start, end = int(e["start"]), int(e["end"])
            if not 0 <= start < end <= len(text) or e["label"] not in labels:
                raise RuntimeError(f"backend returned an invalid entity: {e}")
            entities.append(
                {
                    "text": text[start:end],
                    "label": str(e["label"]),
                    "start": start,
                    "end": end,
                    "score": float(e["score"]),
                }
            )
        return {
            "entities": entities,
            "n_entities": len(entities),
            "labels": list(labels),
            "threshold": threshold,
            "model_id": MODEL_ID,
            "model_revision": MODEL_REVISION,
            "encoder_model_id": ENCODER_MODEL_ID,
            "encoder_revision": ENCODER_REVISION,
        }

    def evaluate(
        self,
        records: Sequence[dict[str, Any]],
        labels: Sequence[str] | None = None,
        threshold: float = DEFAULT_THRESHOLD,
    ) -> dict[str, Any]:
        """Evaluate exact-span metrics over a dataset of NER records."""
        pass  # standalone rewrite (build_notebook.py): `from .metrics import evaluate_ner_dataset` removed — names are kernel globals defined by the carried modules
        pass  # standalone rewrite (build_notebook.py): `from .samples import validate_dataset` removed — names are kernel globals defined by the carried modules

        if labels is None:
            extracted_labels = sorted({
                s["label"]
                for r in records
                for s in (r.get("spans") or r.get("entities", []))
            })
            if not extracted_labels:
                raise ValueError(
                    "Cannot infer labels: records contain no entity spans and labels was not provided"
                )
            eval_labels = extracted_labels
        else:
            eval_labels = list(labels)
        # An evaluation set must use only the evaluated labels, but need not contain every one of them.
        validate_dataset(records, eval_labels, require_all_labels=False, min_records=1)

        predictions: list[list[dict[str, Any]]] = []
        for r in records:
            res = self.detect(r["text"], eval_labels, threshold=threshold)
            predictions.append(res["entities"])

        metrics = evaluate_ner_dataset(predictions, records, eval_labels)
        return {
            "micro": {
                "precision": metrics["micro_precision"],
                "recall": metrics["micro_recall"],
                "f1": metrics["micro_f1"],
                "hits": metrics["hits"],
            },
            "macro": {
                "f1": metrics["macro_f1"],
            },
            "per_class": metrics["per_class"],
            "num_samples": len(records),
            "labels": eval_labels,
            "threshold": threshold,
        }

    def adapt(
        self,
        train_records: Sequence[dict[str, Any]],
        val_records: Sequence[dict[str, Any]] | None = None,
        *,
        epochs: int = 3,
        learning_rate: float = 5e-5,
        batch_size: int = 4,
        freeze_text_encoder: bool = True,
        weight_decay: float = 0.01,
        threshold: float = DEFAULT_THRESHOLD,
        seed: int = 42,
        labels: Sequence[str] | None = None,
    ) -> dict[str, Any]:
        """Run bounded fine-tuning loop over domain records without external dependencies.

        `labels` is the entity-type set (default: every label in the training and validation records); each
        must occur in `train_records`. A pipeline that was already adapted is refused: reload the pretrained
        model first, so one call is one training run from the verified base.
        """
        if self.model is None:
            raise RuntimeError("Cannot adapt: pipeline has no underlying PyTorch model loaded.")
        if self.adapted:
            raise RuntimeError(
                "Cannot adapt: this pipeline was already adapted (or loaded an adapter) in memory, so "
                "training would continue from those weights. Reload the pretrained model with "
                "GLiNERPipeline.from_pretrained first."
            )
        pass  # standalone rewrite (build_notebook.py): `from .samples import _labels_in, validate_dataset` removed — names are kernel globals defined by the carried modules

        if not train_records:
            raise ValueError("train_records cannot be empty")
        all_labels = sorted(labels) if labels is not None else _labels_in(
            list(train_records) + (list(val_records) if val_records else [])
        )
        validate_dataset(train_records, all_labels)
        if val_records is not None:
            validate_dataset(val_records, all_labels, require_all_labels=False, min_records=1)

        import random

        import torch

        random.seed(seed)
        torch.manual_seed(seed)
        if torch.cuda.is_available():
            torch.cuda.manual_seed_all(seed)

        if freeze_text_encoder:
            if hasattr(self.model, "freeze_component"):
                self.model.freeze_component("text_encoder")
            elif hasattr(self.model, "model") and hasattr(self.model.model, "token_rep_layer"):
                for p in self.model.model.token_rep_layer.parameters():
                    p.requires_grad = False

        trainable_params = [p for p in self.model.parameters() if p.requires_grad]
        if not trainable_params:
            raise RuntimeError("No trainable parameters found for adaptation.")
        parameter_summary = _parameter_summary(self.model)
        self.adapted = True  # from here on the weights in memory are no longer the pretrained ones

        optimizer = torch.optim.AdamW(trainable_params, lr=learning_rate, weight_decay=weight_decay)
        collator = self.model._create_data_collator()

        history: list[dict[str, Any]] = []
        n_train = len(train_records)
        train_examples = [
            {"tokenized_text": r["tokenized_text"], "ner": r["ner"]}
            for r in train_records
        ]

        self.model.train()
        for epoch in range(1, epochs + 1):
            epoch_loss = 0.0
            num_batches = 0

            indices = list(range(n_train))
            random.shuffle(indices)

            for i in range(0, n_train, batch_size):
                batch_indices = indices[i : i + batch_size]
                batch_data = [train_examples[idx] for idx in batch_indices]
                batch = collator(batch_data)
                batch = {
                    k: v.to(self.device) if hasattr(v, "to") else v
                    for k, v in batch.items()
                }

                optimizer.zero_grad()
                output = self.model(**batch)
                loss = output.loss
                loss.backward()
                optimizer.step()

                epoch_loss += float(loss.item())
                num_batches += 1

            avg_loss = epoch_loss / max(1, num_batches)
            epoch_summary: dict[str, Any] = {
                "epoch": epoch,
                "loss": round(avg_loss, 6),
                "num_batches": num_batches,
            }

            if val_records:
                self.model.eval()
                val_res = self.evaluate(val_records, labels=all_labels, threshold=threshold)
                epoch_summary["val_f1"] = val_res["micro"]["f1"]
                epoch_summary["val_precision"] = val_res["micro"]["precision"]
                epoch_summary["val_recall"] = val_res["micro"]["recall"]
                self.model.train()

            history.append(epoch_summary)

        self.model.eval()

        final_eval: dict[str, Any] | None = None
        if val_records:
            final_eval = self.evaluate(val_records, labels=all_labels, threshold=threshold)

        return {
            "epochs": epochs,
            "learning_rate": learning_rate,
            "batch_size": batch_size,
            "freeze_text_encoder": freeze_text_encoder,
            "weight_decay": weight_decay,
            "seed": seed,
            "labels": all_labels,
            **parameter_summary,
            "train_samples": n_train,
            "val_samples": len(val_records) if val_records else 0,
            "history": history,
            "final_eval": final_eval,
        }

    def save_artifact(
        self,
        output_path: str | Path,
        metadata: dict[str, Any] | None = None,
    ) -> Path:
        """Export adapter state dict and lineage metadata to a .pt artifact."""
        if self.model is None:
            raise RuntimeError("Cannot save artifact: pipeline has no model loaded.")
        import torch

        path = Path(output_path)
        path.parent.mkdir(parents=True, exist_ok=True)

        trainable_names = {name for name, p in self.model.named_parameters() if p.requires_grad}
        if trainable_names:
            adapter_weights = {
                k: v.detach().cpu().clone()
                for k, v in self.model.state_dict().items()
                if k in trainable_names
            }
        else:
            adapter_weights = {
                k: v.detach().cpu().clone()
                for k, v in self.model.state_dict().items()
            }

        payload: dict[str, Any] = {
            "format": ARTIFACT_FORMAT,
            "format_version": ARTIFACT_FORMAT_VERSION,
            "base_model": {
                "model_id": MODEL_ID,
                "model_revision": MODEL_REVISION,
                "encoder_model_id": ENCODER_MODEL_ID,
                "encoder_revision": ENCODER_REVISION,
            },
            "adapter_state_dict": adapter_weights,
            "adapter_manifest": {name: _tensor_record(t) for name, t in sorted(adapter_weights.items())},
            "metadata": metadata or {},
        }
        torch.save(payload, path)
        return path

    def load_artifact(self, artifact_path: str | Path) -> dict[str, Any]:
        """Load adapter state dict into the current pipeline's model."""
        if self.model is None:
            raise RuntimeError("Cannot load artifact: pipeline has no model loaded.")
        import torch

        path = Path(artifact_path)
        if not path.is_file():
            raise FileNotFoundError(f"Artifact file not found: {path}")

        payload = torch.load(path, map_location=self.device, weights_only=True)

        if not isinstance(payload, dict):
            raise ValueError(f"Invalid artifact format: expected dict, got {type(payload).__name__}")
        if payload.get("format") != ARTIFACT_FORMAT:
            raise ValueError(
                f"Artifact format mismatch: {payload.get('format')!r} != {ARTIFACT_FORMAT!r}"
            )
        base = payload.get("base_model", {})
        if base.get("model_id") != MODEL_ID:
            raise ValueError(
                f"Artifact base model mismatch: {base.get('model_id')!r} != {MODEL_ID!r}"
            )
        expected_base = {
            "model_revision": MODEL_REVISION,
            "encoder_model_id": ENCODER_MODEL_ID,
            "encoder_revision": ENCODER_REVISION,
        }
        for key, value in expected_base.items():
            if base.get(key) != value:
                raise ValueError(
                    f"Artifact base {key} mismatch: {base.get(key)!r} != {value!r}; the adapter was trained "
                    "on a different base snapshot and cannot be overlaid on this one"
                )

        weights = payload.get("adapter_state_dict")
        if not isinstance(weights, dict) or not weights:
            raise ValueError("Artifact adapter_state_dict is empty or missing: no adapter tensors to load")
        model_state = self.model.state_dict()
        unexpected = sorted(k for k in weights if k not in model_state)
        if unexpected:
            raise ValueError(
                f"Artifact holds {len(unexpected)} tensor name(s) this model does not have "
                f"(e.g. {unexpected[:3]}): the adapter is for another architecture or its keys were renamed"
            )
        mismatched = sorted(k for k, v in weights.items() if tuple(v.shape) != tuple(model_state[k].shape))
        if mismatched:
            raise ValueError(f"Artifact tensor shapes differ from the model for {mismatched[:3]}")
        manifest = payload.get("adapter_manifest")
        if manifest is not None or payload.get("format_version") != "1.0":
            if not isinstance(manifest, dict) or sorted(manifest) != sorted(weights):
                raise ValueError("Artifact adapter_manifest is missing or does not list exactly its tensors")
            corrupt = sorted(k for k, v in weights.items() if _tensor_record(v) != manifest[k])
            if corrupt:
                raise ValueError(f"Adapter tensors differ from manifest: {corrupt[:3]}")

        weights = {
            k: v.to(self.device) if hasattr(v, "to") else v
            for k, v in weights.items()
        }
        result = self.model.load_state_dict(weights, strict=False)
        # strict=False only because the adapter is a subset of the model; every tensor must be consumed.
        if list(getattr(result, "unexpected_keys", [])):
            raise ValueError(f"Adapter tensors not consumed by the model: {list(result.unexpected_keys)[:3]}")
        self.adapted = True
        self.model.eval()
        return payload.get("metadata", {})

    def adapter_parity(self, other: GLiNERPipeline, names: Sequence[str]) -> dict[str, Any]:
        """Compare the named tensors of this pipeline's model with `other`'s exactly (reload check)."""
        if self.model is None or other.model is None:
            raise RuntimeError("Both pipelines need a loaded model to compare adapter tensors.")
        mine, theirs = self.model.state_dict(), other.model.state_dict()
        differing = [n for n in names if _tensor_record(mine[n]) != _tensor_record(theirs[n])]
        return {
            "tensors_compared": len(names),
            "tensors_identical": len(names) - len(differing),
            "differing": differing[:5],
        }

    @classmethod
    def from_artifact(
        cls,
        artifact_path: str | Path,
        device: str | None = None,
        weights_dir: str | Path | None = None,
        allow_download: bool = False,
        encoder_dir: str | Path | None = None,
    ) -> GLiNERPipeline:
        """Construct GLiNERPipeline from base weights and overlay adapter artifact weights."""
        pipeline = cls.from_pretrained(
            device=device,
            weights_dir=weights_dir,
            allow_download=allow_download,
            encoder_dir=encoder_dir,
        )
        pipeline.load_artifact(artifact_path)
        return pipeline

**Module 3/3:** `src/gliner_ner_pipeline/samples.py` (carried verbatim; see the note above)

In [ ]:
# @title Infrastructure: carried module `src/gliner_ner_pipeline/samples.py` (you may run it without reading it)
"""Deterministic in-code sample data and contracts for GLiNER multi-v2.1 domain adaptation.

Provides deterministic synthetic domain NER datasets (biomedical/clinical entities: disease,
chemical_drug, gene_protein) and BYOD validation routines conforming to DIMER NOTEBOOK_SPEC 2.2.
The sample uses the three biomedical labels; a BYOD dataset may use any label set (`allowed_labels=None`
infers it from the records), and every declared label must occur in the training split.
"""
# ruff: noqa: E501

from __future__ import annotations

import json
import random
from collections.abc import Sequence
from pathlib import Path
from typing import Any

# Canonical biomedical / clinical NER adaptation vocabulary
ADAPT_CLASSES: tuple[str, ...] = ("disease", "chemical_drug", "gene_protein")
DATASET_REPRESENTATION = "io.github.kurtvalcorza.dataset.nlp.ner-spans.v1"

# Pinned ceilings from pipeline.py / gliner_config.json
MIN_DATASET_EXAMPLES = 4  # per split: the training and the validation split each need at least this many records
MAX_DATASET_EXAMPLES = 1_000
MAX_TOKENS_PER_EXAMPLE = 384
MAX_TOKEN_CHARS = 100
BYOD_SUFFIXES = (".json", ".jsonl")
RECORD_SCHEMA_HINT = (
    "each record is {id: str, tokenized_text (or tokens): [str, ...], ner: [[start_token, end_token, label], ...]} "
    "with token indices counted from 0 and an inclusive end"
)

TUTORIAL_TEXT = "Marie Curie conducted pioneering research on radioactivity in Paris and Warsaw."
TUTORIAL_LABELS = ["person", "location", "scientific_field"]
TUTORIAL_SPANS = [
    {"start": 0, "end": 11, "label": "person", "text": "Marie Curie"},
    {"start": 62, "end": 67, "label": "location", "text": "Paris"},
    {"start": 72, "end": 78, "label": "location", "text": "Warsaw"},
]

# 24 deterministic domain sentences covering disease, chemical_drug, gene_protein
_RAW_DOMAIN_EXAMPLES: list[dict[str, Any]] = [
    {
        "id": "bio-000",
        "tokens": ["Aspirin", "inhibits", "PTGS2", "to", "reduce", "inflammation", "and", "pain", "."],
        "ner": [
            [0, 0, "chemical_drug"],
            [2, 2, "gene_protein"],
            [5, 5, "disease"],
            [7, 7, "disease"],
        ],
    },
    {
        "id": "bio-001",
        "tokens": ["Imatinib", "targets", "BCR-ABL1", "fusion", "in", "chronic", "myeloid", "leukemia", "."],
        "ner": [
            [0, 0, "chemical_drug"],
            [2, 2, "gene_protein"],
            [5, 7, "disease"],
        ],
    },
    {
        "id": "bio-002",
        "tokens": ["Trastuzumab", "binds", "ERBB2", "receptors", "overexpressed", "in", "breast", "cancer", "."],
        "ner": [
            [0, 0, "chemical_drug"],
            [2, 2, "gene_protein"],
            [6, 7, "disease"],
        ],
    },
    {
        "id": "bio-003",
        "tokens": ["Metformin", "activates", "PRKAA1", "for", "glycemic", "control", "in", "type", "2", "diabetes", "."],
        "ner": [
            [0, 0, "chemical_drug"],
            [2, 2, "gene_protein"],
            [7, 9, "disease"],
        ],
    },
    {
        "id": "bio-004",
        "tokens": ["Erlotinib", "inhibits", "mutant", "EGFR", "in", "non-small", "cell", "lung", "carcinoma", "."],
        "ner": [
            [0, 0, "chemical_drug"],
            [3, 3, "gene_protein"],
            [5, 8, "disease"],
        ],
    },
    {
        "id": "bio-005",
        "tokens": ["Pembrolizumab", "blocks", "PDCD1", "pathway", "activation", "in", "metastatic", "melanoma", "."],
        "ner": [
            [0, 0, "chemical_drug"],
            [2, 2, "gene_protein"],
            [6, 7, "disease"],
        ],
    },
    {
        "id": "bio-006",
        "tokens": ["Rituximab", "targets", "MS4A1", "surface", "antigen", "on", "cells", "in", "lymphoma", "."],
        "ner": [
            [0, 0, "chemical_drug"],
            [2, 2, "gene_protein"],
            [8, 8, "disease"],
        ],
    },
    {
        "id": "bio-007",
        "tokens": ["Olaparib", "inhibits", "PARP1", "in", "tumors", "harboring", "mutant", "BRCA1", "ovarian", "cancer", "."],
        "ner": [
            [0, 0, "chemical_drug"],
            [2, 2, "gene_protein"],
            [7, 7, "gene_protein"],
            [8, 9, "disease"],
        ],
    },
    {
        "id": "bio-008",
        "tokens": ["Vemurafenib", "selectively", "inhibits", "mutated", "BRAF", "kinase", "in", "cutaneous", "melanoma", "."],
        "ner": [
            [0, 0, "chemical_drug"],
            [4, 4, "gene_protein"],
            [7, 8, "disease"],
        ],
    },
    {
        "id": "bio-009",
        "tokens": ["Dabrafenib", "combined", "with", "Trametinib", "delays", "resistance", "in", "thyroid", "carcinoma", "."],
        "ner": [
            [0, 0, "chemical_drug"],
            [3, 3, "chemical_drug"],
            [7, 8, "disease"],
        ],
    },
    {
        "id": "bio-010",
        "tokens": ["Bevacizumab", "binds", "circulating", "VEGFA", "to", "suppress", "angiogenesis", "in", "glioblastoma", "."],
        "ner": [
            [0, 0, "chemical_drug"],
            [3, 3, "gene_protein"],
            [8, 8, "disease"],
        ],
    },
    {
        "id": "bio-011",
        "tokens": ["Sunitinib", "inhibits", "KDR", "and", "PDGFRB", "in", "renal", "cell", "carcinoma", "."],
        "ner": [
            [0, 0, "chemical_drug"],
            [2, 2, "gene_protein"],
            [4, 4, "gene_protein"],
            [6, 8, "disease"],
        ],
    },
    {
        "id": "bio-012",
        "tokens": ["Paclitaxel", "stabilizes", "TUBB", "microtubules", "to", "induce", "apoptosis", "in", "pancreatic", "cancer", "."],
        "ner": [
            [0, 0, "chemical_drug"],
            [2, 2, "gene_protein"],
            [8, 9, "disease"],
        ],
    },
    {
        "id": "bio-013",
        "tokens": ["Cisplatin", "forms", "DNA", "crosslinks", "inducing", "cell", "death", "in", "testicular", "cancer", "."],
        "ner": [
            [0, 0, "chemical_drug"],
            [8, 9, "disease"],
        ],
    },
    {
        "id": "bio-014",
        "tokens": ["Doxorubicin", "intercalates", "DNA", "and", "inhibits", "TOP2A", "in", "sarcoma", "."],
        "ner": [
            [0, 0, "chemical_drug"],
            [5, 5, "gene_protein"],
            [7, 7, "disease"],
        ],
    },
    {
        "id": "bio-015",
        "tokens": ["Tamoxifen", "antagonizes", "ESR1", "signaling", "in", "estrogen-receptor-positive", "breast", "cancer", "."],
        "ner": [
            [0, 0, "chemical_drug"],
            [2, 2, "gene_protein"],
            [6, 7, "disease"],
        ],
    },
    {
        "id": "bio-016",
        "tokens": ["Anastrozole", "suppresses", "CYP19A1", "reducing", "estrogen", "synthesis", "in", "postmenopausal", "breast", "cancer", "."],
        "ner": [
            [0, 0, "chemical_drug"],
            [2, 2, "gene_protein"],
            [8, 9, "disease"],
        ],
    },
    {
        "id": "bio-017",
        "tokens": ["Bortezomib", "disrupts", "PSMB5", "proteasome", "activity", "in", "multiple", "myeloma", "."],
        "ner": [
            [0, 0, "chemical_drug"],
            [2, 2, "gene_protein"],
            [6, 7, "disease"],
        ],
    },
    {
        "id": "bio-018",
        "tokens": ["Lenalidomide", "modulates", "CRBN", "ubiquitin", "ligase", "complex", "in", "myelodysplastic", "syndrome", "."],
        "ner": [
            [0, 0, "chemical_drug"],
            [2, 2, "gene_protein"],
            [7, 8, "disease"],
        ],
    },
    {
        "id": "bio-019",
        "tokens": ["Ibrutinib", "irreversibly", "binds", "BTK", "blocking", "B-cell", "activation", "in", "mantle", "cell", "lymphoma", "."],
        "ner": [
            [0, 0, "chemical_drug"],
            [3, 3, "gene_protein"],
            [8, 10, "disease"],
        ],
    },
    {
        "id": "bio-020",
        "tokens": ["Ruxolitinib", "inhibits", "JAK1", "and", "JAK2", "signaling", "in", "primary", "myelofibrosis", "."],
        "ner": [
            [0, 0, "chemical_drug"],
            [2, 2, "gene_protein"],
            [4, 4, "gene_protein"],
            [7, 8, "disease"],
        ],
    },
    {
        "id": "bio-021",
        "tokens": ["Gefitinib", "blocks", "tyrosine", "kinase", "phosphorylation", "of", "EGFR", "in", "pulmonary", "adenocarcinoma", "."],
        "ner": [
            [0, 0, "chemical_drug"],
            [6, 6, "gene_protein"],
            [8, 9, "disease"],
        ],
    },
    {
        "id": "bio-022",
        "tokens": ["Lapatinib", "co-inhibits", "EGFR", "and", "ERBB2", "pathways", "in", "metastatic", "breast", "cancer", "."],
        "ner": [
            [0, 0, "chemical_drug"],
            [2, 2, "gene_protein"],
            [4, 4, "gene_protein"],
            [7, 9, "disease"],
        ],
    },
    {
        "id": "bio-023",
        "tokens": ["Crizotinib", "inhibits", "ALK", "and", "ROS1", "rearrangements", "in", "anaplastic", "large", "cell", "lymphoma", "."],
        "ner": [
            [0, 0, "chemical_drug"],
            [2, 2, "gene_protein"],
            [4, 4, "gene_protein"],
            [7, 10, "disease"],
        ],
    },
]


def _build_char_spans(tokens: list[str], ner_triples: list[list[Any]]) -> tuple[str, list[dict[str, Any]]]:
    """Reconstruct text and calculate exact character offsets from token spans."""
    token_offsets: list[tuple[int, int]] = []
    text_parts: list[str] = []
    current_char = 0
    for idx, token in enumerate(tokens):
        if idx > 0 and token not in {".", ",", ";", ":", ")", "]"}:
            # Add space before token unless it's closing punctuation
            text_parts.append(" ")
            current_char += 1
        start_char = current_char
        text_parts.append(token)
        current_char += len(token)
        token_offsets.append((start_char, current_char))

    full_text = "".join(text_parts)
    spans: list[dict[str, Any]] = []
    for start_t, end_t, label in ner_triples:
        char_start = token_offsets[start_t][0]
        char_end = token_offsets[end_t][1]
        spans.append(
            {
                "start": char_start,
                "end": char_end,
                "label": str(label),
                "text": full_text[char_start:char_end],
            }
        )
    return full_text, spans


def generate_synthetic_ner_dataset() -> list[dict[str, Any]]:
    """Return the deterministic 24-sentence biomedical/clinical NER dataset."""
    records: list[dict[str, Any]] = []
    for item in _RAW_DOMAIN_EXAMPLES:
        full_text, spans = _build_char_spans(item["tokens"], item["ner"])
        records.append(
            {
                "id": item["id"],
                "tokenized_text": list(item["tokens"]),
                "ner": [list(span) for span in item["ner"]],
                "text": full_text,
                "spans": spans,
            }
        )
    return records


def split_ner_dataset(
    records: Sequence[dict[str, Any]],
    val_fraction: float = 0.25,
    seed: int = 42,
    labels: Sequence[str] | None = None,
) -> tuple[list[dict[str, Any]], list[dict[str, Any]]]:
    """Split dataset into disjoint train and validation sets with a seeded shuffle (no stratification).

    Each split must hold at least MIN_DATASET_EXAMPLES records, and every label in `labels` (default: every
    label found in the records) must occur in the training split; otherwise a ValueError names the split or
    the label and what to change. A label may still be absent from the validation split: see
    `split_label_coverage`.
    """
    if not 0.0 < val_fraction < 1.0:
        raise ValueError(f"val_fraction must be in (0, 1), got {val_fraction}")
    n_val = max(1, round(len(records) * val_fraction))
    n_train = len(records) - n_val
    if n_val < MIN_DATASET_EXAMPLES or n_train < MIN_DATASET_EXAMPLES:
        small = "validation" if n_val < MIN_DATASET_EXAMPLES else "training"
        raise ValueError(
            f"the {small} split would hold {min(n_val, n_train)} records (train {n_train}, validation {n_val} of "
            f"{len(records)} at val_fraction={val_fraction}); each split needs at least {MIN_DATASET_EXAMPLES}. "
            f"Supply at least {minimum_records(val_fraction)} records at this val_fraction, or change VAL_FRACTION."
        )

    record_list = [dict(r) for r in records]
    rng = random.Random(seed)
    rng.shuffle(record_list)

    val_records = record_list[:n_val]
    train_records = record_list[n_val:]

    wanted = _labels_in(records) if labels is None else list(labels)
    in_train = _labels_in(train_records)
    absent = [lbl for lbl in wanted if lbl not in in_train]
    if absent:
        raise ValueError(
            f"label(s) {absent} occur in no training record with seed={seed}; the model cannot learn them. "
            "Add training examples of these labels, or change SEED."
        )
    return train_records, val_records


def minimum_records(val_fraction: float = 0.25) -> int:
    """Smallest dataset size whose seeded split leaves MIN_DATASET_EXAMPLES records in each split."""
    if not 0.0 < val_fraction < 1.0:
        raise ValueError(f"val_fraction must be in (0, 1), got {val_fraction}")
    n = 2 * MIN_DATASET_EXAMPLES
    while True:
        n_val = max(1, round(n * val_fraction))
        if n_val >= MIN_DATASET_EXAMPLES and n - n_val >= MIN_DATASET_EXAMPLES:
            return n
        n += 1


def _labels_in(records: Sequence[dict[str, Any]]) -> list[str]:
    """Sorted labels of the well-formed `[start, end, label]` spans in `records`."""
    found = {
        span[2]
        for r in records
        if isinstance(r, dict)
        for span in (r.get("ner") or [])
        if isinstance(span, list | tuple) and len(span) == 3 and isinstance(span[2], str) and span[2].strip()
    }
    return sorted(found)


def split_label_coverage(
    train_records: Sequence[dict[str, Any]],
    val_records: Sequence[dict[str, Any]],
    labels: Sequence[str],
) -> dict[str, Any]:
    """Span counts per label in each split, and the labels the validation split cannot measure."""

    def counts(records: Sequence[dict[str, Any]]) -> dict[str, int]:
        out = dict.fromkeys(labels, 0)
        for r in records:
            for span in r.get("ner", []):
                if span[2] in out:
                    out[span[2]] += 1
        return out

    train, val = counts(train_records), counts(val_records)
    return {
        "train_spans": train,
        "validation_spans": val,
        "labels_missing_from_validation": [lbl for lbl in labels if val[lbl] == 0],
    }


def validate_dataset(
    records: Sequence[dict[str, Any]],
    allowed_labels: Sequence[str] = ADAPT_CLASSES,
    *,
    require_all_labels: bool = True,
    min_records: int = MIN_DATASET_EXAMPLES,
) -> dict[str, Any]:
    """Validate that NER records conform strictly to tokenized span schema and ceilings.

    Every span label must be one of `allowed_labels`. With `require_all_labels` (the default) every allowed label
    must also occur at least once; evaluation sets are checked with `require_all_labels=False`.
    """
    if not isinstance(records, Sequence) or isinstance(records, str | bytes):
        raise TypeError(f"records must be a sequence of dicts, got {type(records).__name__}")
    if len(records) < min_records:
        raise ValueError(f"dataset requires at least {min_records} records, got {len(records)}")
    if len(records) > MAX_DATASET_EXAMPLES:
        raise ValueError(f"dataset exceeds ceiling of {MAX_DATASET_EXAMPLES} records, got {len(records)}")

    label_set = set(allowed_labels)
    seen_ids: set[str] = set()
    label_counts: dict[str, int] = {lbl: 0 for lbl in label_set}
    total_spans = 0

    for idx, r in enumerate(records):
        if not isinstance(r, dict):
            raise TypeError(f"record[{idx}] must be a dict, got {type(r).__name__}; {RECORD_SCHEMA_HINT}")
        if "id" not in r:
            raise KeyError(f"record[{idx}] missing required key 'id'; {RECORD_SCHEMA_HINT}")
        if "tokenized_text" not in r:
            raise KeyError(f"record[{idx}] missing required key 'tokenized_text'; {RECORD_SCHEMA_HINT}")
        if "ner" not in r:
            raise KeyError(f"record[{idx}] missing required key 'ner'; {RECORD_SCHEMA_HINT}")

        doc_id = str(r["id"]).strip()
        if not doc_id:
            raise ValueError(f"record[{idx}] id cannot be empty")
        if doc_id in seen_ids:
            raise ValueError(f"record[{idx}] duplicates id {doc_id!r}")
        seen_ids.add(doc_id)

        tokens = r["tokenized_text"]
        if not isinstance(tokens, list) or not tokens:
            raise TypeError(f"record[{idx}] tokenized_text must be a non-empty list of tokens")
        if len(tokens) > MAX_TOKENS_PER_EXAMPLE:
            raise ValueError(f"record[{idx}] has {len(tokens)} tokens; ceiling is {MAX_TOKENS_PER_EXAMPLE}")
        for t_idx, token in enumerate(tokens):
            if not isinstance(token, str) or not token.strip():
                raise TypeError(f"record[{idx}] token[{t_idx}] must be a non-empty string")
            if len(token) > MAX_TOKEN_CHARS:
                raise ValueError(f"record[{idx}] token[{t_idx}] exceeds {MAX_TOKEN_CHARS} chars")

        ner_spans = r["ner"]
        if not isinstance(ner_spans, list):
            raise TypeError(f"record[{idx}] ner must be a list of spans")

        occupied_tokens: set[int] = set()
        for span_idx, span in enumerate(ner_spans):
            if not isinstance(span, list | tuple) or len(span) != 3:
                raise ValueError(
                    f"record[{idx}] span[{span_idx}] must be a list [start_token, end_token, label] (inclusive end, "
                    f'e.g. [0, 1, "person"]), got {type(span).__name__} {str(span)[:80]}'
                )
            start, end, label = span
            if isinstance(start, bool) or not isinstance(start, int) or start < 0:
                raise TypeError(f"record[{idx}] span[{span_idx}] start must be non-negative int")
            if isinstance(end, bool) or not isinstance(end, int) or end < start:
                raise ValueError(f"record[{idx}] span[{span_idx}] end must be >= start")
            if end >= len(tokens):
                raise ValueError(
                    f"record[{idx}] span[{span_idx}] end index {end} out of bounds for {len(tokens)} tokens"
                )
            if not isinstance(label, str) or not label.strip():
                raise TypeError(f"record[{idx}] span[{span_idx}] label must be non-empty string")
            if label not in label_set:
                raise ValueError(f"record[{idx}] span[{span_idx}] label {label!r} not in {sorted(label_set)}")

            # Check for overlapping spans
            span_range = set(range(start, end + 1))
            if span_range & occupied_tokens:
                raise ValueError(f"record[{idx}] span[{span_idx}] overlaps with an existing entity span")
            occupied_tokens.update(span_range)

            label_counts[label] += 1
            total_spans += 1

    missing_labels = sorted(lbl for lbl, count in label_counts.items() if count == 0)
    if missing_labels and require_all_labels:
        raise ValueError(
            f"dataset missing examples for required labels: {missing_labels}; remove them from the declared "
            "label set or add examples"
        )

    return {
        "verdict": "accepted",
        "representation": DATASET_REPRESENTATION,
        "n_records": len(records),
        "n_spans": total_spans,
        "label_distribution": label_counts,
        "labels": list(allowed_labels),
    }


def load_byod_dataset(
    source: str | Path,
    allowed_labels: Sequence[str] | None = ADAPT_CLASSES,
) -> list[dict[str, Any]]:
    """Load and validate a user-supplied JSON or JSONL NER dataset.

    `allowed_labels=None` takes the label set from the records themselves (every label that occurs); otherwise
    every span label must be one of `allowed_labels` and each of them must occur.
    """
    path = Path(source)
    if not path.is_file():
        raise FileNotFoundError(f"BYOD dataset file not found: {path}")
    if path.suffix.lower() not in BYOD_SUFFIXES:
        raise ValueError(
            f"BYOD expects a .json file (one JSON array of records) or a .jsonl file (one record per line), got "
            f"{path.name!r}; {RECORD_SCHEMA_HINT}. Convert the file (for example from CSV) and supply it again."
        )

    raw_text = path.read_text(encoding="utf-8").strip()
    if not raw_text:
        raise ValueError(f"BYOD dataset file is empty: {path}")

    records: list[dict[str, Any]] = []
    if path.suffix.lower() == ".jsonl":
        for line_no, line in enumerate(raw_text.splitlines(), start=1):
            line = line.strip()
            if not line:
                continue
            try:
                item = json.loads(line)
            except json.JSONDecodeError as exc:
                raise ValueError(
                    f"{path.name} line {line_no} is not valid JSON ({exc}); a .jsonl file holds one JSON object per line"
                ) from exc
            records.append(item)
    else:
        try:
            data = json.loads(raw_text)
        except json.JSONDecodeError as exc:
            raise ValueError(
                f"{path.name} is not valid JSON ({exc}); a .json file holds one array of records: {RECORD_SCHEMA_HINT}"
            ) from exc
        if not isinstance(data, list):
            raise TypeError(f"JSON dataset must contain a top-level array of objects, got {type(data).__name__}")
        records = data

    for r in records:
        if isinstance(r, dict) and "tokens" in r and "tokenized_text" not in r:
            r["tokenized_text"] = r.pop("tokens")
    labels = _labels_in(records) if allowed_labels is None else list(allowed_labels)
    if not labels:
        # No well-formed span anywhere: report the first malformed record or span, if there is one.
        validate_dataset(records, [], require_all_labels=False, min_records=0)
        raise ValueError(f"no entity labels found in {path.name}; {RECORD_SCHEMA_HINT}")
    # Validate before deriving text and character spans, so a malformed span gets the schema message.
    validate_dataset(records, labels)
    for r in records:
        if "text" not in r:
            full_text, spans = _build_char_spans(r["tokenized_text"], r["ner"])
            r["text"] = full_text
            r["spans"] = spans
    return records


# Pre-built immutable instance of the synthetic biomedical dataset
SAMPLE_BIOMEDICAL_DATASET: list[dict[str, Any]] = generate_synthetic_ner_dataset()

## 3. Pin, stage and verify the model

> **Infrastructure.** You may run this section without studying its implementation; the learning activities start after Section 3.

The model identity is carried twice — `MODEL_ID`/`MODEL_REVISION` in the module above and the `3`-file manifest below (paths, byte sizes, SHA-256) — and the cell first asserts they agree. It writes the manifest into the working-directory snapshot, then `stage_missing_files(..., allow_download=True)` fetches exactly the entries that are absent from the Hugging Face Hub **at revision `443d26d654e0…`** (never `main`), `verify_snapshot` re-hashes every file and raises on the first size or digest mismatch, and only then does `GLiNERPipeline.from_pretrained(weights_dir=WEIGHTS_DIR, encoder_dir=ENCODER_WEIGHTS_DIR)` load the verified files. There is no fallback to a different download and no remote model code is executed. The package also pins a second snapshot `mdeberta-v3-base-tokenizer` (4 files), carried and verified the same way. The effective identity, device and weight source are printed before any inference.

In [ ]:
# @title Infrastructure: pin, stage and verify the model (you may run it without reading it)
import json

MANIFEST = {
  "format": "dimer_hf_snapshot",
  "formatVersion": 1,
  "modelKey": "gliner-multi-v2.1",
  "modelId": "urchade/gliner_multi-v2.1",
  "revision": "443d26d654e0324125a96bebd8e796c14ff2efe6",
  "files": [
    {
      "path": "README.md",
      "bytes": 4770,
      "sha256": "820125f2ea897716cc38645d94f73f390ec9dfd392980c436be1d25c2e106b55"
    },
    {
      "path": "gliner_config.json",
      "bytes": 477,
      "sha256": "e25f61d91620df84aae8076811ee592e926e94d341b82e7bc1be359718f83017"
    },
    {
      "path": "model.safetensors",
      "bytes": 1155830112,
      "sha256": "2100142f31627531497850659dcb3821c99d5e71c08a8e01a98e4b11ef32a199"
    }
  ],
  "totalBytes": 1155835359
}

if (MANIFEST['modelId'], MANIFEST['revision']) != (MODEL_ID, MODEL_REVISION):
    raise RuntimeError('inline manifest does not name the identity carried by the pipeline module; the notebook was not regenerated after a change')
WEIGHTS_DIR = DEFAULT_WEIGHTS_DIR
WEIGHTS_DIR.mkdir(parents=True, exist_ok=True)
with open(WEIGHTS_DIR / MANIFEST_NAME, 'w', encoding='utf-8') as handle:
    json.dump(MANIFEST, handle, indent=2)
print({'model_id': MODEL_ID, 'revision': MODEL_REVISION, 'license': MODEL_LICENSE, 'files': len(MANIFEST['files']), 'total_bytes': MANIFEST['totalBytes']})
fetched = stage_missing_files(WEIGHTS_DIR, allow_download=True)
print({'weights_dir': str(WEIGHTS_DIR), 'fetched': fetched})
snapshot = verify_snapshot(WEIGHTS_DIR)
_files = snapshot.get('files', []) if isinstance(snapshot, dict) else []
print({'verified_files': len(_files) if isinstance(_files, list) else _files, 'revision': snapshot.get('revision', MODEL_REVISION) if isinstance(snapshot, dict) else MODEL_REVISION})

ENCODER_MANIFEST = {
  "format": "dimer_hf_snapshot",
  "formatVersion": 1,
  "modelKey": "mdeberta-v3-base-tokenizer",
  "modelId": "microsoft/mdeberta-v3-base",
  "revision": "a0484667b22365f84929a935b5e50a51f71f159d",
  "role": "encoder-tokenizer-and-config for gliner-multi-v2.1 (no weights)",
  "files": [
    {
      "path": "README.md",
      "bytes": 3667,
      "sha256": "90d5e535e04fc9ac4f39336ed33c1adc7d0136b8ce3bbe211c18a0d47b72342f"
    },
    {
      "path": "config.json",
      "bytes": 579,
      "sha256": "bcffcd343dc5efa5ef2d5a58d2b405eed108f01cc45b48d0a907b333ec41801f"
    },
    {
      "path": "spm.model",
      "bytes": 4305025,
      "sha256": "13c8d666d62a7bc4ac8f040aab68e942c861f93303156cc28f5c7e885d86d6e3"
    },
    {
      "path": "tokenizer_config.json",
      "bytes": 52,
      "sha256": "3f3978e0c036f2c2588cac34a6047cbb0af0b0dc1814254e291028529805496d"
    }
  ],
  "totalBytes": 4309323
}

if (ENCODER_MANIFEST['modelId'], ENCODER_MANIFEST['revision']) != (ENCODER_MODEL_ID, ENCODER_REVISION):
    raise RuntimeError('inline mdeberta-v3-base-tokenizer manifest does not name the identity carried by the pipeline module')
ENCODER_WEIGHTS_DIR.mkdir(parents=True, exist_ok=True)
with open(ENCODER_WEIGHTS_DIR / MANIFEST_NAME, 'w', encoding='utf-8') as handle:
    json.dump(ENCODER_MANIFEST, handle, indent=2)
fetched_mdeberta_v3_base_tokenizer = stage_missing_encoder_files(ENCODER_WEIGHTS_DIR, allow_download=True)
print({'weights_dir': str(ENCODER_WEIGHTS_DIR), 'fetched': fetched_mdeberta_v3_base_tokenizer})
_extra = verify_encoder_snapshot(ENCODER_WEIGHTS_DIR)
_extra_files = _extra.get('files', []) if isinstance(_extra, dict) else []
print({'verified_files_mdeberta_v3_base_tokenizer': len(_extra_files) if isinstance(_extra_files, list) else _extra_files})
pipe = GLiNERPipeline.from_pretrained(weights_dir=WEIGHTS_DIR, encoder_dir=ENCODER_WEIGHTS_DIR)
print({'device': getattr(pipe, 'device', None), 'source': getattr(pipe, 'source', 'local-snapshot')})

{'model_id': 'urchade/gliner_multi-v2.1', 'revision': '443d26d654e0324125a96bebd8e796c14ff2efe6', 'license': 'apache-2.0', 'files': 3, 'total_bytes': 1155835359}


{'weights_dir': '/content/weights/gliner-multi-v2.1', 'fetched': ['README.md', 'gliner_config.json', 'model.safetensors']}


{'verified_files': 3, 'revision': '443d26d654e0324125a96bebd8e796c14ff2efe6'}


{'weights_dir': '/content/weights/mdeberta-v3-base-tokenizer', 'fetched': ['README.md', 'config.json', 'spm.model', 'tokenizer_config.json']}
{'verified_files_mdeberta_v3_base_tokenizer': 4}


The tokenizer you are loading from '/content/weights/mdeberta-v3-base-tokenizer' with an incorrect regex pattern: https://huggingface.co/mistralai/Mistral-Small-3.1-24B-Instruct-2503/discussions/84#69121093e8b480e709447d5e. This will lead to incorrect tokenization. You should set the `fix_mistral_regex=True` flag when loading this tokenizer to fix this issue.


{'device': 'cuda:0', 'source': 'local-snapshot'}


### What the loader reported, and a clean starting point

This short cell finishes the setup. It prints `load_warnings` (the Python warnings caught while the two snapshots were loaded; see Prerequisites) and defines `reset_to_pretrained()`. `pipe.adapt` changes the model **in memory**, so a cell that runs after an earlier adaptation would otherwise see adapted weights. Sections 5 and 6 call `reset_to_pretrained()` first: if `pipe` has been adapted, it is reloaded from the verified snapshots (a few seconds; nothing is downloaded again), so a re-run always measures the pretrained model and always trains from it.

In [ ]:
import gc

print({'load_warnings': pipe.load_warnings})


def reset_to_pretrained():
    """Sections 5 and 6 start from the pretrained model: if an earlier pass adapted `pipe`, reload it from the verified snapshots."""
    global pipe
    if not pipe.adapted:
        return
    pipe = None
    gc.collect()
    if torch.cuda.is_available():
        torch.cuda.empty_cache()
    pipe = GLiNERPipeline.from_pretrained(weights_dir=WEIGHTS_DIR, encoder_dir=ENCODER_WEIGHTS_DIR)
    print({'reloaded': 'the pretrained model, from the verified snapshots', 'reason': 'an earlier pass had adapted the model in memory'})

{'load_warnings': ['UserWarning: The sentencepiece tokenizer that you are converting to a fast tokenizer uses the byte fallback option which is not implemented in the fast tokenizers. In practice this means that the fast version of the tokenizer can produce unknown tokens whereas the sentencepiece version would have converted these unknown tokens into a sequence of byte tokens matching the original piece of text.']}


## 4. Build, validate and split the sentences

An NER record here is a **tokenized sentence** plus its labelled **entity spans**. A span is `[start_token, end_token, label]` over token positions, counted from 0, with an **inclusive** end: in `["Imatinib", "targets", "BCR-ABL1", ...]` the span `[0, 0, "chemical_drug"]` is *Imatinib*. The pipeline rebuilds the sentence text and converts each span to **character offsets** (`start`, `end`, end exclusive), which is what the model returns and what the scoring compares.

By default the cell builds the 24-sentence biomedical sample in code. `validate_dataset` checks every record against the contract (ids, tokens, span bounds, no overlapping spans, labels from the declared set, every declared label present) and `split_ner_dataset` shuffles the records with `SEED` and keeps `VAL_FRACTION` of them for validation: 18 training and 6 validation sentences. The split is **not stratified**; instead it refuses a split that leaves any label out of the training split, and the cell prints how many spans of each label landed on each side (a label missing from validation would be named). Four refusal probes then show the validator stopping bad records before any model runs.

**Bring your own data:** set `USE_BYOD = True`, then either leave `BYOD_PATH` empty to get an upload dialog (Colab only) or set it to a `.json` / `.jsonl` file already in this runtime, and choose **Runtime → Run after** on this cell. `BYOD_LABELS` takes your comma-separated label set (empty = every label in the file); `BYOD_INFERENCE_TEXT` takes a sentence for Section 8 (empty = the first validation sentence, which is then not unseen). The cell prints the record schema, an example and the limits before it reads anything. A cancelled upload, a runtime without the Colab dialog and no `BYOD_PATH`, a missing path, a file that is not `.json` / `.jsonl` (a CSV, for example), spans written as objects instead of `[start, end, label]` lists, a label outside `BYOD_LABELS`, too few records for the split, or a label that lands only in validation each stop here with a message naming the rule and the fix. Under BYOD the printed provenance and `result.json` describe your file.

**Predict before running:** the split is a plain shuffle, not stratified. Do you expect all three labels to appear in the 6 validation sentences?

In [ ]:
import hashlib
import json
from pathlib import Path

USE_BYOD = False  # @param {type:"boolean"}
BYOD_PATH = ''  # @param {type:"string"}
BYOD_LABELS = ''  # @param {type:"string"}
BYOD_INFERENCE_TEXT = ''  # @param {type:"string"}
VAL_FRACTION = 0.25  # @param {type:"number"}
SEED = 42  # @param {type:"integer"}

BYOD_EXAMPLE = [
    {'id': 'doc-1', 'tokenized_text': ['Ada', 'Lovelace', 'worked', 'in', 'London', '.'], 'ner': [[0, 1, 'person'], [4, 4, 'location']]},
    {'id': 'doc-2', 'tokenized_text': ['Grace', 'Hopper', 'joined', 'the', 'US', 'Navy', '.'], 'ner': [[0, 1, 'person'], [4, 5, 'organization']]},
]
print({'record_schema': RECORD_SCHEMA_HINT, 'file_types': list(BYOD_SUFFIXES), 'example_jsonl_lines': [json.dumps(r) for r in BYOD_EXAMPLE]})
print({'limits': {'records_min': minimum_records(VAL_FRACTION), 'records_max': MAX_DATASET_EXAMPLES, 'records_min_per_split': MIN_DATASET_EXAMPLES, 'tokens_per_record_max': MAX_TOKENS_PER_EXAMPLE, 'chars_per_token_max': MAX_TOKEN_CHARS, 'labels_max': MAX_LABELS, 'chars_per_label_max': MAX_LABEL_CHARS, 'inference_text_chars_max': MAX_TEXT_CHARS}})

if USE_BYOD:
    if BYOD_PATH.strip():
        byod_path = Path(BYOD_PATH.strip()).expanduser()
        if not byod_path.is_file():
            raise FileNotFoundError(f'BYOD_PATH = {BYOD_PATH!r} is not a file in this runtime: fix the path, or leave BYOD_PATH empty to upload a file in Colab.')
        file_name = byod_path.name
    else:
        try:
            from google.colab import files
        except ImportError:
            raise RuntimeError('The upload dialog needs Google Colab. Elsewhere, put the .json or .jsonl file in this runtime and set BYOD_PATH to its path.') from None
        uploaded = files.upload()
        if len(uploaded) != 1:
            raise ValueError(f'Upload exactly one .json or .jsonl file (received {len(uploaded)}). Run this cell again and choose a file, or set BYOD_PATH to a file already in this runtime.')
        file_name, payload = next(iter(uploaded.items()))
        byod_path = Path('work') / Path(file_name).name
        byod_path.parent.mkdir(parents=True, exist_ok=True)
        byod_path.write_bytes(payload)
    declared_labels = [label.strip() for label in BYOD_LABELS.split(',') if label.strip()] or None
    raw_dataset = load_byod_dataset(byod_path, allowed_labels=declared_labels)
    LABELS = declared_labels or sorted({span[2] for record in raw_dataset for span in record['ner']})
    data_source = 'BYOD (' + file_name + ')'
else:
    raw_dataset = generate_synthetic_ner_dataset()
    LABELS = list(ADAPT_CLASSES)
    data_source = 'Synthetic biomedical dataset (24 records)'

validate_inputs('label check', LABELS)  # the label set must also satisfy the model's input ceilings
val_manifest = validate_dataset(raw_dataset, LABELS)
train_records, val_records = split_ner_dataset(raw_dataset, val_fraction=VAL_FRACTION, seed=SEED, labels=LABELS)
coverage = split_label_coverage(train_records, val_records, LABELS)
dataset_sha256 = hashlib.sha256(json.dumps([{k: r[k] for k in ('id', 'tokenized_text', 'ner')} for r in raw_dataset], sort_keys=True).encode('utf-8')).hexdigest()

print({'data_source': data_source, 'labels': LABELS, 'total_records': len(raw_dataset), 'train_records': len(train_records), 'val_records': len(val_records), 'dataset_sha256': dataset_sha256[:16] + '...'})
print({'span_counts': coverage})
example = train_records[0]
print({'example': {'id': example['id'], 'tokenized_text': example['tokenized_text'], 'ner': example['ner'], 'text': example['text'], 'character_spans': [(s['start'], s['end'], s['label'], s['text']) for s in example['spans']]}})

probes = {
    'duplicate id': [dict(r, id='same') for r in train_records[:4]],
    'span past the last token': [dict(train_records[0], ner=[[0, 99, LABELS[0]]]), *train_records[1:4]],
    'span written as an object': [dict(train_records[0], ner=[{'start': 0, 'end': 0, 'label': LABELS[0]}]), *train_records[1:4]],
    'too few records to split': train_records[:5],
}
for name, probe in probes.items():
    try:
        validate_dataset(probe, LABELS, require_all_labels=False)
        split_ner_dataset(probe, val_fraction=VAL_FRACTION, seed=SEED, labels=[])
        print({'probe': name, 'verdict': 'accepted'})
    except (KeyError, TypeError, ValueError) as exc:
        print({'probe': name, 'rejected': str(exc)[:150]})

{'record_schema': 'each record is {id: str, tokenized_text (or tokens): [str, ...], ner: [[start_token, end_token, label], ...]} with token indices counted from 0 and an inclusive end', 'file_types': ['.json', '.jsonl'], 'example_jsonl_lines': ['{"id": "doc-1", "tokenized_text": ["Ada", "Lovelace", "worked", "in", "London", "."], "ner": [[0, 1, "person"], [4, 4, "location"]]}', '{"id": "doc-2", "tokenized_text": ["Grace", "Hopper", "joined", "the", "US", "Navy", "."], "ner": [[0, 1, "person"], [4, 5, "organization"]]}']}
{'limits': {'records_min': 14, 'records_max': 1000, 'records_min_per_split': 4, 'tokens_per_record_max': 384, 'chars_per_token_max': 100, 'labels_max': 25, 'chars_per_label_max': 100, 'inference_text_chars_max': 5000}}
{'data_source': 'Synthetic biomedical dataset (24 records)', 'labels': ['disease', 'chemical_drug', 'gene_protein'], 'total_records': 24, 'train_records': 18, 'val_records': 6, 'dataset_sha256': 'ed10b1db454078f1...'}
{'span_counts': {'train_spans': {'di

**What to notice:** the schema and limits printed first, 18/6 records, the span counts per label on each side, one example in both token and character form (`(0, 7, 'chemical_drug', 'Aspirin')`-style tuples), and four rejected probes.

<details><summary>Check your reasoning</summary>Yes, with the default seed all three labels appear in validation: 6 `disease`, 6 `chemical_drug` and 6 `gene_protein` spans in the 6 validation sentences, 19 / 19 / 21 in training. That is a property of this sample, not of the split: every sentence has a drug and a disease, and 22 of 24 have a gene or protein, so almost any shuffle covers all three. A BYOD file with a rare label can land it only in validation; the split then stops and names the label, and the fix is more examples or another `SEED`.</details>

## 5. Measure the zero-shot baseline

Before any training, `pipe.evaluate(val_records, labels=LABELS)` asks the pretrained model for every label on each validation sentence and compares its entities with the gold spans. Scoring is **exact-span**: a prediction counts as a hit only if its `(start, end, label)` equals a gold span exactly, so *metastatic melanoma* against gold *melanoma* is both a false positive and a miss. **Precision** = hits / predicted spans, **recall** = hits / gold spans, **F1** = their harmonic mean. **Micro** averages pool every span of every label (the big labels dominate); **macro** F1 is the plain mean of the per-label F1 scores (each label counts once). `per_class` holds `hits`, `predicted` and `gold` per label, from which all of these follow.

The cell first calls `reset_to_pretrained()`, so on a re-run this is still the pretrained model, never an adapted one.

**Predict before running:** the labels are the snake-case names `disease`, `chemical_drug` and `gene_protein`. Which one do you expect the zero-shot model to find worst, and why?

In [ ]:
reset_to_pretrained()
baseline_eval = pipe.evaluate(val_records, labels=LABELS)


def metrics_table(evaluation):
    rows = [f"{'label':<16}{'gold':>5}{'predicted':>10}{'hits':>6}{'precision':>10}{'recall':>8}{'F1':>8}"]
    for label, row in evaluation['per_class'].items():
        rows.append(f"{label:<16}{row['gold']:>5}{row['predicted']:>10}{row['hits']:>6}{row['precision']:>10.4f}{row['recall']:>8.4f}{row['f1']:>8.4f}")
    micro = evaluation['micro']
    rows.append(f"{'micro (pooled)':<16}{'':>5}{'':>10}{micro['hits']:>6}{micro['precision']:>10.4f}{micro['recall']:>8.4f}{micro['f1']:>8.4f}")
    rows.append(f"{'macro F1 (mean of labels)':<56}{evaluation['macro']['f1']:>8.4f}")
    return '\n'.join(rows)


print(f"Zero-shot baseline on the validation split ({baseline_eval['num_samples']} sentences, threshold {baseline_eval['threshold']}):")
print(metrics_table(baseline_eval))

Asking to truncate to max_length but no maximum length is provided and the model has no predefined maximum length. Default to no truncation.


Zero-shot baseline on the validation split (6 sentences, threshold 0.5):
label            gold predicted  hits precision  recall      F1
disease             6         6     5    0.8333  0.8333  0.8333
chemical_drug       6         8     6    0.7500  1.0000  0.8571
gene_protein        6         3     1    0.3333  0.1667  0.2222
micro (pooled)                     12    0.7059  0.6667  0.6857
macro F1 (mean of labels)                                 0.6375


**What to notice:** one row per label with its gold, predicted and hit counts, then the micro and macro lines. Find the label whose `hits` is far below its `gold`.

<details><summary>Check your reasoning</summary>`gene_protein` is the weak one: F1 0.2222, 1 hit of 6 gold spans, against much better `disease` and `chemical_drug` rows; micro F1 is 0.6857 (precision 0.7059, recall 0.6667) and macro F1 0.6375. Two things are at work and this notebook cannot separate them: gene symbols such as *PDCD1* or *MS4A1* are hard to recognise without domain context, and the prompt is the artificial label name `gene_protein`, not natural words such as "gene or protein". Adaptation trains on exactly these label names, so part of any later gain may be the model learning what the names mean.</details>

## 6. Adapt the layers above the frozen encoder

`pipe.adapt(train_records, val_records, ...)` runs a short PyTorch training loop on the 18 training sentences. **Freezing** the mDeBERTa text encoder (`freeze_text_encoder=True`) means its 277.5 M parameters are not updated; what trains is everything above it — the projection on top of the encoder (`token_rep_layer` projection), the BiLSTM (`rnn`), the span representation layer and the prompt (label) projection, 11.4 M parameters in all. The cell prints the trainable and frozen counts by module. Training only this small part keeps memory low, runs on CPU and limits how far 18 sentences can pull the model. AdamW updates those weights to lower GLiNER's span-matching loss, `EPOCHS` passes over the training set in batches of `BATCH_SIZE`; after each epoch the validation split is scored, so the history shows loss and validation F1 side by side. The validation split is therefore also the **training monitor**: there is no separate, untouched test split (Section 7 comes back to this).

The cell starts with `reset_to_pretrained()`: if you change a field and re-run from here, training starts again from the pretrained model instead of continuing from the last run. `pipe.adapt` itself refuses a model that was already adapted.

**Predict before running:** will the training loss fall steadily over the three epochs, and will the validation F1 rise every epoch?

In [ ]:
import time

EPOCHS = 3  # @param {type:"integer"}
LEARNING_RATE = 5e-5  # @param {type:"number"}
BATCH_SIZE = 4  # @param {type:"integer"}

reset_to_pretrained()
started = time.perf_counter()
adapt_result = pipe.adapt(
    train_records=train_records,
    val_records=val_records,
    epochs=EPOCHS,
    learning_rate=LEARNING_RATE,
    batch_size=BATCH_SIZE,
    freeze_text_encoder=True,
    seed=SEED,
    labels=LABELS,
)
elapsed = time.perf_counter() - started

print({'trainable_parameters': adapt_result['trainable_parameters'], 'frozen_parameters': adapt_result['frozen_parameters']})
print({'trainable_modules': adapt_result['trainable_modules'], 'frozen_modules': adapt_result['frozen_modules']})
print(f'Adaptation completed in {elapsed:.1f} s ({EPOCHS} epochs, learning rate {LEARNING_RATE}, batch size {BATCH_SIZE}, seed {SEED}).')
for step in adapt_result['history']:
    print(f"  Epoch {step['epoch']}: loss={step['loss']:.4f}  validation micro F1={step.get('val_f1', 0.0):.4f}")

{'trainable_parameters': 11418112, 'frozen_parameters': 277531392}
{'trainable_modules': {'model.prompt_rep_layer': 2099712, 'model.rnn': 1576960, 'model.span_rep_layer': 7347712, 'model.token_rep_layer': 393728}, 'frozen_modules': {'model.token_rep_layer': 277531392}}
Adaptation completed in 1.6 s (3 epochs, learning rate 5e-05, batch size 4, seed 42).
  Epoch 1: loss=11.3457  validation micro F1=0.8108
  Epoch 2: loss=8.9581  validation micro F1=0.9189
  Epoch 3: loss=6.8758  validation micro F1=0.9189


**What to notice:** the trainable count (about 11.4 M) beside the frozen one (about 277.5 M), the module names on each side (`model.token_rep_layer` appears in both: its mDeBERTa encoder is frozen, its projection trains), and the loss and validation F1 per epoch.

<details><summary>Check your reasoning</summary>In this repository's CPU check the loss fell every epoch (11.28, 8.02, 6.26) and the validation micro F1 rose every epoch (0.8108, 0.8649, 0.9189). Neither is guaranteed: with 18 sentences in 5 batches of up to 4 the loss is noisy, and another seed, learning rate or device can make an epoch stall; GPU arithmetic can also move the losses in the second decimal. The counts were 11,418,112 trainable and 277,531,392 frozen parameters: `model.span_rep_layer` 7,347,712, `model.prompt_rep_layer` 2,099,712, `model.rnn` 1,576,960 and the `model.token_rep_layer` projection 393,728 trainable; all 277.5 M frozen ones are the mDeBERTa encoder inside `model.token_rep_layer`.</details>

## 7. Compare before and after

The adapted model is scored on the same 6 validation sentences, the deltas are computed against the Section 5 baseline, and everything is written to `outputs/gliner_ner_evaluation_report.json`. The cell also appends this run to `run_history`, so a re-run with other settings (Section 11) shows up as a second row.

**How much to read into it.** These are **tutorial evidence, not a benchmark**: 6 validation sentences with 18 gold spans, all written from one template (the drug first, then a gene, then a disease), and the same split monitored training every epoch, so nothing here is an independent test. With 18 spans one span is worth about 0.05 of recall. The baseline was prompted with snake-case label names that the adapted model has now been trained on (Section 5). A large delta here says the adaptation fitted this template and these label names; it does not say how the model would do on other biomedical text.

**Predict before running:** which label will gain the most F1, and could any label get worse?

In [ ]:
import os

os.makedirs('outputs', exist_ok=True)
adapted_eval = pipe.evaluate(val_records, labels=LABELS)

delta_f1 = round(adapted_eval['micro']['f1'] - baseline_eval['micro']['f1'], 4)
delta_precision = round(adapted_eval['micro']['precision'] - baseline_eval['micro']['precision'], 4)
delta_recall = round(adapted_eval['micro']['recall'] - baseline_eval['micro']['recall'], 4)
delta_macro_f1 = round(adapted_eval['macro']['f1'] - baseline_eval['macro']['f1'], 4)
per_label_delta = {label: round(adapted_eval['per_class'][label]['f1'] - baseline_eval['per_class'][label]['f1'], 4) for label in LABELS}
evidence_note = f"tutorial evidence: {len(val_records)} validation sentences, {sum(r['gold'] for r in adapted_eval['per_class'].values())} gold spans; the validation split also monitored training (no independent test split)" + ('; one sentence template; the label names were part of the training' if not USE_BYOD else '')

eval_report = {
    'task': 'domain-adapted named-entity recognition',
    'domain': 'biomedical / clinical' if not USE_BYOD else data_source,
    'labels': list(LABELS),
    'evidence': evidence_note,
    'baseline_eval': baseline_eval,
    'adapted_eval': adapted_eval,
    'delta': {
        'f1_delta': delta_f1,
        'precision_delta': delta_precision,
        'recall_delta': delta_recall,
        'macro_f1_delta': delta_macro_f1,
        'per_label_f1_delta': per_label_delta,
    },
    'split': {'seed': SEED, 'val_fraction': VAL_FRACTION, 'train_ids': [r['id'] for r in train_records], 'validation_ids': [r['id'] for r in val_records], 'dataset_sha256': dataset_sha256},
    'training_summary': adapt_result,
}

with open('outputs/gliner_ner_evaluation_report.json', 'w', encoding='utf-8') as f:
    json.dump(eval_report, f, indent=2)

run_history = globals().get('run_history', [])
run_history.append({'data': data_source, 'epochs': EPOCHS, 'learning_rate': LEARNING_RATE, 'seed': SEED, 'epoch1_loss': adapt_result['history'][0]['loss'] if adapt_result['history'] else None, 'baseline_micro_f1': baseline_eval['micro']['f1'], 'adapted_micro_f1': adapted_eval['micro']['f1'], 'adapted_macro_f1': adapted_eval['macro']['f1']})

print('Adapted model on the validation split:')
print(metrics_table(adapted_eval))
print(f"\nMicro F1 {baseline_eval['micro']['f1']:.4f} -> {adapted_eval['micro']['f1']:.4f} ({delta_f1:+.4f}); macro F1 {baseline_eval['macro']['f1']:.4f} -> {adapted_eval['macro']['f1']:.4f} ({delta_macro_f1:+.4f})")
print({'per_label_f1_delta': per_label_delta})
print('Reading: ' + evidence_note + '.')
print('Runs so far in this session:')
for i, run in enumerate(run_history, 1):
    print(f'  {i}. {run}')

Adapted model on the validation split:
label            gold predicted  hits precision  recall      F1
disease             6         6     5    0.8333  0.8333  0.8333
chemical_drug       6         6     6    1.0000  1.0000  1.0000
gene_protein        6         7     6    0.8571  1.0000  0.9231
micro (pooled)                     17    0.8947  0.9444  0.9189
macro F1 (mean of labels)                                 0.9188

Micro F1 0.6857 -> 0.9189 (+0.2332); macro F1 0.6375 -> 0.9188 (+0.2813)
{'per_label_f1_delta': {'disease': 0.0, 'chemical_drug': 0.1429, 'gene_protein': 0.7009}}
Reading: tutorial evidence: 6 validation sentences, 18 gold spans; the validation split also monitored training (no independent test split); one sentence template; the label names were part of the training.
Runs so far in this session:
  1. {'data': 'Synthetic biomedical dataset (24 records)', 'epochs': 3, 'learning_rate': 5e-05, 'seed': 42, 'epoch1_loss': 11.345737, 'baseline_micro_f1': 0.6857, 'adapted_micr

**What to notice:** the per-label table after adaptation, the micro and macro deltas, the per-label deltas, and the `Reading:` line that states the sample size.

<details><summary>Check your reasoning</summary>`gene_protein` gains most: F1 0.2222 → 0.9231 (1 → 6 hits of 6 gold spans, with 7 predicted). `chemical_drug` goes 0.8571 → 1.0 because its two extra predictions disappear (6 hits both times), and `disease` stays at 0.8333 (5 of 6 both times). Micro F1 0.6857 → 0.9189 (+0.2332; hits 12 → 17 of 18), macro F1 0.6375 → 0.9188 (+0.2813). A label could get worse — nothing in the training forces every label up — but none did here. The whole gain is five more hits on six sentences written from one template, with label names the model has now been trained on, so it is a demonstration of the workflow, not an estimate for other text.</details>

## 8. Extract entities from a new sentence

`pipe.detect(text, labels=LABELS, threshold=DEFAULT_THRESHOLD)` returns each entity with its `text`, `label`, character `start`/`end` (end exclusive) and `score`. The **score** is the model's match score between span and label, between 0 and 1; it is not a calibrated probability, and spans below the 0.5 threshold are dropped. The cell checks the output contract — offsets inside the text, `text` equal to the slice it claims, labels from the requested set — and stops with an error if any check fails. The default sentence was written for this notebook: none of its entities occurs in the training split, although a near-twin (*Pembrolizumab blocks PDCD1 pathway activation in metastatic melanoma*) is one of the validation sentences. Under BYOD the sentence is `BYOD_INFERENCE_TEXT`.

**Predict before running:** which three spans should come back for *Pembrolizumab blocks PDCD1 receptor to treat metastatic melanoma in adult patients.*, and will *receptor* or *adult patients* be included?

In [ ]:
if USE_BYOD:
    test_sentence = BYOD_INFERENCE_TEXT.strip() or val_records[0]['text']
    if not BYOD_INFERENCE_TEXT.strip():
        print('BYOD_INFERENCE_TEXT is empty: using the first validation sentence, so this is not unseen text.')
else:
    test_sentence = 'Pembrolizumab blocks PDCD1 receptor to treat metastatic melanoma in adult patients.'
inference_result = pipe.detect(test_sentence, labels=LABELS, threshold=DEFAULT_THRESHOLD)

entities = inference_result['entities']
checks = {
    'offsets_inside_text': all(0 <= e['start'] < e['end'] <= len(test_sentence) for e in entities),
    'span_text_matches': all(test_sentence[e['start']:e['end']] == e['text'] for e in entities),
    'labels_valid': all(e['label'] in LABELS for e in entities),
}
if not all(checks.values()):
    raise RuntimeError(f'The entities break the output contract: {checks}')

print(f'Input: "{test_sentence}"')
print(f'Detected {len(entities)} entities (checks: {checks}):')
for i, e in enumerate(entities, 1):
    print(f"  {i}. [{e['start']:>2}:{e['end']:>2}] {e['label']:<14} (score={e['score']:.4f}): {e['text']}")

Input: "Pembrolizumab blocks PDCD1 receptor to treat metastatic melanoma in adult patients."
Detected 3 entities (checks: {'offsets_inside_text': True, 'span_text_matches': True, 'labels_valid': True}):
  1. [ 0:13] chemical_drug  (score=0.9936): Pembrolizumab
  2. [21:26] gene_protein   (score=0.9507): PDCD1
  3. [45:64] disease        (score=0.9957): metastatic melanoma


**What to notice:** three entities with their character offsets and scores, and all three contract checks `True`.

<details><summary>Check your reasoning</summary>*Pembrolizumab* → `chemical_drug` (characters 0–13, score 0.9979), *PDCD1* → `gene_protein` (21–26, 0.9515) and *metastatic melanoma* → `disease` (45–64, 0.9975) in this repository's CPU check. Neither *receptor* nor *adult patients* is returned. *metastatic melanoma* comes back as one span, the way the training sentences annotate diseases; an annotation scheme that marked only *melanoma* would score this as a miss under exact-span matching.</details>

## 9. Export the adapter and check the reload

To share the adapted model without copying the 1.16 GB base checkpoint, `pipe.save_artifact` writes only the trained tensors, a manifest of their names, shapes, dtypes and SHA-256 digests, the base-model identity (both snapshot revisions) and run metadata to `outputs/gliner_ner_adapter.pt`. `GLiNERPipeline.from_artifact` then loads the pretrained snapshots again and overlays the adapter. It refuses an adapter with no tensors, tensor names this model does not have, a shape or digest that differs from the manifest, or a different base or encoder revision. The cell compares **every adapter tensor** of the reloaded model with the in-memory one (bit for bit) and the predictions of both models on **every validation sentence** plus the Section 8 sentence, and stops with an error if anything differs.

**Predict before running:** should the reloaded model's scores equal the in-memory model's exactly, or only approximately? Why?

In [ ]:
artifact_path = Path('outputs/gliner_ner_adapter.pt')
pipe.save_artifact(
    artifact_path,
    metadata={
        'domain': 'biomedical' if not USE_BYOD else data_source,
        'classes': list(LABELS),
        'epochs': EPOCHS,
        'learning_rate': LEARNING_RATE,
        'weight_decay': adapt_result['weight_decay'],
        'seed': SEED,
        'trainable_parameters': adapt_result['trainable_parameters'],
        'metrics': adapted_eval['micro'],
    },
)
adapter_manifest = torch.load(artifact_path, map_location='cpu', weights_only=True)['adapter_manifest']
print(f'Saved adapter artifact: {artifact_path} ({artifact_path.stat().st_size / (1024*1024):.2f} MB, {len(adapter_manifest)} tensors)')

reloaded_pipe = GLiNERPipeline.from_artifact(
    artifact_path,
    weights_dir=WEIGHTS_DIR,
    encoder_dir=ENCODER_WEIGHTS_DIR,
)
tensor_parity = pipe.adapter_parity(reloaded_pipe, list(adapter_manifest))
if tensor_parity['tensors_identical'] != tensor_parity['tensors_compared']:
    raise RuntimeError(f'Reload parity failed: adapter tensors differ: {tensor_parity}')

parity_texts = [r['text'] for r in val_records] + [test_sentence]
prediction_mismatches = []
for text in parity_texts:
    original = pipe.detect(text, labels=LABELS, threshold=DEFAULT_THRESHOLD)['entities']
    reloaded = reloaded_pipe.detect(text, labels=LABELS, threshold=DEFAULT_THRESHOLD)['entities']
    same = len(original) == len(reloaded) and all(
        (a['start'], a['end'], a['label']) == (b['start'], b['end'], b['label']) and abs(a['score'] - b['score']) < 1e-5
        for a, b in zip(original, reloaded, strict=False)
    )
    if not same:
        prediction_mismatches.append(text)
if prediction_mismatches:
    raise RuntimeError(f'Reload parity failed: predictions differ on {len(prediction_mismatches)} of {len(parity_texts)} sentences, e.g. {prediction_mismatches[0]!r}')
reload_parity = {'adapter_tensors': tensor_parity, 'sentences_compared': len(parity_texts), 'sentences_identical': len(parity_texts) - len(prediction_mismatches)}
print({'reload_parity': reload_parity})
print('Parity verification passed: the reloaded artifact has identical adapter tensors and identical predictions.')

Saved adapter artifact: outputs/gliner_ner_adapter.pt (43.57 MB, 26 tensors)


The tokenizer you are loading from '/content/weights/mdeberta-v3-base-tokenizer' with an incorrect regex pattern: https://huggingface.co/mistralai/Mistral-Small-3.1-24B-Instruct-2503/discussions/84#69121093e8b480e709447d5e. This will lead to incorrect tokenization. You should set the `fix_mistral_regex=True` flag when loading this tokenizer to fix this issue.


Asking to truncate to max_length but no maximum length is provided and the model has no predefined maximum length. Default to no truncation.


{'reload_parity': {'adapter_tensors': {'tensors_compared': 26, 'tensors_identical': 26, 'differing': []}, 'sentences_compared': 7, 'sentences_identical': 7}}
Parity verification passed: the reloaded artifact has identical adapter tensors and identical predictions.


**What to notice:** the tensor count, `tensors_identical` equal to `tensors_compared`, and every sentence identical.

<details><summary>Check your reasoning</summary>Exactly. The reloaded model is the same digest-verified base files plus the same adapter tensors (checked against their SHA-256 in the manifest on load and compared bit for bit here), and inference in evaluation mode is deterministic on one device, so the scores are identical; the 1e-5 tolerance in the comparison is only a guard. In this repository's CPU check: 26 tensors identical, 7 of 7 sentences identical, a 45.7 MB adapter file. A mismatch would mean the file or the base snapshots changed, and the cell would stop.</details>

## 10. Write the outputs and lineage

Two more files go under `outputs/`: `gliner_ner_result.json` with the provenance (both model identities, the repository revision, the runtime), the label set, the split ids and dataset digest, the evaluation report, the reload check and the Section 8 entities; and `gliner_ner_entities.csv` with those entities as a table. The cell lists all four output files.

In [ ]:
import csv
import platform

result_payload = {
    'task': 'named-entity recognition domain adaptation',
    'pipeline_class': 'GLiNERPipeline',
    'model_id': MODEL_ID,
    'model_revision': MODEL_REVISION,
    'model_license': MODEL_LICENSE,
    'encoder_model_id': ENCODER_MODEL_ID,
    'encoder_revision': ENCODER_REVISION,
    'encoder_license': ENCODER_LICENSE,
    'repository_revision': NOTEBOOK_SOURCE['repository_revision'],
    'artifact_format': ARTIFACT_FORMAT,
    'artifact_format_version': ARTIFACT_FORMAT_VERSION,
    'notebook_source': NOTEBOOK_SOURCE,
    'data_source': data_source,
    'labels': list(LABELS),
    'evaluation_report': eval_report,
    'reload_parity': reload_parity,
    'run_history': run_history,
    'inference_sample': {
        'text': test_sentence,
        'entities': entities,
    },
    'runtime': {
        'python': platform.python_version(),
        'torch': torch.__version__,
        'transformers': transformers.__version__,
        'gliner': gliner.__version__,
        'device': pipe.device,
    },
}

with open('outputs/gliner_ner_result.json', 'w', encoding='utf-8') as f:
    json.dump(result_payload, f, indent=2)

with open('outputs/gliner_ner_entities.csv', 'w', encoding='utf-8', newline='') as f:
    writer = csv.writer(f)
    writer.writerow(['index', 'start', 'end', 'label', 'score', 'text'])
    for idx, e in enumerate(entities, 1):
        writer.writerow([idx, e['start'], e['end'], e['label'], f"{e['score']:.6f}", e['text']])

print('Generated release artifacts under outputs/:')
for fname in sorted(os.listdir('outputs')):
    fpath = Path('outputs') / fname
    print(f'  - {fname} ({fpath.stat().st_size / 1024:.1f} KB)')

Generated release artifacts under outputs/:
  - gliner_ner_adapter.pt (44614.9 KB)
  - gliner_ner_entities.csv (0.2 KB)
  - gliner_ner_evaluation_report.json (4.9 KB)
  - gliner_ner_result.json (7.8 KB)


## 11. Your turn — change one thing: the number of epochs

**Predict → Change one thing → Run → Observe → Explain**

1. **Predict.** With one epoch instead of three, will the adapted micro F1 be lower, the same or higher than the default run's? Will the epoch-1 loss change? Write your guess down first.
2. **Change one thing.** In Section 6 set `EPOCHS = 1`. Leave every other field as it is.
3. **Run.** Select the Section 6 cell and choose **Runtime → Run after**. Section 6 reloads the pretrained model before training (so this is a fresh one-epoch run, not a fourth epoch), and Sections 7–11 run again; the output files are overwritten with the one-epoch run.
4. **Observe.** The table below prints every run of this session from `run_history`. Compare `epoch1_loss`, `adapted_micro_f1` and `adapted_macro_f1` between the rows, and look at the per-label table Section 7 printed.
5. **Explain.** Why is the epoch-1 loss the same in both rows? Is the difference in F1 larger than one span (about 0.05 on these 18 gold spans)? Set `EPOCHS = 3` and **Run after** from Section 6 again to restore the default outputs.

**Optional experiments** (each re-run from the cell named): a different `SEED` in Section 4 (Run after from Section 4) changes which sentences are held out; a larger `LEARNING_RATE` in Section 6 (Run after from Section 6) shows how quickly 18 sentences can be over-fitted; your own data via BYOD (Section 4).

In [ ]:
print(f"{'run':<5}{'epochs':>7}{'learning_rate':>15}{'seed':>6}{'epoch1_loss':>13}{'baseline_F1':>13}{'adapted_F1':>12}{'adapted_macro':>15}")
for i, run in enumerate(run_history, 1):
    print(f"{i:<5}{run['epochs']:>7}{run['learning_rate']:>15}{run['seed']:>6}{run['epoch1_loss']:>13.4f}{run['baseline_micro_f1']:>13.4f}{run['adapted_micro_f1']:>12.4f}{run['adapted_macro_f1']:>15.4f}")
if len(run_history) == 1:
    print('One run so far: follow steps 2-3 above to add a one-epoch row.')

run   epochs  learning_rate  seed  epoch1_loss  baseline_F1  adapted_F1  adapted_macro
1          3          5e-05    42      11.3457       0.6857      0.9189         0.9188
One run so far: follow steps 2-3 above to add a one-epoch row.


**What to notice:** one row after **Run all**; a second row after the activity, with the same `epoch1_loss` and `baseline_F1`.

<details><summary>Check your reasoning</summary>In this repository's CPU check the one-epoch run gave micro F1 0.8108 and macro F1 0.8077 (`gene_protein` 0.6667, 4 of 6 hits) against 0.9189 / 0.9188 for three epochs; the epoch-1 loss was 11.2828 in both rows. It is the same because Section 6 reloads the pretrained model and reseeds before training, so the first epoch of both runs sees the same weights, batches and dropout; a fresh start is what makes the comparison fair. The difference is two hits (15 versus 17 of 18 gold spans), more than one span but still a small, single-template sample: it shows that this sample needs more than one pass, not how many epochs other data needs.</details>

## Interpretation and limits

Adaptation here trains the layers above a frozen mDeBERTa encoder so that span and label representations match this sample's entity types and label names. Freezing the encoder keeps the trained part small (11.4 M of 289 M parameters) and limits how far 18 sentences can move the model. Each returned span is an exact character slice of the input; its score is the model's uncalibrated span–label match score. Exact-span F1 requires exact agreement on `(start, end, label)` triples: boundary mismatches of even one character receive zero credit. The exported adapter carries only the trained tensors, their manifest and the lineage headers.

The numbers are **tutorial evidence**: 6 validation sentences with 18 gold spans, all written from one template, a validation split that also monitored training (there is no independent test split), and a zero-shot baseline prompted with snake-case label names that the adapted model was then trained on. The delta shows that the adaptation fits this template and these label names; it is not an estimate of performance on other biomedical text, and it says nothing about other domains.

Successful execution proves that the recorded repository revision's pipeline modules, carried in this standalone notebook, can acquire and digest-verify the pinned model, validate the demonstrated dataset contract, execute bounded fine-tuning, evaluate metrics against the pre-adaptation baseline, and emit and reload the shown machine-readable artifacts — without the repository being reachable. It does **not** establish benchmark superiority, production fitness, or generalized performance across unseen domains.

## Troubleshooting

- **Section 1 stops with "needs a Linux x86_64 runtime".** The locked environment uses manylinux wheels; use Google Colab, Kaggle or a Linux Jupyter server.
- **Section 1 fails while downloading.** PyPI or the managed-Python download was interrupted: run the cell again (finished parts are reused). A size/SHA-256 refusal of the `uv` wheel that repeats means the download is being altered.
- **"The isolated environment's Python process exited".** Usually out of memory. Restart the session and choose **Run all**; on a small runtime, close other notebooks first.
- **Section 3 fails with a digest or size mismatch.** A snapshot file was truncated or altered: delete the `weights/` folder in the runtime and run Section 3 again.
- **The Section 5 numbers look like the adapted ones.** They cannot after this fix (`reset_to_pretrained()`); if you edited the cells, re-run from Section 3.
- **BYOD errors.** Each names the failed rule: a file that is not `.json`/`.jsonl`, a span that is not a `[start_token, end_token, label]` list, a label outside `BYOD_LABELS`, too few records (at least 14 at `VAL_FRACTION = 0.25`), or a label that landed only in validation (add examples or change `SEED`). Fix the file or the field and **Run after** from Section 4.
- **Reload parity failed.** The adapter file was changed after export, or the base snapshots differ: re-run from Section 6.

## Glossary

- **Named-entity recognition (NER):** finding spans of text that name things of given types (a drug, a disease, a person).
- **Span:** a contiguous piece of text, given here either as token positions `[start_token, end_token]` (inclusive) or as character offsets `start`/`end` (end exclusive).
- **Zero-shot:** using label names the model was never trained on, as plain text prompts.
- **Exact-span match:** a prediction counts only if its start, end and label all equal a gold span.
- **Precision / recall / F1:** hits divided by predicted spans / by gold spans / their harmonic mean.
- **Micro / macro average:** micro pools all spans of all labels; macro averages the per-label F1 scores, each label counting once.
- **Frozen encoder:** the mDeBERTa layers whose weights are not updated during adaptation.
- **Adapter:** the trained tensors saved separately from the base checkpoint and overlaid on it at load time.
- **Threshold:** the score below which a span is dropped (0.5 here).
- **Training monitor:** data scored during training to watch progress; here it is the validation split, so it is not an independent test.

## Conclusion (your notes)

Optional; not a required submission. Complete these sentences from your own run:

1. The zero-shot model was weakest on the label ___, because ___.
2. After adaptation the micro F1 moved from ___ to ___; with 18 gold spans, one span is worth about ___.
3. With one epoch instead of three, I observed ___.
4. Before trusting a gain like this on my own text, I would want ___.

## References

- Repository README: https://github.com/kurtvalcorza/gliner-ner-pipeline/blob/main/README.md
- Repository model card: https://github.com/kurtvalcorza/gliner-ner-pipeline/blob/main/MODEL_CARD.md
- Upstream model: https://huggingface.co/urchade/gliner_multi-v2.1
- Encoder assets: https://huggingface.co/microsoft/mdeberta-v3-base
- Upstream code: https://github.com/urchade/GLiNER
- GLiNER paper: https://arxiv.org/abs/2311.08526